# PRAM: main strategy grid, hospital-level analyses and supplementary computations (notebook 01)

Set the environment variables described in the repository README (`PRAM_DATA_ROOT` and, for the temporal and hospital analyses, `PRAM_MIMIC4_ROOT` and `PRAM_EICU_ROOT`) and run the cells from top to bottom. Finished units are stored in `out_pram/resume/`. When the notebook is run again, finished units are skipped (`[SKIP]`), so an interrupted run continues where it stopped, and cells added at the end do not recompute earlier steps.

Each analysis section has a switch `RUN_*` in the settings cell (cell 1); switching a section off removes both its computation and its report.

## Order of execution

1. Settings cell, data loading and fit of the frozen models (about 75 seconds), input check table.
2. Pre-check: finished units, units to run, expected time.
3. Reference check of the 42-column grid, then the strategy grid and all analysis sections. On an empty working folder everything is computed; finished units found in `resume/` are read instead of recomputed.
4. Rescoring 1 (30 pooled units) and rescoring 2 (6 starting-bank units).
5. Closing cell: summary, map of result files, and the result zip `out_pram.zip` in the working folder.

With `RESCORE_ONLY_GUARD = True` in the settings cell the notebook stops before computing, and prints the reason, whenever a unit outside the rescoring would have to run anew. This protects a working folder that already holds finished results from a rerun of 12 to 16 hours when the settings have changed. The default is `False`, so a first run computes everything. If the status table shows a comparison mismatch, that value should not be used.

**Environment**: a CPU with 16 or more cores and 32 GB of memory or more is recommended (approximate). Only the TabPFN analysis (A9) benefits from a GPU. The input check table at the bottom of cell 1 lists every input and whether it was found. `RUN_PROFILE = "check"` runs the pipeline once on the real data in 1 to 2 hours to verify that it completes; its results are not final results.

## Sections

Four computations refit models (N1 to N4) and three re-count existing results (tail, outcome-specific thresholds, tau). Each section has a switch `RUN_*` in cell 1.

| Section | What it gives | Default |
|---|---|---|
| Reference check | before the 42-column grid, four units are run and compared with reference values stored in the check cell (four decimals) | on |
| N2 33-column set (A8) | the 9 sparse columns are removed before input; the same grid (eight strategies and bank sizes) and the hospital level are run in the main pipeline | on |
| N4 update hierarchy, closed testing, retrieval then Platt (A4, A5) | intercept only, intercept and slope, model revision (anchored ridge), closed-testing choice, and Platt after retrieval, on the frozen logistic model. CatBoost gets Platt only | on |
| N3 TabPFN (A9) | a retrieval layer on a frozen TabPFN (eICU and MIMIC-III; bank sizes 500, 1,000, 5,000 and full) | on |
| N1 temporal validation (A1) | eICU 2014 bank versus 2015 test (paired comparison with a same-period bank of the same size); MIMIC-III development model versus MIMIC-IV earlier-period bank versus 2017 to 2019 test | on |
| Tail (A2) | per-draw tail of bank sizes 500, 1,000, 2,000 and 5,000 (100 draws per cell), probability of losing more than the non-inferiority margin with a 95% interval | on |
| Outcome-specific thresholds (A3) | net benefit over the full 1 to 50% threshold grid, fraction of hospitals whose AUROC winner and net benefit winner differ | on |
| Tau, prediction interval, event counts (A6) | REML tau squared, Q and degrees of freedom, Hartung-Knapp untruncated and truncated forms, prediction interval, hospital x test-fold event counts, variance diagnosis (original DeLong core, standard DeLong, bootstrap) | on |
| A7 unlabeled network | discrimination and calibration of unlabeled hospitals that receive the mean partner probability (the partner prevalence scenarios are off unless `PARTNER_PREV_FACTORS` is changed to [0.5, 1.0, 2.0]) | on |
| A11 strategy selection rule | regret of selection by inner cross-validation within a bank | off |
| A10 feature labels | only the number of input and transformed columns is printed | removed |
| A1c, A1d, variants without temperature or INR | refit of the development model inside MIMIC-IV, hospital temporal validation, one variable removed | off |

## Result files of the added sections

| Paper position | Value | Notebook section | Result csv |
|---|---|---|---|
| Table 2 q column (full bank, six cells) | paired DeLong p of one full-bank draw (standard implementation) with Benjamini-Hochberg adjustment over six cells | rescoring 1 | `RESCORE_table2_q.csv` |
| Supplementary S7 Panel A (median p, q, verdict over draws) | median p per bank size, BH-adjusted q over six cells | rescoring 1 | `RESCORE_S7_panelA.csv` |
| Supplementary S7 Panel B (fraction of significant draws, 42 columns) | fraction of draws with p < 0.05 | rescoring 1 | `RESCORE_S7_panelB.csv` |
| Supplementary Table S14 (starting table, p column) | paired DeLong p by local bank size, starting from the source bank (development split) | rescoring 2 | `RESCORE_coldstart_S20.csv` |

## Expected run time (approximate)

| Section | Approximate | Basis |
|---|---|---|
| Reference check | about 30 minutes | four units, reused as units of the 42-column grid |
| 42-column grid (eight strategies and update hierarchy) | about 4 hours | about 1,200 local CatBoost fits, anchored ridge (including the full bank), closed testing, retrieval fits |
| 33-column grid | about 4 hours | same as above |
| Hospital level (42 and 33) | about 30 minutes each | 126 hospitals x 3 folds |
| A7 network (42 and 33) | about 10 to 20 minutes each (rough) | one cosine neighbor query to at most 20 partners per eligible hospital and 21 mixing weights scored; no model fit |
| TabPFN | about 1 hour on a GPU; tens of minutes on a CPU with a reduced sample | the code prints the expected time first |
| Temporal validation | about 2 hours | five strategies, no local CatBoost |
| Tail (33-column eICU) | about 1 hour; about 1 to 1.5 hours more if the 42-column tail is also computed | retrieval, recalibration, anchored ridge, 100 draws per cell |
| Variance diagnosis, thresholds, meta-analysis | about 20 minutes | aggregation without refitting and the within-hospital bootstrap |

These times are estimates from the number of fits and the model size. If they are wrong, scale them from the time of the first unit.


In [ ]:
# ================================================================
# PRAM Cell 1. Settings and input check (all constants live in this cell only)
# ================================================================
import os, sys, io, gc, json, time, math, zipfile, hashlib, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from scipy.optimize import minimize, minimize_scalar

import sklearn
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, brier_score_loss, average_precision_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.isotonic import IsotonicRegression
from sklearn.exceptions import ConvergenceWarning
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


def _ov(name, default):
    """Used only when a test overrides values. Defaults apply in a normal run."""
    return globals().get(name + "_OVERRIDE", default)


# ---- run profile ---------------------------------------------------
#   "full"  : for the paper (default). The repetition counts are set in one place below.
#   "check" : pipeline check. Only verifies that the run completes on real data. Results are not used in the paper.
RUN_PROFILE = _ov("RUN_PROFILE", "full")
_PROFILES = {
    "full":  dict(N_DRAWS_MAIN=10, N_DRAWS_TAIL=100, BANK_SIZES=[500, 1000, 2000, 5000, "full"],
                  TAIL_BANKS=[500, 1000, 2000, 5000], LIGHT_BANKS=[1000, "full"], SITE_LIMIT=None,
                  TABPFN_BANKS=[500, 1000, 5000, "full"], COLDSTART_BANKS=[0, 100, 300, 500, 1000, 2000, 5000, "full"]),
    "check": dict(N_DRAWS_MAIN=2, N_DRAWS_TAIL=10, BANK_SIZES=[500, "full"],
                  TAIL_BANKS=[500], LIGHT_BANKS=["full"], SITE_LIMIT=8, TABPFN_BANKS=[500, "full"], COLDSTART_BANKS=[0, 500, "full"]),
}
_P = _PROFILES[RUN_PROFILE]

# ---- shared settings (seed, bootstrap repetitions, input folders) are defined once in code/pram_config.py ----
_cfg_dir = next((p / "code" for p in [Path.cwd(), *Path.cwd().parents] if (p / "code" / "pram_config.py").is_file()), None)
if _cfg_dir is None:
    raise FileNotFoundError("code/pram_config.py not found: open this notebook inside the repository checkout")
sys.path.insert(0, str(_cfg_dir))
from pram_config import SEED, BOOT_SEED, N_BOOT_WITHIN, THRESH_RANGE, DATA_ROOT, MIMIC4_ROOT, EICU_ROOT  # noqa: E402

# ---- reproducibility and repetition counts (one value for the whole analysis) --------------------------
ALPHA_GRID = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]   # mixing weight grid
K_NEIGHBORS = 50                            # default of retrieval_R1_cosine (Methods)
N_FOLDS = 5                                 # number of mock-batch folds
N_INNER = 3                                 # number of inner cross-validation folds within a bank (inner_cv_bank)
N_SITE_SPLITS = 3                           # number of folds in the hospital-level analysis
N_DRAWS_MAIN = _ov("N_DRAWS_MAIN", _P["N_DRAWS_MAIN"])    # number of draws per bank size
N_DRAWS_TAIL = _ov("N_DRAWS_TAIL", _P["N_DRAWS_TAIL"])    # tail risk: bank draws per cell = 5 folds x 20 draws per fold (100, as in Supplementary S7 Panel D)
BANK_SIZES = _ov("BANK_SIZES", _P["BANK_SIZES"])          # bank sizes of the strategy grid
TAIL_BANKS = _ov("TAIL_BANKS", _P["TAIL_BANKS"])          # the four bank sizes of Supplementary S7 Panel D
TABPFN_BANKS = _ov("TABPFN_BANKS", _P["TABPFN_BANKS"])
LIGHT_BANKS = _ov("LIGHT_BANKS", _P["LIGHT_BANKS"])       # bank sizes of the variants without temperature only and without INR only
SITE_LIMIT = _ov("SITE_LIMIT", _P["SITE_LIMIT"])          # None means all eligible hospitals
DEV_CB_ITER = _ov("DEV_CB_ITER", 500)       # frozen CatBoost iterations
LOCAL_CB_ITER = _ov("LOCAL_CB_ITER", 300)   # local CatBoost iterations
LAM_GRID = np.logspace(-4, 4, 17)           # anchored ridge regularization grid
ANCHORED_MAX_BANK = None                    # largest bank size at which the anchored ridge (model revision step) is fitted. None also fits the full bank
REVISION_RIDGE = 1e-5                       # tiny regularization for numerical stability of the model-revision fit (mean-loss units)
CLOSED_TEST_ALPHA = 0.05                    # closed-testing significance level
N_EVENT_MIN = 2                             # minimum number of events and non-events in a bank (below this the update fit is skipped)
N_TIME_SPLITS = 2                           # A1 temporal validation: number of folds that split the test period by outcome stratum (test fold and same-period bank alternate)

# ---- population and outcome definitions ----------------------
DROP_COLS = ["crp_min", "crp_max", "crp_last", "albumin_last", "albumin_max", "albumin_min",
             "bmi", "height_cm_min", "height_cm_last", "height_cm_max"]
OUTCOME_MAP = {"AKI_168h": "aki_168h", "mortality_168h": "death_168h"}
OUTCOMES = ["AKI_168h", "mortality_168h"]
TARGETS = ["MIMIC-IV", "MIMIC-III", "eICU"]
MISS_CUT_PCT = 30                           # variables whose maximum missingness over the three databases exceeds this value are dropped from the 33-column set
EXPECTED_SPARSE_9 = ["inr_last", "inr_max", "inr_min", "temp_c_last", "temp_c_max", "temp_c_min",
                     "weight_kg_last", "weight_kg_max", "weight_kg_min"]      # the nine variables expected to be dropped

# ---- hospital level and network ---------------------------
MIN_SITE_N = _ov("MIN_SITE_N", 300)
MIN_SITE_EVENTS = _ov("MIN_SITE_EVENTS", 15)    # minimum number of events and of non-events each
MAX_PARTNERS = 20
TEST_FRAC_NET = 0.30
NET_ALPHA_FIXED = 0.30                      # fixed mixing weight of an unlabeled hospital
NET_ALPHA_GRID = [round(a, 2) for a in np.arange(0.0, 1.0001, 0.05)]   # 0 to 1.0 in steps of 0.05
PARTNER_PREV_FACTORS = _ov("PARTNER_PREV_FACTORS", [1.0])                # partner bank prevalence multipliers. With 1.0 only, the whole partner cohort is used. Changing to [0.5, 1.0, 2.0] turns on the partner prevalence scenarios of A7

# ---- net benefit thresholds (A3) --------------------------------------------------
THRESH_GRID = np.round(np.arange(0.01, 0.5001, 0.01), 2)      # full 1 to 50% grid, stored in all tables
THRESH_MS = 0.20                            # main threshold of the main analysis
#   THRESH_RANGE (outcome-specific threshold ranges) is defined in code/pram_config.py; the summary changes when the ranges are edited.
#   the full-grid (1 to 50%) net benefit is written to csv, so changing the range needs no rerun.
T_SITE = {oc: round(float(np.mean(r)), 2) for oc, r in THRESH_RANGE.items()}   # midpoints of the outcome-specific ranges (representative thresholds of A2 and A11)
RANGE_ORIGINAL = (0.02, 0.30)               # range 2 to 30%

# ---- A2 tail risk and non-inferiority margin (post hoc definitions, labeled post hoc in the summary) ---------
NI_MARGINS_AUROC = [0.0, 0.005, 0.01, 0.02]
NI_MARGIN_AUROC = 0.01
NI_MARGINS_NB = [0.0, 0.0005, 0.001, 0.002]
NI_MARGIN_NB = 0.001                        # net benefit 0.001 = one net true positive per 1,000 patients
CP_LEVEL = 0.95                             # Clopper-Pearson interval level
SAFE_UPPER_BOUND = 0.05                     # "safest minimum bank size" = first size at which the upper bound of the probability of losing is at or below this value

A2_TARGETS_F33 = ["eICU"]                   # targets for which the 33-column tail is computed anew (eICU)
A2_RECOMPUTE_F42 = _ov("A2_RECOMPUTE_F42", False)                    # if False and the reference 42-column tail file (`a54_alpha_stability.jsonl`) exists in REFERENCE_RESUME_DIR, the 42-column tail is read from it (AUROC gain only, no model fit)
REFERENCE_RESUME_DIR = Path("reference") / "resume"                            # folder of stored reference result files (optional)
A54_PATH = REFERENCE_RESUME_DIR / "a54_alpha_stability.jsonl"
A29_PATH = REFERENCE_RESUME_DIR / "a29_real_sites.jsonl"
A51_PATH = REFERENCE_RESUME_DIR / "a51_hospital_thresholds.jsonl"
A59_PATH = REFERENCE_RESUME_DIR / "a59_hospital_diag.jsonl"
REPRO_UNITS = _ov("REPRO_UNITS", [("AKI_168h", "eICU", 1000), ("mortality_168h", "eICU", "full"),
                                  ("AKI_168h", "MIMIC-III", 500), ("mortality_168h", "MIMIC-IV", 2000)])   # units compared with the reference values before the 42-column grid
REPRO_TOL = 0.00005                         # the reference values have four decimals, so a rounding tolerance is allowed

# ---- rescoring: per-draw paired DeLong (Table 2 q column, Supplementary S7 Panels A and B, development-bank starting table p column) -------------
#   Only the two sections below (rescoring and starting bank) compute anew. The other sections skip units already finished in resume.
RUN_RESCORE = _ov("RUN_RESCORE", True)               # turns on the rescoring section (pooled DeLong, 42 columns) and its pre-check
RESCORE_COLDSTART = _ov("RESCORE_COLDSTART", True)   # p column of the development-bank starting table. Turning it off skips the computation of that column
RESCORE_ONLY_GUARD = _ov("RESCORE_ONLY_GUARD", False)  # if True, stops before computing when a unit outside the rescoring would run anew (guards against rerunning 12 to 16 hours in a folder that already holds finished results). Off by default so that the notebook runs from scratch
RESCORE_NAMES = ("rescore_F42", "coldstart_F42")      # only the resume file with this name is written anew
COLDSTART_BANKS = _ov("COLDSTART_BANKS", _P["COLDSTART_BANKS"])   # 0 = source bank only, number = source bank + local bank
COLDSTART_SRC_OOF_SPLITS = 3                # number of cross-validation folds for the source-bank base prediction of _source_bank
SIG_LEVEL = 0.05                            # significance level of the per-draw test (criterion of Supplementary S7 Panel B)

# ---- A1 temporal validation ------------------------------------------------------
A1_POLICIES = _ov("A1_POLICIES", ("P", "S"))                    # P past bank, S same-period bank. Changing to ("P", "S", "R") also gives a random (periods mixed) bank R
HOSP_MIN_N_PER_YEAR = _ov("HOSP_MIN_N_PER_YEAR", 300)    # hospital-level temporal analysis: minimum number of ICU stays in each of the two years
HOSP_MIN_EVENTS_PER_YEAR = _ov("HOSP_MIN_EVENTS_PER_YEAR", 15)
HOSP_TIME_MIN_K = 10                        # if fewer hospitals qualify, only descriptive statistics instead of meta-analysis
RUN_A1C = _ov("RUN_A1C", False)                             # A1c: refit the development model on the earlier MIMIC-IV period (turned off)
YEAR_RULES = ("strict", "majority")         # MIMIC-IV year-period assignment: only when entirely inside a period / period covering the majority
RUN_A1D = _ov("RUN_A1D", False)                             # A1d: hospital-level temporal validation (turned off; the number of hospitals above the criterion in both years is unknown)

# ---- A9 TabPFN ----------------------------------------------------------
TABPFN_N_ESTIMATORS = _ov("TABPFN_N_ESTIMATORS", 4)
TABPFN_CTX_GPU = _ov("TABPFN_CTX_GPU", 10000)       # development subsample size (context limit of v2: 10,000 rows)
TABPFN_CTX_CPU = _ov("TABPFN_CTX_CPU", 2000)
TABPFN_SUBSETS_GPU = _ov("TABPFN_SUBSETS_GPU", 3)   # number of subsample ensemble members
TABPFN_SUBSETS_CPU = _ov("TABPFN_SUBSETS_CPU", 1)
TABPFN_CPU_TARGET_CAP = _ov("TABPFN_CPU_TARGET_CAP", 6000)   # maximum number of rows per target cohort on CPU
TABPFN_MAX_MINUTES = _ov("TABPFN_MAX_MINUTES", 240)          # if the expected time exceeds this, the sample is reduced
TABPFN_BATCH = _ov("TABPFN_BATCH", 128)   # rows per prediction batch. On an 8 GB GPU a batch of 10,000 context rows x 5,000 test rows requires 72 GiB and runs out of memory. Start at 128 and halve and retry on out-of-memory (tabpfn_predict)
TABPFN_TARGETS = ["eICU", "MIMIC-III"]              # eICU and MIMIC-III external validation

# ---- A11 strategy selection rule -------------------------------------------------
SEL_CANDIDATES = ["b1_base", "b5_base", "platt_b1", "recal_iso", "pram_b1", "pram_b5", "local_lr"]
#   pooled_lr needs the development record and cannot be chosen by a hospital, and local_cb and anchored ridge are left out of the candidates because their inner cross-validation within a bank is costly.

# ---- sections to run (each can be switched on or off; off removes both its computation and its report) -------------
RUN_MATRIX = _ov("RUN_MATRIX", True)       # strategy grid (main analysis, the 33-column set of A8, A4, A5 and A11 are derived from it)
RUN_A1 = _ov("RUN_A1", True)           # temporal validation
RUN_A2 = _ov("RUN_A2", True)           # tail risk
RUN_HOSPITAL = _ov("RUN_HOSPITAL", True)     # hospital-level computation (the hospital parts of A3, A6 and A8 are derived from it)
RUN_A3 = _ov("RUN_A3", True)           # decision-curve threshold report (on top of the hospital-level results)
RUN_A4 = _ov("RUN_A4", True)           # update hierarchy and closed testing (hier and anch groups from the grid and the hospital level)
RUN_A5 = _ov("RUN_A5", True)           # retrieval + Platt combination order
RUN_A6 = _ov("RUN_A6", True)           # hospital-level meta-analysis (tau, prediction interval, event counts)
RUN_A7 = _ov("RUN_A7", True)           # network of unlabeled hospitals and partner prevalence (including the network part of A8). The 33-column network is also derived from it
RUN_A9 = _ov("RUN_A9", True)           # TabPFN
RUN_REAGG = _ov("RUN_REAGG", True)      # if the reference hospital-level files (`a29_real_sites`, `a51_hospital_thresholds`, `a59_hospital_diag`) exist in REFERENCE_RESUME_DIR, recount them and compare with the hospital computation of this notebook (skipped if absent)
RUN_A11 = _ov("RUN_A11", False)         # strategy selection rule within a bank (turned off)

# ---- strategy group ----------------------------------------------------------
GROUPS_FULL = (("frozen", "pram", "recal", "local", "localcb") + (("anch", "hier") if RUN_A4 else ())
               + (("combo",) if RUN_A5 else ()) + (("sel",) if RUN_A11 else ()))
GROUPS_LIGHT = ("frozen", "pram", "recal")
GROUPS_HOSP = ("frozen", "pram", "recal", "local", "localcb") + (("anch", "hier") if RUN_A4 else ()) + (("combo",) if RUN_A5 else ())
M_EIGHT = ["b1_base", "b5_base", "recal_iso", "pram_b1", "pram_b5", "local_lr", "pooled_lr", "local_cb"]   # the eight strategies of the manuscript
FS_SPEC = {
    "F42":        dict(drop_rule=None, drop_prefix=(), targets=TARGETS, banks=BANK_SIZES,
                       groups=GROUPS_FULL, hospital=True, network=True),
    "F33":        dict(drop_rule="miss", drop_prefix=(), targets=TARGETS, banks=BANK_SIZES,
                       groups=GROUPS_FULL, hospital=True, network=True),
    "F42_noTemp": dict(drop_rule=None, drop_prefix=("temp_c_",), targets=["eICU"], banks=LIGHT_BANKS,
                       groups=GROUPS_LIGHT, hospital=False, network=False),
    "F42_noINR":  dict(drop_rule=None, drop_prefix=("inr_",), targets=["eICU"], banks=LIGHT_BANKS,
                       groups=GROUPS_LIGHT, hospital=False, network=False),
}
FS_RUN = _ov("FS_RUN", ["F42", "F33"])      # "F42_noTemp" and "F42_noINR" (variants without temperature only or INR only) run when included

# ---- output --------------------------------------------------------------
OUT_NAME = "out_pram" + ("" if RUN_PROFILE == "full" else "_CHECK")
OUT_DIR = Path(OUT_NAME)
CSV_DIR = OUT_DIR / "github_csv"
RESUME_DIR = OUT_DIR / "resume"
for _d in (OUT_DIR, CSV_DIR, RESUME_DIR):
    _d.mkdir(parents=True, exist_ok=True)
ZIP_NAME = OUT_NAME + ".zip"

# ---- candidate paths of the temporal and hospital raw data --------------------------
#   candidates for the eICU patient table are the list below plus the .gz variants.
#   MIMIC-IV candidates are listed below. If not found, write the actual path in EXTRA_RAW_PATHS below.
EXTRA_RAW_PATHS = {"eicu_patient": [EICU_ROOT / "patient.csv", EICU_ROOT / "patient.csv.gz"], "mimic4_patients": [], "mimic4_icustays": []}
#   MIMIC-IV folder (patients and icustays are expected under hosp and icu in this folder).
#   Below, hosp/patients.csv(.gz) and icu/icustays.csv(.gz) are searched. If not found, the paths searched are printed in the input check table and only A1b is skipped.
_E = ["./eICU/patient.csv", "eICU/patient.csv", "./eicu/patient.csv",
      DATA_ROOT / "eICU" / "patient.csv", DATA_ROOT / "eicu" / "patient.csv", DATA_ROOT / "eicu" / "hospital_map.csv"]
EICU_PATIENT_CANDIDATES = [Path(p) for p in _E + [str(p) + ".gz" for p in _E[:5]]
                           + EXTRA_RAW_PATHS["eicu_patient"]]
_M4DIRS = [MIMIC4_ROOT, DATA_ROOT / "mimic4", "."]
MIMIC4_PATIENTS_CANDIDATES = [Path(d) / s for d in _M4DIRS for s in ("hosp/patients.csv", "hosp/patients.csv.gz", "patients.csv")] \
    + [Path(p) for p in EXTRA_RAW_PATHS["mimic4_patients"]]
MIMIC4_ICUSTAYS_CANDIDATES = [Path(d) / s for d in _M4DIRS for s in ("icu/icustays.csv", "icu/icustays.csv.gz", "icustays.csv")] \
    + [Path(p) for p in EXTRA_RAW_PATHS["mimic4_icustays"]]


def find_first(cands):
    """the first existing path among the candidates, and the list of absolute paths searched."""
    tried = []
    for p in cands:
        try:
            ap = Path(p).resolve()
        except OSError:
            ap = Path(p)
        tried.append(str(ap))
        try:
            if Path(p).is_file():
                return Path(p), tried
        except OSError:
            continue
    return None, tried


def header_cols(path):
    try:
        return [str(c) for c in pd.read_csv(path, nrows=3).columns]
    except Exception as exc:  # noqa: BLE001
        print("  [read failed]", path, type(exc).__name__, str(exc)[:80])
        return []


# ---- input check table ---------------------------------------------------------
MAIN_FILES = {
    "MIMIC-IV features": DATA_ROOT / "mimic4" / "X_folded.parquet",
    "MIMIC-IV outcomes": DATA_ROOT / "mimic4" / "binary_outcomes_mimic4.csv",
    "MIMIC-III features": DATA_ROOT / "mimic3" / "X_external.parquet",
    "MIMIC-III outcomes": DATA_ROOT / "mimic3" / "binary_outcomes_mimic3.csv",
    "eICU features": DATA_ROOT / "eicu" / "X_external.parquet",
    "eICU outcomes": DATA_ROOT / "eicu" / "binary_outcomes_eicu.csv",
}
INPUT_ROWS = []
for _k, _p in MAIN_FILES.items():
    INPUT_ROWS.append((_k, str(Path(_p).resolve()), "present" if Path(_p).is_file() else "absent"))
DATA_OK = all(r[2] == "present" for r in INPUT_ROWS)

EICU_PATIENT_PATH, _tried_e = find_first(EICU_PATIENT_CANDIDATES)
EICU_COLS = header_cols(EICU_PATIENT_PATH) if EICU_PATIENT_PATH else []
_lowE = {c.lower(): c for c in EICU_COLS}
EICU_COL_STAY = _lowE.get("patientunitstayid")
EICU_COL_HOSP = _lowE.get("hospitalid")
EICU_COL_YEAR = _lowE.get("hospitaldischargeyear")
HAVE_HOSPITALID = bool(EICU_COL_STAY and EICU_COL_HOSP)
HAVE_EICU_YEAR = bool(EICU_COL_STAY and EICU_COL_YEAR)
INPUT_ROWS.append(("eICU patient table (hospital id)", str(EICU_PATIENT_PATH) if EICU_PATIENT_PATH else "not found",
                   "column present" if HAVE_HOSPITALID else "column absent"))
INPUT_ROWS.append(("eICU patient table (hospitaldischargeyear)", str(EICU_PATIENT_PATH) if EICU_PATIENT_PATH else "not found",
                   "column present" if HAVE_EICU_YEAR else "column absent"))

MIMIC4_PATIENTS_PATH, _tried_p = find_first(MIMIC4_PATIENTS_CANDIDATES)
MIMIC4_ICUSTAYS_PATH, _tried_i = find_first(MIMIC4_ICUSTAYS_CANDIDATES)
_cp = {c.lower(): c for c in (header_cols(MIMIC4_PATIENTS_PATH) if MIMIC4_PATIENTS_PATH else [])}
_ci = {c.lower(): c for c in (header_cols(MIMIC4_ICUSTAYS_PATH) if MIMIC4_ICUSTAYS_PATH else [])}
_m4_miss_p = [k for k in ("subject_id", "anchor_year_group") if k not in _cp]
_m4_miss_i = [k for k in ("stay_id", "subject_id") if k not in _ci]
HAVE_MIMIC4_YEAR = bool(MIMIC4_PATIENTS_PATH and MIMIC4_ICUSTAYS_PATH and not _m4_miss_p and not _m4_miss_i)
#   the unit-transfer adjustment works only when patients has anchor_year and icustays has intime.
#   for a copy without anchor_year (anonymized), the periods are split by a patient-level approximation (see read_mimic4_periods and the A1b cell).
M4_HAS_SHIFT = bool(HAVE_MIMIC4_YEAR and "anchor_year" in _cp and "intime" in _ci)
M4_MODE = "unit-transfer adjusted" if M4_HAS_SHIFT else ("patient-level approximation" if HAVE_MIMIC4_YEAR else "absent")
M4_APPROX_WHY = "" if (M4_HAS_SHIFT or not HAVE_MIMIC4_YEAR) else ("anchor_year absent" if "anchor_year" not in _cp else "intime absent")
M4_WHY = []                                  # reason A1b cannot run (human-readable). It runs when empty
if not MIMIC4_PATIENTS_PATH:
    M4_WHY.append("not found: patients")
elif _m4_miss_p:
    M4_WHY.append("column absent: " + ", ".join(_m4_miss_p) + " (patients)")
if not MIMIC4_ICUSTAYS_PATH:
    M4_WHY.append("not found: icustays")
elif _m4_miss_i:
    M4_WHY.append("column absent: " + ", ".join(_m4_miss_i) + " (icustays)")
INPUT_ROWS.append(("MIMIC-IV patients(anchor_year_group)", str(MIMIC4_PATIENTS_PATH) if MIMIC4_PATIENTS_PATH else "not found",
                   ("column present" if not _m4_miss_p else "column absent: " + ", ".join(_m4_miss_p)) if MIMIC4_PATIENTS_PATH else "not found"))
INPUT_ROWS.append(("MIMIC-IV icustays(stay_id · subject_id)", str(MIMIC4_ICUSTAYS_PATH) if MIMIC4_ICUSTAYS_PATH else "not found",
                   ("column present" if not _m4_miss_i else "column absent: " + ", ".join(_m4_miss_i)) if MIMIC4_ICUSTAYS_PATH else "not found"))
INPUT_ROWS.append(("MIMIC-IV period assignment method (A1b)", "unit-transfer adjustment if anchor_year and intime exist, otherwise patient-level approximation", M4_MODE + (" (" + M4_APPROX_WHY + ")" if M4_APPROX_WHY else "")))

try:
    import tabpfn as _tabpfn_mod            # noqa: F401
    HAVE_TABPFN = True
except Exception:  # noqa: BLE001
    HAVE_TABPFN = False
TABPFN_PKG_VERSION = "not installed"
if HAVE_TABPFN:
    try:
        from importlib.metadata import version as _pkg_version
        TABPFN_PKG_VERSION = _pkg_version("tabpfn")
    except Exception:  # noqa: BLE001
        TABPFN_PKG_VERSION = str(getattr(_tabpfn_mod, "__version__", "unknown"))
HAVE_CUDA = False
if HAVE_TABPFN:                              # torch is imported only when TabPFN is present, to look at the GPU (avoids import conflicts of torch in environments without it)
    try:
        import torch as _torch_mod
        HAVE_CUDA = bool(_torch_mod.cuda.is_available())
    except Exception:  # noqa: BLE001
        HAVE_CUDA = False
INPUT_ROWS.append(("tabpfn package", "pip install tabpfn . A9 selects the v2 weights explicitly (create_default_for_version(ModelVersion.V2)). The v2 weights are downloaded on the first run",
                   ("installed " + TABPFN_PKG_VERSION) if HAVE_TABPFN else "absent"))
INPUT_ROWS.append(("GPU(CUDA)", "used by A9 only", "present" if HAVE_CUDA else "absent"))

print("=" * 100)
print("  PRAM profile =", RUN_PROFILE, "(use full when the results go into the paper)" if RUN_PROFILE == "full" else "(for checking only; results are not used in the paper)")
print("=" * 100)
print("| input | path | status |")
print("|---|---|---|")
for _r in INPUT_ROWS:
    print("| %s | %s | %s |" % _r)
if not DATA_OK:
    print("\n  [not a stop] one of the six main inputs is missing. All analysis cells below are skipped. Fix DATA_ROOT and run again ->", DATA_ROOT.resolve())
if not HAVE_HOSPITALID:
    print("\n  The hospital id column was not found. Hospital-level analysis, network and hospital-level temporal analysis are marked not available and skipped. Paths searched:")
    for _t in _tried_e:
        print("    ", _t)
if not HAVE_EICU_YEAR:
    print("\n  The eICU discharge year column (hospitaldischargeyear) was not found. The eICU temporal validation of A1 is skipped as not available.")
if HAVE_MIMIC4_YEAR and not M4_HAS_SHIFT:
    print("\n  The MIMIC-IV period assignment is a patient-level approximation (%s). All stays of the same patient are placed in that patient anchor_year_group period. intime is not used to order patients." % M4_APPROX_WHY)
if not HAVE_MIMIC4_YEAR:
    print("\n  The MIMIC-IV time chain (A1b) cannot run , " + " · ".join(M4_WHY) + ". The MIMIC-IV time chain of A1 is skipped as not available. Paths searched (patients):")
    for _t in _tried_p:
        print("    ", _t)
    print("  paths searched (icustays):")
    for _t in _tried_i:
        print("    ", _t)
print("\n  python %s | numpy %s | pandas %s | sklearn %s" % (platform.python_version(), np.__version__, pd.__version__, sklearn.__version__))
print("  SEED=%d | folds %d | inner folds %d | hospital folds %d | draws %s (tail %s) | bank sizes %s" % (
    SEED, N_FOLDS, N_INNER, N_SITE_SPLITS, N_DRAWS_MAIN, N_DRAWS_TAIL, BANK_SIZES))
print("  output folder %s | zip %s" % (OUT_DIR.resolve(), ZIP_NAME))


## Definitions

Preprocessors (`PreprocessorGeneral`, `PreprocessorCatBoost`), data loading (`_load_x`, `_load_y`, `load_mimic4`, `load_external`), cosine retrieval (`retrieval_R1_cosine`), `compute_ece`, `_logit`, the DeLong functions (`_midrank`, `_delong_core_original`), `inner_cv_bank`, `sample_bank`, the anchored ridge (`_anchored_ridge`, `_pick_lam`) and `Resumable`, the store of finished units.

| Definition | Note |
|---|---|
| `load_external`, `load_mimic4` | `load_external` also returns `stay_ids`. `load_mimic4` counts outcomes that are missing and fills them with 0 (a warning is printed) |
| `_midrank` | uses `scipy.stats.rankdata`, which gives the same values as a loop implementation |
| `_delong_core_original` | DeLong core in which `np.arange` is used in place of the within-class midranks. It overestimates the variance of the paired AUROC difference when scores within a class are not sorted (see A6 for its size in the hospital-level data). It is used only by diagnostic cells; every analysis uses the standard fastDeLong `_delong_core` defined with the new functions |
| `inner_cv_bank` | the number of folds defaults to `N_INNER`. Retrieval rule R1 (cosine) does not use mutual information, so no mutual-information computation is made |
| `_anchored_ridge`, `_pick_lam` | the number of folds is `N_INNER` |
| `Resumable` | stores finished units under the resume folder (`out_pram/resume/`) |


In [ ]:
# ================================================================
# PRAM definitions: preprocessors, data loading, retrieval, DeLong, inner cross-validation, anchored ridge, Resumable
# ================================================================
def _log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def stable_seed(*parts):
    s = "|".join(str(p) for p in parts) + f"|SEED={SEED}"
    return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16) % (2**31 - 1)


def rng_of(*parts):
    return np.random.RandomState(stable_seed(*parts))


def _jsafe(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (Path,)):
        return str(o)
    return str(o)


class Resumable:
    """append per row + completion marker per unit. Even if the kernel dies, units already completed are not recomputed."""

    def __init__(self, name):
        self.name = name
        self.rows_path = RESUME_DIR / f"{name}.jsonl"
        self.units_path = RESUME_DIR / f"{name}.units.jsonl"
        self.rows, self.units = [], set()
        self.n_dropped = 0                                       # counts the incomplete lines that are discarded
        if self.rows_path.exists():
            with open(self.rows_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.rows.append(json.loads(line))
                    except json.JSONDecodeError:
                        self.n_dropped += 1                      # discard the incomplete last line at the time of interruption
            if self.n_dropped:
                print(f"  [{name}] incomplete line {self.n_dropped} lines discarded (last line at the time of interruption)")
        if self.units_path.exists():
            with open(self.units_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        self.units.add(line)

    @staticmethod
    def _ukey(unit):
        return "::".join(str(x) for x in unit)

    def unit_done(self, unit):
        return self._ukey(unit) in self.units

    def add_row(self, row):
        with open(self.rows_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row, default=_jsafe) + "\n")
            f.flush()
        self.rows.append(row)

    def mark_unit(self, unit):
        k = self._ukey(unit)
        with open(self.units_path, "a", encoding="utf-8") as f:
            f.write(k + "\n")
            f.flush()
        self.units.add(k)

    def df(self):
        return pd.DataFrame(self.rows) if self.rows else pd.DataFrame()


class PreprocessorGeneral:
    def __init__(self, cat_cols, num_cols, random_state=SEED):
        self.cat_cols, self.num_cols = cat_cols, num_cols
        self.cat_imputer = SimpleImputer(strategy="most_frequent")
        self.ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        self.num_imputer = IterativeImputer(max_iter=10, random_state=random_state, sample_posterior=False)
        self.scaler = StandardScaler()
        self._fitted = False
        self._all_nan_cols = []

    def fit(self, X_df):
        if len(self.cat_cols) > 0:
            Xi = self.cat_imputer.fit_transform(X_df[self.cat_cols].values)
            self.ohe.fit(Xi)
        if len(self.num_cols) > 0:
            X_num = X_df[self.num_cols].values
            self._all_nan_cols = [i for i in range(X_num.shape[1]) if np.all(np.isnan(X_num[:, i]))]
            if self._all_nan_cols:
                X_num = X_num.copy()
                X_num[:, self._all_nan_cols] = 0.0
            Xni = self.num_imputer.fit_transform(X_num)
            self.scaler.fit(Xni)
        self._fitted = True
        return self

    def transform(self, X_df):
        parts = []
        if len(self.cat_cols) > 0:
            parts.append(self.ohe.transform(self.cat_imputer.transform(X_df[self.cat_cols].values)))
        if len(self.num_cols) > 0:
            X_num = X_df[self.num_cols].values.copy()
            if self._all_nan_cols:
                X_num[:, self._all_nan_cols] = 0.0
            parts.append(self.scaler.transform(self.num_imputer.transform(X_num)))
        return np.hstack(parts).astype(np.float32) if parts else np.empty((len(X_df), 0), dtype=np.float32)

    def fit_transform(self, X_df):
        self.fit(X_df)
        return self.transform(X_df)

    def column_names(self):
        """column names after the transform (A10). An auxiliary function; not used in any computation."""
        names = []
        if len(self.cat_cols) > 0:
            names += [str(n) for n in self.ohe.get_feature_names_out(self.cat_cols)]
        names += list(self.num_cols)
        return names


class PreprocessorCatBoost:
    def __init__(self, cat_cols, num_cols, random_state=SEED):
        self.cat_cols, self.num_cols = cat_cols, num_cols
        self.cat_imputer = SimpleImputer(strategy="most_frequent")
        self.num_imputer = IterativeImputer(max_iter=10, random_state=random_state, sample_posterior=False)
        self.cat_feature_indices = list(range(len(cat_cols)))
        self._all_nan_cols = []

    def fit(self, X_df):
        if len(self.cat_cols) > 0:
            self.cat_imputer.fit(X_df[self.cat_cols].values)
        if len(self.num_cols) > 0:
            X_num = X_df[self.num_cols].values
            self._all_nan_cols = [i for i in range(X_num.shape[1]) if np.all(np.isnan(X_num[:, i]))]
            if self._all_nan_cols:
                X_num = X_num.copy()
                X_num[:, self._all_nan_cols] = 0.0
            self.num_imputer.fit(X_num)
        return self

    def transform(self, X_df):
        parts = []
        if len(self.cat_cols) > 0:
            parts.append(self.cat_imputer.transform(X_df[self.cat_cols].values))
        if len(self.num_cols) > 0:
            X_num = X_df[self.num_cols].values.copy()
            if self._all_nan_cols:
                X_num[:, self._all_nan_cols] = 0.0
            parts.append(self.num_imputer.transform(X_num))
        if not parts:
            return pd.DataFrame()
        out = pd.DataFrame(np.hstack(parts), columns=self.cat_cols + self.num_cols)
        for c in self.cat_cols:
            out[c] = out[c].round().astype(int)
        return out

    def fit_transform(self, X_df):
        self.fit(X_df)
        return self.transform(X_df)


# ---- data loading ------------------------------------------
def _detect_cols(X_df):
    cat = [c for c in X_df.columns if X_df[c].dropna().isin([0, 1]).all() and X_df[c].nunique(dropna=True) <= 2]
    num = [c for c in X_df.columns if c not in cat]
    return cat, num


def _load_x(db_name):
    """reads the feature matrix and returns (X without stay_id and fold_id, stay_id array)."""
    if db_name == "mimic4":
        fp = DATA_ROOT / "mimic4" / "X_folded.parquet"
    else:
        fp = DATA_ROOT / db_name / "X_external.parquet"
    X_f = pd.read_parquet(fp)
    stay_ids = X_f["stay_id"].values
    X_df = X_f.drop(columns=[c for c in ["stay_id", "fold_id"] if c in X_f.columns])
    X_df = X_df.drop(columns=[c for c in DROP_COLS if c in X_df.columns])
    return X_df, stay_ids


def _load_y(db_name):
    fp = DATA_ROOT / db_name / f"binary_outcomes_{db_name}.csv"
    return pd.read_csv(fp)


def split_56_14_30(y):
    """70/30 stratified split, then the 70 part is split 80/20 stratified. Train 56, validation 14, test 30."""
    idx_all = np.arange(len(y))
    idx_tr70, idx_te, _, _ = train_test_split(idx_all, y, test_size=0.30, stratify=y, random_state=SEED)
    idx_tr, idx_va, _, _ = train_test_split(idx_tr70, y[idx_tr70], test_size=0.20, stratify=y[idx_tr70], random_state=SEED)
    return idx_tr, idx_va, idx_te


def load_mimic4(outcome="AKI_168h"):
    X_df, stay_ids = _load_x("mimic4")
    y_df = _load_y("mimic4")
    y_col = OUTCOME_MAP[outcome]
    merged = pd.DataFrame({"stay_id": stay_ids}).merge(y_df[["stay_id", y_col]], on="stay_id", how="left")
    n_miss = int(merged[y_col].isna().sum())                     # the number of missing outcomes is counted; they are filled with 0
    if n_miss:
        print(f"  [warning] MIMIC-IV {outcome}: stays without outcome {n_miss} rows were filled with 0 (missing outcomes are filled with 0).")
    y = merged[y_col].fillna(0).values.astype(int)
    cat_cols, num_cols = _detect_cols(X_df)
    idx_tr, idx_va, idx_te = split_56_14_30(y)
    return {"X_df": X_df, "y": y, "stay_ids": stay_ids,
            "idx_train": idx_tr, "idx_val": idx_va, "idx_test": idx_te,
            "cat_cols": cat_cols, "num_cols": num_cols,
            "name": "MIMIC-IV", "outcome": outcome, "n_outcome_missing": n_miss}


def load_external(db_name, outcome):
    X_ref, _ = _load_x("mimic4")
    cat_cols, num_cols = _detect_cols(X_ref)
    X_df, stay_ids = _load_x(db_name)
    y_df = _load_y(db_name)
    y_col = OUTCOME_MAP[outcome]
    X_aligned = pd.DataFrame(index=range(len(stay_ids)))
    for col in cat_cols + num_cols:
        X_aligned[col] = X_df[col].values if col in X_df.columns else np.nan
    merged = X_aligned.copy()
    merged["stay_id"] = stay_ids
    merged = merged.merge(y_df[["stay_id", y_col]], on="stay_id", how="inner")
    y = merged[y_col].values.astype(int)
    stay_out = merged["stay_id"].values                          # stay_id is kept after the merge (for linking to time and hospital)
    X_out = merged.drop(columns=["stay_id", y_col])
    missing_cols = [c for c in cat_cols + num_cols if c not in X_df.columns]
    if missing_cols:
        _log(f"  {db_name}: structurally missing -> {missing_cols}")
    display_name = {"mimic3": "MIMIC-III", "eicu": "eICU"}[db_name]
    return {"X_df": X_out, "y": y, "stay_ids": stay_out, "cat_cols": cat_cols, "num_cols": num_cols,
            "name": display_name, "outcome": outcome}


# ---- retrieval and helper functions ----------------
def retrieval_R1_cosine(c, k=K_NEIGHBORS):
    k_ = min(k, len(c["X_tr_s"]))
    if k_ < 1:
        return np.full(len(c["X_te_s"]), 0.5)
    nn = NearestNeighbors(n_neighbors=k_, metric="cosine", algorithm="brute", n_jobs=-1)
    nn.fit(c["X_tr_s"])
    d, idx = nn.kneighbors(c["X_te_s"])
    w = 1.0 / (d + 1e-6)
    return (w * c["y_tr"][idx]).sum(1) / w.sum(1)


RETRIEVAL_FN = retrieval_R1_cosine


def compute_ece(y, p, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (p >= lo) & (p < hi)
        if m.sum() == 0:
            continue
        ece += m.sum() * abs(y[m].mean() - p[m].mean())
    return ece / len(y)


def _logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


# ---- DeLong -----------------------------------------------
def _midrank(x):
    """Compute midranks for tied values (scipy implementation; gives the same values as a loop implementation)."""
    return scipy_stats.rankdata(np.asarray(x, dtype=float), method="average")


def _delong_core_original(preds_sorted_T, m):
    """The DeLong core that uses np.arange in place of the within-class midranks (the original core). Defect: if the scores within a class
    are not in sorted order, the variance of the paired AUROC difference is overestimated. It is correct only when sorted by score within each class.
    All analyses of this notebook use `_delong_core` below; this function is called only by diagnostic cells that give the original variance for comparison."""
    n = preds_sorted_T.shape[1] - m
    k = preds_sorted_T.shape[0]
    aucs = np.zeros(k)
    v01 = np.zeros((k, m))
    v10 = np.zeros((k, n))
    for i in range(k):
        tx = _midrank(preds_sorted_T[i, :])
        aucs[i] = (np.sum(tx[:m]) - m * (m + 1) / 2.0) / (m * n)
        v10[i] = (tx[m:] - np.arange(1, n + 1)) / m
        v01[i] = 1.0 - (tx[:m] - np.arange(1, m + 1)) / n
    if m > 1:
        S01 = np.cov(v01) if k > 1 else np.array([[np.var(v01[0], ddof=1)]])
    else:
        S01 = np.zeros((k, k))
    if n > 1:
        S10 = np.cov(v10) if k > 1 else np.array([[np.var(v10[0], ddof=1)]])
    else:
        S10 = np.zeros((k, k))
    S = S01 / m + S10 / n
    return aucs, S


def _delong_core(preds_sorted_T, m):
    """Standard fastDeLong (Sun and Xu 2014). Builds the structural components from midranks within positives and negatives (tx, ty) and overall midranks (tz).
    preds_sorted_T is (k, n_total), positives first. Returns (AUROCs, covariance matrix). An internal check compares with the O(mn) definition and with a bootstrap."""
    n = preds_sorted_T.shape[1] - m
    k = preds_sorted_T.shape[0]
    pos, neg = preds_sorted_T[:, :m], preds_sorted_T[:, m:]
    tx = np.array([_midrank(pos[i]) for i in range(k)])
    ty = np.array([_midrank(neg[i]) for i in range(k)])
    tz = np.array([_midrank(preds_sorted_T[i]) for i in range(k)])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    S01 = np.cov(v01) if k > 1 else np.array([[np.var(v01[0], ddof=1)]])
    S10 = np.cov(v10) if k > 1 else np.array([[np.var(v10[0], ddof=1)]])
    return aucs, S01 / m + S10 / n


# ---- mixing-weight selection within a bank and stratified draw ----------------------
def inner_cv_bank(X_bank, y_bank, p_base_bank, r_fn, n_splits=N_INNER, key=""):
    """Select (a) alpha and (b) produce OOF PRAM predictions by cross-validation inside the bank. OOF predictions avoid self-retrieval.
    compute_mi is not called, because R1 cosine does not read mutual information."""
    n = len(y_bank)
    if n < n_splits * 5 or len(np.unique(y_bank)) < 2:
        return {"alpha": 0.0, "oof_pram": p_base_bank.copy(), "ok": False}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                          random_state=stable_seed("innercv", key))
    scores = {a: [] for a in ALPHA_GRID}
    oof_retr = np.full(n, np.nan)
    for itr, iva in skf.split(np.zeros(n), y_bank):
        mi_i, mp_i = None, None
        pr = r_fn({"X_tr_s": X_bank[itr], "y_tr": y_bank[itr],
                   "X_te_s": X_bank[iva], "y_te": y_bank[iva],
                   "mi": mi_i, "mi_prob": mp_i})
        oof_retr[iva] = pr
        if len(np.unique(y_bank[iva])) > 1:
            for a in ALPHA_GRID:
                scores[a].append(roc_auc_score(y_bank[iva],
                                               (1 - a) * p_base_bank[iva] + a * pr))
    best_a = max(ALPHA_GRID, key=lambda a: np.nanmean(scores[a]) if scores[a] else -1)
    oof_retr = np.where(np.isnan(oof_retr), p_base_bank, oof_retr)
    return {"alpha": best_a, "oof_pram": (1 - best_a) * p_base_bank + best_a * oof_retr,
            "ok": True}


def sample_bank(pool_y, size, key):
    """stratified draw. Uses round()."""
    r = rng_of("bank", key)
    if size == "full":
        return np.arange(len(pool_y))
    prev = pool_y.mean()
    n_pos = max(1, int(round(size * prev)))
    n_neg = max(1, size - n_pos)
    pi, ni = np.where(pool_y == 1)[0], np.where(pool_y == 0)[0]
    sp = r.choice(pi, min(n_pos, len(pi)), replace=len(pi) < n_pos)
    sn = r.choice(ni, min(n_neg, len(ni)), replace=len(ni) < n_neg)
    return np.sort(np.concatenate([sp, sn]))


# ---- anchored ridge -------------------------------------------
def _anchored_ridge(X, y, w_src, b_src, lam, anchor_intercept=True, maxiter=500):
    """minimizes mean log loss + lam*||w - w_src||^2. Returns (w, b, ||u||, converged)"""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, d = X.shape
    off = X @ w_src + b_src

    def fun(t):
        u = t[:d]
        db = 0.0 if anchor_intercept else t[d]
        z = off + X @ u + db
        loss = float(np.mean(np.logaddexp(0.0, z) - y * z)) + lam * float(u @ u)
        g = (1.0 / (1.0 + np.exp(-z)) - y) / n
        gu = X.T @ g + 2.0 * lam * u
        return (loss, gu) if anchor_intercept else (loss, np.concatenate([gu, [g.sum()]]))

    t0 = np.zeros(d if anchor_intercept else d + 1)
    r = minimize(fun, t0, jac=True, method="L-BFGS-B",
                 options={"maxiter": maxiter, "ftol": 1e-12, "gtol": 1e-9})
    u = r.x[:d]
    db = 0.0 if anchor_intercept else float(r.x[d])
    return w_src + u, b_src + db, float(np.sqrt(u @ u)), bool(r.success)


def _pick_lam(Xb, yb, w_src, b_src, anchor_intercept, key, n_splits=N_INNER):
    """lam is chosen within the bank only."""
    if len(np.unique(yb)) < 2 or len(yb) < n_splits * 5:
        return float(LAM_GRID[-1])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                          random_state=stable_seed("a53", key))
    scores = {float(l): [] for l in LAM_GRID}
    for itr, iva in skf.split(np.zeros(len(yb)), yb):
        if len(np.unique(yb[iva])) < 2:
            continue
        for l in LAM_GRID:
            w, b, _, _ = _anchored_ridge(Xb[itr], yb[itr], w_src, b_src,
                                         float(l), anchor_intercept, maxiter=200)
            scores[float(l)].append(roc_auc_score(yb[iva], Xb[iva] @ w + b))
    best = max(scores, key=lambda l: np.nanmean(scores[l]) if scores[l] else -1)
    return float(best)


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))


## New functions

Every analysis of this notebook calls one engine (`bank_predictions`) that returns the test predictions of the strategies for one bank and one set of test patients, so the main analysis, temporal validation, hospital level and tail risk share the same code.

| Definition | What |
|---|---|
| `full_metrics` | AUROC, AUPRC, Brier score, ECE, calibration slope and intercept, and net benefit over the full 1 to 50% threshold grid. The slope is an unpenalized Newton fit |
| `update_hierarchy` | update steps 0 to 3 and closed testing, on the frozen logistic model only: intercept only (1, maximum likelihood), intercept and slope (2, unpenalized), model revision (3). The test is a likelihood-ratio test of the unpenalized revision; when step 3 is chosen, the prediction is the **anchored ridge** with the penalty chosen within the bank. The order of the closed-testing steps follows Vergouwe et al. (2017) |
| `bank_predictions` | test predictions per strategy group for one bank and one set of test patients |
| `run_tail_unit` | tail risk: one (bank, test fold) pair; 5 folds x 20 draws per fold |
| `run_temporal` | temporal validation: the test period is split into 2 folds that alternate as test; the past bank (P) and the same-period bank (S) are compared at the same size |
| `hospital_oof`, `hospital_unit` | one hospital, 3 folds |
| `network_unit` | an unlabeled hospital receives the mean probability of its partner hospitals |
| `meta_reml_hk` | REML tau squared, Hartung-Knapp interval (untruncated and truncated forms), prediction interval, inverse-variance I squared, Q and degrees of freedom |
| `delong_delta_var`, `_delong_core` | paired AUROC difference and its variance (standard fastDeLong, Sun and Xu 2014). Checked against the O(mn) definition and a bootstrap |


In [ ]:
# ================================================================
# PRAM Cell 3. Newly written definitions (reporting, metrics, update hierarchy)
# ================================================================
try:
    from IPython.display import display, Markdown
    from IPython import get_ipython as _get_ipy
    _IN_IPY = _get_ipy() is not None
except Exception:  # noqa: BLE001
    _IN_IPY = False

REPORT_LINES = []                       # every table shown on screen is also accumulated in the result md


def emit(text=""):
    REPORT_LINES.append(str(text))
    if _IN_IPY:
        display(Markdown(str(text)))
    else:
        print(text)


def md_table(df, digits=4, max_rows=80):
    if df is None or len(df) == 0:
        return "(empty)"
    d = df.head(max_rows)
    cols = [str(c) for c in d.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in d.itertuples(index=False):
        cells = []
        for v in r:
            if isinstance(v, (float, np.floating)):
                cells.append("" if not np.isfinite(v) else f"{v:.{digits}f}")
            else:
                cells.append(str(v))
        lines.append("| " + " | ".join(cells) + " |")
    if len(df) > max_rows:
        lines.append(f"... {len(df) - max_rows} rows omitted (full table in csv)")
    return "\n".join(lines)


def emit_table(title, df, digits=4, max_rows=80):
    emit("\n**" + title + "**\n")
    emit(md_table(df, digits=digits, max_rows=max_rows))


def save_csv(df, name):
    path = CSV_DIR / (name + ".csv")
    df.to_csv(path, index=False, encoding="utf-8-sig", float_format="%.6g")
    emit(f"- saved `{path.as_posix()}` ({len(df)}rows)")
    return path


# ---- metrics ------------------------------------------------------------------
NB_COLS = [f"nb@{float(t)}" for t in THRESH_GRID]


def _newton_2(y, x, iters=60):
    """unpenalized maximum likelihood of p = sigmoid(a + b x). Starts at (0, 1). Returns (nan, nan) on failure."""
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    if (not np.isfinite(x).all()) or x.std() < 1e-9 or len(np.unique(y)) < 2:
        return np.nan, np.nan

    def nll(a_, b_):
        z = a_ + b_ * x
        return float(np.sum(np.logaddexp(0.0, z) - y * z))

    a, b = 0.0, 1.0
    cur = nll(a, b)
    for _ in range(iters):
        pr = _sigmoid(a + b * x)
        w = pr * (1 - pr) + 1e-12
        g = np.array([np.sum(y - pr), np.sum((y - pr) * x)])
        H = np.array([[w.sum(), (w * x).sum()], [(w * x).sum(), (w * x * x).sum()]]) + 1e-10 * np.eye(2)
        step = np.linalg.solve(H, g)
        t = 1.0
        while t > 1e-4 and nll(a + t * step[0], b + t * step[1]) > cur + 1e-12:
            t *= 0.5
        a, b = a + t * step[0], b + t * step[1]
        new = nll(a, b)
        done = abs(cur - new) < 1e-10 or max(abs(t * step[0]), abs(t * step[1])) < 1e-9
        cur = new
        if done:
            break
    return float(a), float(b)


def calib_slope_intercept(y, p):
    """(slope, intercept). Unpenalized calibration line on the logit."""
    a, b = _newton_2(y, _logit(np.asarray(p, dtype=float)))
    return b, a


def nb_curve(y, p, grid=THRESH_GRID):
    """net benefit over the whole threshold grid. One sort counts all thresholds."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    n = len(y)
    order = np.argsort(-p, kind="stable")
    ps, ys = p[order], y[order]
    ctp = np.cumsum(ys)
    cfp = np.cumsum(1 - ys)
    k = np.searchsorted(-ps, -np.asarray(grid, dtype=float), side="right")      # number of patients with p >= t
    tp = np.where(k > 0, ctp[np.maximum(k - 1, 0)], 0)
    fp = np.where(k > 0, cfp[np.maximum(k - 1, 0)], 0)
    g = np.asarray(grid, dtype=float)
    return tp / n - fp / n * (g / (1 - g))


def full_metrics(y, p, orig_slope=False):
    """if orig_slope=True, adds the slope of a scikit-learn C=1e6 logistic fit as slope_orig. For reproduction checks only."""
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    out = {}
    try:
        out["auroc"] = float(roc_auc_score(y, p))
        out["auprc"] = float(average_precision_score(y, p))
        out["brier"] = float(brier_score_loss(y, p))
    except ValueError:
        out["auroc"] = out["auprc"] = out["brier"] = np.nan
    out["ece"] = float(compute_ece(y, p))
    sl, ic = calib_slope_intercept(y, p)
    out["slope"], out["intercept"] = sl, ic
    if orig_slope:
        lp = _logit(p)
        out["slope_orig"] = float(LogisticRegression(max_iter=1000, C=1e6).fit(lp.reshape(-1, 1), y).coef_[0][0])             if (np.isfinite(lp).all() and lp.std() > 1e-9 and len(np.unique(y)) > 1) else np.nan
    out["mean_pred"] = float(p.mean())
    out["obs_rate"] = float(y.mean())
    for c, v in zip(NB_COLS, nb_curve(y, p)):
        out[c] = float(v)
    return out


def delong_delta_var(y, pa, pb, original=False):
    """paired DeLong. (AUROC_a, AUROC_b, difference, variance of the difference). nan if one class is empty. If original=True, the original implementation (for diagnosis)."""
    y = np.asarray(y).astype(int)
    if len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan, np.nan
    order = np.argsort(y)[::-1]
    m = int(y.sum())
    if m < 2 or len(y) - m < 2:
        return np.nan, np.nan, np.nan, np.nan
    preds = np.vstack([np.asarray(pa, dtype=float)[order], np.asarray(pb, dtype=float)[order]])
    aucs, S = (_delong_core_original if original else _delong_core)(preds, m)
    var = float(S[0, 0] + S[1, 1] - 2 * S[0, 1])
    return float(aucs[0]), float(aucs[1]), float(aucs[0] - aucs[1]), max(var, 0.0)


def cp_interval(k, n, level=CP_LEVEL):
    """Clopper-Pearson exact interval."""
    if n <= 0:
        return np.nan, np.nan
    a = 1 - level
    lo = 0.0 if k == 0 else scipy_stats.beta.ppf(a / 2, k, n - k + 1)
    hi = 1.0 if k == n else scipy_stats.beta.ppf(1 - a / 2, k + 1, n - k)
    return float(lo), float(hi)


def area_under_curve(grid, vals, lo, hi):
    """trapezoid area within [lo, hi]."""
    g = np.asarray(grid, dtype=float)
    v = np.asarray(vals, dtype=float)
    m = (g >= lo - 1e-9) & (g <= hi + 1e-9)
    if m.sum() < 2:
        return np.nan
    trap = getattr(np, "trapezoid", None) or np.trapz
    return float(trap(v[m], g[m]))


# ---- update hierarchy and closed testing (A4) ---------------------------------------------
def _nll_sum(y, z):
    return float(np.sum(np.logaddexp(0.0, z) - y * z))


def fit_intercept_only(y, off):
    """re-estimates the intercept only (recalibration-in-the-large). The slope is fixed at 1."""
    a = 0.0
    for _ in range(60):
        pr = _sigmoid(off + a)
        step = float(np.sum(y - pr) / (np.sum(pr * (1 - pr)) + 1e-12))
        step = max(-5.0, min(5.0, step))
        a += step
        if abs(step) < 1e-10:
            break
    return a


def fit_revision(X, y, off, ridge=REVISION_RIDGE):
    """model revision: the frozen linear predictor is an offset; the intercept and all coefficients are re-estimated. The tiny regularization is for numerical stability."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, d = X.shape

    def fun(t):
        a, u = t[0], t[1:]
        z = off + a + X @ u
        loss = float(np.mean(np.logaddexp(0.0, z) - y * z)) + ridge * float(u @ u)
        g = (_sigmoid(z) - y) / n
        return loss, np.concatenate([[g.sum()], X.T @ g + 2.0 * ridge * u])

    r = minimize(fun, np.zeros(d + 1), jac=True, method="L-BFGS-B",
                 options={"maxiter": 300, "ftol": 1e-12, "gtol": 1e-8})
    return float(r.x[0]), r.x[1:].copy(), bool(r.success)


def update_hierarchy(Xb, yb, offb, Xt, offt, p_frozen_t, p_rev_alt=None):
    """fits update steps 0 to 3 in the bank and chooses one step by closed testing.
    0 no update | 1 intercept only | 2 intercept + slope | 3 model revision. Closed testing (likelihood ratio, significance level CLOSED_TEST_ALPHA):
    (1) if revision versus no update (d+1 degrees of freedom) is not significant, 0. (2) if significant and revision versus step 2 (d-1 degrees of freedom) is significant, 3.
    (3) otherwise, if step 2 versus step 1 (1 degree of freedom) is significant, 2, otherwise 1.
    if p_rev_alt is given, its prediction (anchored ridge with the penalty chosen within the bank) is used when step 3 is chosen. The test always uses the likelihood of the unpenalized revision."""
    res = {"upd1": p_frozen_t, "upd2": p_frozen_t, "upd3": p_frozen_t, "ct": p_frozen_t,
           "step": 0, "p_all": np.nan, "p_rev_vs_recal": np.nan, "p_slope": np.nan,
           "slope2": np.nan, "ok": False}
    yb = np.asarray(yb).astype(float)
    n1 = int(yb.sum())
    n0 = int(len(yb) - n1)
    if min(n1, n0) < N_EVENT_MIN or not np.isfinite(offb).all() or not np.isfinite(offt).all():
        return res
    d = Xb.shape[1]
    ll0 = -_nll_sum(yb, offb)
    a1 = fit_intercept_only(yb, offb)
    ll1 = -_nll_sum(yb, offb + a1)
    a2, b2 = _newton_2(yb, offb)
    if not np.isfinite(a2):
        return res
    ll2 = -_nll_sum(yb, a2 + b2 * offb)
    a3, u3, _ok3 = fit_revision(Xb, yb, offb)
    ll3 = -_nll_sum(yb, offb + a3 + Xb @ u3)
    res["upd1"] = _sigmoid(offt + a1)
    res["upd2"] = _sigmoid(a2 + b2 * offt)
    res["upd3"] = _sigmoid(offt + a3 + Xt @ u3)
    res["slope2"] = float(b2)
    lr_all = max(0.0, 2 * (ll3 - ll0))
    lr_rev = max(0.0, 2 * (ll3 - ll2))
    lr_slope = max(0.0, 2 * (ll2 - ll1))
    res["p_all"] = float(scipy_stats.chi2.sf(lr_all, d + 1))
    res["p_rev_vs_recal"] = float(scipy_stats.chi2.sf(lr_rev, max(d - 1, 1)))
    res["p_slope"] = float(scipy_stats.chi2.sf(lr_slope, 1))
    if res["p_all"] >= CLOSED_TEST_ALPHA:
        step = 0
    elif res["p_rev_vs_recal"] < CLOSED_TEST_ALPHA:
        step = 3
    elif res["p_slope"] < CLOSED_TEST_ALPHA:
        step = 2
    else:
        step = 1
    res["step"] = step
    res["ct"] = [p_frozen_t, res["upd1"], res["upd2"], res["upd3"] if p_rev_alt is None else p_rev_alt][step]
    res["ok"] = True
    return res


# ---- auxiliary fits -----------------------------------------------------------
def platt_sklearn(p_fit, y_fit, p_apply):
    """Platt scaling: logistic regression with default regularization on the logit."""
    lr_p = LogisticRegression(max_iter=1000, random_state=SEED)
    lr_p.fit(_logit(p_fit).reshape(-1, 1), y_fit)
    return lr_p.predict_proba(_logit(p_apply).reshape(-1, 1))[:, 1]


def _logloss(y, p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def inner_oof_candidates(Xb, yb, p1b, p5b, icv1, icv5, key, cand):
    """A11: builds the within-bank predictions of each candidate strategy by inner cross-validation. Only strategies that a hospital can actually choose are candidates."""
    out = {}
    if "b1_base" in cand:
        out["b1_base"] = p1b
    if "b5_base" in cand:
        out["b5_base"] = p5b
    if "pram_b1" in cand:
        out["pram_b1"] = icv1["oof_pram"]
    if "pram_b5" in cand:
        out["pram_b5"] = icv5["oof_pram"]
    need_fit = [c for c in ("platt_b1", "recal_iso", "local_lr") if c in cand]
    if need_fit:
        oofs = {c: p1b.copy() for c in need_fit}
        skf = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=stable_seed("selcv", key))
        for itr, iva in skf.split(np.zeros(len(yb)), yb):
            if len(np.unique(yb[itr])) < 2:
                continue
            if "platt_b1" in oofs:
                oofs["platt_b1"][iva] = platt_sklearn(p1b[itr], yb[itr], p1b[iva])
            if "recal_iso" in oofs:
                iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(p1b[itr], yb[itr])
                oofs["recal_iso"][iva] = iso.predict(p1b[iva])
            if "local_lr" in oofs:
                lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xb[itr], yb[itr])
                oofs["local_lr"][iva] = lr.predict_proba(Xb[iva])[:, 1]
        out.update(oofs)
    return out


# ---- strategy engine -----------------------------------------------------------------
def bank_predictions(C, bank, test, key, groups, cb_iter=None):
    """test predictions of the strategies fitted on one bank (row indices). Same rule as run_matrix_unit.
    C is the target cohort context (build_ctx). groups: frozen pram recal local localcb anch hier combo sel."""
    cb_iter = LOCAL_CB_ITER if cb_iter is None else cb_iter
    Xg, y, pb1, pb5 = C["Xg"], C["y"], C["pb1"], C["pb5"]
    Xb, yb, p1b, p5b = Xg[bank], y[bank], pb1[bank], pb5[bank]
    Xt, yt, p1t, p5t = Xg[test], y[test], pb1[test], pb5[test]
    P = {"b1_base": p1t, "b5_base": p5t}
    info = {}
    two = len(np.unique(yb)) > 1
    icv1 = icv5 = p_retr = None

    if "pram" in groups or "combo" in groups or "sel" in groups:
        p_retr = RETRIEVAL_FN({"X_tr_s": Xb, "y_tr": yb, "X_te_s": Xt, "y_te": yt, "mi": None, "mi_prob": None})
        icv1 = inner_cv_bank(Xb, yb, p1b, RETRIEVAL_FN, key="b1|" + key)
        icv5 = inner_cv_bank(Xb, yb, p5b, RETRIEVAL_FN, key="b5|" + key)
        P["pram_b1"] = (1 - icv1["alpha"]) * p1t + icv1["alpha"] * p_retr
        P["pram_b5"] = (1 - icv5["alpha"]) * p5t + icv5["alpha"] * p_retr
        info["alpha_b1"], info["alpha_b5"] = icv1["alpha"], icv5["alpha"]

    if "recal" in groups:
        if two:
            iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(p1b, yb)
            P["recal_iso"] = iso.predict(p1t)
            P["platt_b1"] = platt_sklearn(p1b, yb, p1t)
            P["platt_b5"] = platt_sklearn(p5b, yb, p5t)          # CatBoost gets Platt only (the update hierarchy is defined on the logistic model)
        else:
            P["recal_iso"], P["platt_b1"], P["platt_b5"] = p1t, p1t, p5t

    if "local" in groups:
        if two:
            lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xb, yb)
            P["local_lr"] = lr.predict_proba(Xt)[:, 1]
            pl = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
            pl.fit(np.vstack([C["Xs_src"], Xb]), np.concatenate([C["ys_src"], yb]))
            P["pooled_lr"] = pl.predict_proba(Xt)[:, 1]
        else:
            P["local_lr"], P["pooled_lr"] = p1t, p1t

    if "localcb" in groups:
        if two:
            cb = CatBoostClassifier(iterations=cb_iter, depth=6, learning_rate=0.05, verbose=0, random_seed=SEED)
            cb.fit(C["Xc"].iloc[bank], yb, cat_features=C["cb_cat_idx"])
            P["local_cb"] = cb.predict_proba(C["Xc"].iloc[test])[:, 1]
        else:
            P["local_cb"] = p1t

    if "anch" in groups and two and (ANCHORED_MAX_BANK is None or len(bank) <= ANCHORED_MAX_BANK):
        lam = _pick_lam(Xb, yb, C["w_src"], C["b_src"], True, "anch|" + key)
        w_a, b_a, _, _ = _anchored_ridge(Xb, yb, C["w_src"], C["b_src"], lam, True)
        P["anchored_ridge"] = _sigmoid(Xt @ w_a + b_a)
        info["anch_lam"] = lam

    if "hier" in groups:                       # the update hierarchy only on the frozen logistic model (for CatBoost see platt_b5 above)
        u = update_hierarchy(Xb, yb, _logit(p1b), Xt, _logit(p1t), p1t, p_rev_alt=P.get("anchored_ridge"))
        P["upd1_b1"], P["upd2_b1"], P["upd3_b1"], P["ct_b1"] = u["upd1"], u["upd2"], u["upd3"], u["ct"]
        info["ct_step_b1"] = u["step"]
        info["ct_p_all_b1"] = u["p_all"]
        info["ct_p_rev_vs_recal_b1"] = u["p_rev_vs_recal"]
        info["slope2_b1"] = u["slope2"]

    if "combo" in groups and two:
        # retrieval -> Platt: Platt is fitted on the inner cross-validation OOF mixture predictions in the bank and applied to the test mixture prediction 
        P["pram_platt_b1"] = platt_sklearn(icv1["oof_pram"], yb, P["pram_b1"])
        P["pram_platt_b5"] = platt_sklearn(icv5["oof_pram"], yb, P["pram_b5"])
    elif "combo" in groups:
        P["pram_platt_b1"], P["pram_platt_b5"] = p1t, p5t

    if "sel" in groups:
        cand = [c for c in SEL_CANDIDATES if c in P]
        oof_c = inner_oof_candidates(Xb, yb, p1b, p5b, icv1, icv5, key, cand)
        cand = [c for c in cand if c in oof_c]
        t_site = T_SITE[C["outcome"]]
        ll = {c: _logloss(yb, oof_c[c]) for c in cand}
        nbv = {c: float(nb_curve(yb, oof_c[c], np.array([t_site]))[0]) for c in cand}
        pick_ll = min(cand, key=lambda c: ll[c])
        pick_nb = max(cand, key=lambda c: nbv[c])
        P["sel_logloss"], P["sel_nb"] = P[pick_ll], P[pick_nb]
        info["sel_logloss"], info["sel_nb"] = pick_ll, pick_nb
    return P, info


def run_matrix_unit(C, fs, outcome, target, bank_size, n_draws, groups, key_tag="m23", keep=None, delong_pairs=None, orig_slope=False):
    """one unit (feature set, outcome, target, bank size). For each draw (rep), the 5 folds are pooled and the metrics are computed once per bank result (pooled).
    the draw keys use key_tag m23, so the same patients are drawn in every run.
    if keep is given, only those strategies get metrics. If delong_pairs={strategy: reference}, the paired DeLong difference and its variance for that strategy are attached to the row."""
    y = C["y"]
    n = len(y)
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(n), y))
    n_rep = 1 if bank_size == "full" else n_draws
    rows, info_rows = [], []
    for rep in range(n_rep):
        oof = {}
        for fi, (pool_idx, test_idx) in enumerate(folds):
            key = f"{outcome}|{target}|{bank_size}|{fi}|{rep}"
            if bank_size == "full":
                sel = np.arange(len(pool_idx))
            else:
                sel = sample_bank(y[pool_idx], bank_size, key_tag + "|" + key)
            bank = pool_idx[sel]
            P, info = bank_predictions(C, bank, test_idx, key, set(groups))
            for m, v in P.items():
                if m not in oof:
                    oof[m] = np.full(n, np.nan, dtype=np.float32)
                oof[m][test_idx] = v
            info_rows.append({"fs": fs, "outcome": outcome, "target": target, "bank": str(bank_size),
                              "rep": rep, "fold": fi, "n_bank": int(len(bank)),
                              "events_bank": int(y[bank].sum()), **info})
        for m, pm in oof.items():
            if np.isnan(pm).any() or (keep is not None and m not in keep):
                continue
            row = {"fs": fs, "outcome": outcome, "target": target, "bank": str(bank_size),
                   "rep": rep, "method": m, **full_metrics(y, pm.astype(float), orig_slope=orig_slope)}
            if delong_pairs and m in delong_pairs and delong_pairs[m] in oof:
                _a, _b, dd, vv = delong_delta_var(y, pm.astype(float), oof[delong_pairs[m]].astype(float))
                row["d_auroc_base"], row["var_auroc_base"] = dd, vv
            rows.append(row)
    return rows, info_rows


# ---- development model and target context -------------------------------------------------------
def build_dev(X_train_df, y_train, cols, cat_ref, num_ref, tag):
    """fits the two frozen models (logistic and CatBoost) and the preprocessor on the development training data only."""
    cc = [c for c in cat_ref if c in cols]
    nc = [c for c in num_ref if c in cols]
    X_train_df = X_train_df.reset_index(drop=True)
    y_train = np.asarray(y_train).astype(int)
    pg = PreprocessorGeneral(cc, nc)
    Xtr_g = pg.fit_transform(X_train_df)
    pc = PreprocessorCatBoost(cc, nc)
    Xtr_c = pc.fit_transform(X_train_df)
    m1 = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xtr_g, y_train)
    m5 = CatBoostClassifier(iterations=DEV_CB_ITER, depth=6, learning_rate=0.05, verbose=0, random_seed=SEED)
    m5.fit(Xtr_c, y_train, cat_features=pc.cat_feature_indices)
    return {"pg": pg, "pc": pc, "m1": m1, "m5": m5, "Xs_src": Xtr_g, "ys_src": y_train,
            "cb_cat_idx": pc.cat_feature_indices, "cols": cc + nc, "tag": tag,
            "w_src": np.asarray(m1.coef_).ravel().astype(float), "b_src": float(np.asarray(m1.intercept_).ravel()[0])}


def make_ctx(dev, X_df, y, stay_ids, outcome):
    """moves the target cohort through the development preprocessor and attaches the frozen model outputs."""
    X_df = X_df.reset_index(drop=True)
    Xg = dev["pg"].transform(X_df)
    Xc = dev["pc"].transform(X_df)
    return {"Xg": Xg, "Xc": Xc, "y": np.asarray(y).astype(int),
            "pb1": dev["m1"].predict_proba(Xg)[:, 1], "pb5": dev["m5"].predict_proba(Xc)[:, 1],
            "stay_ids": np.asarray(stay_ids), "outcome": outcome,
            "Xs_src": dev["Xs_src"], "ys_src": dev["ys_src"], "cb_cat_idx": dev["cb_cat_idx"],
            "w_src": dev["w_src"], "b_src": dev["b_src"], "d": int(Xg.shape[1])}


# ---- hospital id and discharge year ---------------------------------------------------------
def load_eicu_patient_table():
    """reads only the stay, hospital and discharge year columns of the patient table (those that exist)."""
    cols = [c for c in (EICU_COL_STAY, EICU_COL_HOSP, EICU_COL_YEAR) if c]
    df = pd.read_csv(EICU_PATIENT_PATH, usecols=cols)
    ren = {EICU_COL_STAY: "stay_id"}
    if EICU_COL_HOSP:
        ren[EICU_COL_HOSP] = "hospitalid"
    if EICU_COL_YEAR:
        ren[EICU_COL_YEAR] = "year"
    df = df.rename(columns=ren)
    df["stay_id"] = pd.to_numeric(df["stay_id"], errors="coerce")
    df = df.dropna(subset=["stay_id"]).drop_duplicates("stay_id")
    df["stay_id"] = df["stay_id"].astype("int64")
    return df.set_index("stay_id")


def map_to_stays(table, col, stay_ids):
    s = pd.to_numeric(pd.Series(np.asarray(stay_ids)), errors="coerce")
    return s.map(table[col]).to_numpy(dtype=float)


def eligible_sites(y, hid):
    """eligible hospitals: at least MIN_SITE_N stays, at least MIN_SITE_EVENTS events and non-events each. Largest hospitals first."""
    keep = []
    for h in pd.unique(hid[pd.notna(hid)]):
        m = (hid == h)
        if m.sum() >= MIN_SITE_N and y[m].sum() >= MIN_SITE_EVENTS and (m.sum() - y[m].sum()) >= MIN_SITE_EVENTS:
            keep.append(int(h))
    keep.sort(key=lambda h: -int((hid == h).sum()))
    return keep


# ---- hospital level ----------------------------
def hospital_oof(C, outcome, h, hid, groups):
    """one set of 3-fold test predictions (OOF) of one hospital and the per-fold event counts. (None, None, None) if not eligible."""
    y = C["y"]
    idx = np.where(hid == h)[0]
    ys = y[idx]
    n_sp = min(N_SITE_SPLITS, int(ys.sum()), int(len(ys) - ys.sum()))
    if n_sp < 2:
        return None, None, None
    skf = StratifiedKFold(n_splits=n_sp, shuffle=True, random_state=stable_seed("a29", outcome, int(h)))
    oof, fold_rows = {}, []
    for fi, (bank_i, test_i) in enumerate(skf.split(np.zeros(len(ys)), ys)):
        bank, test = idx[bank_i], idx[test_i]
        yb, yte = y[bank], y[test]
        if len(np.unique(yb)) < 2 or len(np.unique(yte)) < 2:
            return None, None, None
        P, info = bank_predictions(C, bank, test, f"a29|{outcome}|{h}|{fi}", set(groups))
        for m, v in P.items():
            if m not in oof:
                oof[m] = np.full(len(ys), np.nan)
            oof[m][test_i] = v
        fold_rows.append({"outcome": outcome, "hospitalid": int(h), "fold": fi,
                          "n_test": int(len(test)), "events_test": int(yte.sum()),
                          "n_bank": int(len(bank)), "events_bank": int(yb.sum()), **info})
    return ys, oof, fold_rows


def hospital_unit(C, outcome, h, hid, groups):
    ys, oof, fold_rows = hospital_oof(C, outcome, h, hid, groups)
    if ys is None:
        return None, None
    rows = []
    base1, base5 = oof["b1_base"], oof["b5_base"]
    for m, v in oof.items():
        if np.isnan(v).any():
            continue
        _a, _b, d1, v1 = delong_delta_var(ys, v, base1)
        _a, _b, d5, v5 = delong_delta_var(ys, v, base5)
        rows.append({"outcome": outcome, "hospitalid": int(h), "method": m, "n_site": int(len(ys)),
                     "n_events": int(ys.sum()), "prev": float(ys.mean()), **full_metrics(ys, v),
                     "d_auroc_b1": d1, "var_b1": v1, "d_auroc_b5": d5, "var_b5": v5})
    return rows, fold_rows


# ---- network of unlabeled hospitals ------
def _topk_cos(Xb, yb, Xq, k):
    kk = int(min(k, len(Xb)))
    nn = NearestNeighbors(n_neighbors=kk, metric="cosine").fit(Xb)
    d, i = nn.kneighbors(Xq)
    return d, yb[i]


def _site_reply(d, yk):
    """the one probability a hospital returns to the network."""
    w = 1.0 / (d + 1e-6)
    return (w * yk).sum(axis=1) / np.maximum(w.sum(axis=1), 1e-12)


def _resample_prev(gi, y, r, seed):
    """scales the prevalence of the partner bank by a factor r. When raising, all events + some non-events; when lowering, all non-events + some events."""
    rs = np.random.RandomState(seed)
    yg = y[gi]
    pos, neg = gi[yg == 1], gi[yg == 0]
    prev = len(pos) / len(gi)
    q = min(max(prev * r, 0.005), 0.95)
    if q > prev:
        n_neg = max(1, min(int(round(len(pos) * (1 - q) / q)), len(neg)))
        return np.sort(np.concatenate([pos, rs.choice(neg, n_neg, replace=False)]))
    n_pos = max(1, min(int(round(len(neg) * q / (1 - q))), len(pos)))
    return np.sort(np.concatenate([neg, rs.choice(pos, n_pos, replace=False)]))


def network_unit(C, outcome, h, hid, sites, factors, alphas):
    """mixes the mean probability of the partners (up to 20 hospitals) into the 30% test portion of hospital h.
    factors are the partner bank prevalence multipliers. With 1.0 the whole partner cohort is used."""
    Xg, y, pb1 = C["Xg"], C["y"], C["pb1"]
    idx = np.where(hid == h)[0]
    rng = rng_of("a32", outcome, int(h))
    perm = rng.permutation(len(idx))
    n_te = max(60, int(TEST_FRAC_NET * len(idx)))
    te = idx[perm[:n_te]]
    yte = y[te]
    if len(np.unique(yte)) < 2 or len(idx) - n_te < 60:
        return None
    Xte, p1te = Xg[te], pb1[te]
    partners = [g for g in sites if g != h]
    rng.shuffle(partners)
    partners = partners[:MAX_PARTNERS]
    if len(partners) < 4:
        return None
    own_prev = float(y[idx].mean())
    base = full_metrics(yte, p1te)
    nb_c = f"nb@{float(T_SITE[outcome])}"          # column name of the net benefit at the outcome-specific representative threshold (midpoint of the outcome-specific range) (same as nbcol)
    rows = []
    for r in factors:
        P_list, prevs = [], []
        for g in partners:
            gi = np.where(hid == g)[0]
            if r != 1.0:
                gi = _resample_prev(gi, y, r, stable_seed("a7", outcome, int(h), int(g), r))
            d, yk = _topk_cos(Xg[gi], y[gi].astype(float), Xte, K_NEIGHBORS)
            P_list.append(_site_reply(d, yk))
            prevs.append(float(y[gi].mean()))
        p_net = np.column_stack(P_list).mean(axis=1)
        for a in alphas:
            pm = (1 - a) * p1te + a * p_net
            m = full_metrics(yte, pm)
            rows.append({"outcome": outcome, "hospitalid": int(h), "factor": float(r), "alpha": float(a),
                         "n_test": int(len(te)), "n_partners": int(len(partners)),
                         "prev_own": own_prev, "prev_partner": float(np.mean(prevs)),
                         "prev_diff": float(np.mean(prevs)) - own_prev,
                         "d_auroc": m["auroc"] - base["auroc"], "base_auroc": base["auroc"],
                         "base_slope": base["slope"], "base_intercept": base["intercept"],
                         "base_ece": base["ece"], "d_ece": m["ece"] - base["ece"],
                         "rep_threshold": float(T_SITE[outcome]), "base_nb_rep": base[nb_c], "nb_rep": m[nb_c], "d_nb_rep": m[nb_c] - base[nb_c], **m})
    return rows


# ---- meta-analysis (A6) ------------------------------------------------------------
def meta_reml_hk(d, v, exclude_zero_var=True):
    """random-effects meta-analysis of per-hospital estimates d and variances v. REML tau squared, Hartung-Knapp interval (original and truncated forms: q lower-bounded at 1),
    95% prediction interval (t distribution, k-2 degrees of freedom, truncated SE), inverse-variance Higgins I squared, Q and degrees of freedom. None if k<3. A hospital with any nan is dropped.
    hospitals with zero variance (v <= 1e-12) are dropped by default (exclude_zero_var=True). For such a hospital the retrieval prediction equals the frozen prediction, so the gain is also 0,
    and a variance floor of 1e-12 would give weight 1e12 and take over the whole pooled value (one hospital of F42 AKI retrieval held 99.9998% of the weight).
    exclude_zero_var=False keeps the variance floor 1e-12 in the weights (for the diagnostic columns *_orig and the re-aggregation of the reference hospital file a59).
    k_zero_var is the number of dropped hospitals, top1_weight_share is the largest share of weight held by one hospital."""
    d = np.asarray(d, dtype=float)
    v = np.asarray(v, dtype=float)
    zero = np.isfinite(d) & np.isfinite(v) & (v <= 1e-12)
    k_zero_var = int(zero.sum())
    ok = np.isfinite(d) & np.isfinite(v)
    if exclude_zero_var:
        ok &= ~zero
    d, v = d[ok], np.maximum(v[ok], 1e-12)
    k = len(d)
    if k < 3:
        return None
    w = 1.0 / v
    top1_weight_share = float(w.max() / w.sum())
    mu_fe = float((w * d).sum() / w.sum())
    Q = float((w * (d - mu_fe) ** 2).sum())
    I2 = max(0.0, (Q - (k - 1)) / Q) * 100 if Q > 0 else 0.0

    def nll(t2):
        wt = 1.0 / (v + t2)
        mu = (wt * d).sum() / wt.sum()
        return 0.5 * (np.log(v + t2).sum() + np.log(wt.sum()) + (wt * (d - mu) ** 2).sum())

    upper = max(10.0 * float(np.var(d, ddof=1)), 10.0 * float(v.max()), 1e-6)
    r = minimize_scalar(nll, bounds=(0.0, upper), method="bounded", options={"xatol": 1e-12})
    t2 = float(r.x) if nll(float(r.x)) < nll(0.0) - 1e-12 else 0.0
    wt = 1.0 / (v + t2)
    mu = float((wt * d).sum() / wt.sum())
    se_re = float(np.sqrt(1.0 / wt.sum()))
    q = float((wt * (d - mu) ** 2).sum() / (k - 1))
    se_hk = float(np.sqrt(q / wt.sum()))
    se_hkf = float(np.sqrt(max(1.0, q) / wt.sum()))
    tc = float(scipy_stats.t.ppf(0.975, k - 1))
    tp = float(scipy_stats.t.ppf(0.975, k - 2))
    s2 = float((k - 1) * w.sum() / (w.sum() ** 2 - (w ** 2).sum()))
    half_pi = tp * float(np.sqrt(t2 + se_hkf ** 2))
    return {"k": k, "mu_fixed": mu_fe, "Q": Q, "I2_invvar": I2, "tau2": t2, "tau": float(np.sqrt(t2)),
            "I2_reml": float(100 * t2 / (t2 + s2)) if (t2 + s2) > 0 else 0.0,
            "mu_random": mu, "se_random": se_re,
            "ci_z_lo": mu - 1.96 * se_re, "ci_z_hi": mu + 1.96 * se_re,
            "ci_hk_lo": mu - tc * se_hk, "ci_hk_hi": mu + tc * se_hk,
            "ci_hk_trunc_lo": mu - tc * se_hkf, "ci_hk_trunc_hi": mu + tc * se_hkf,
            "df": k - 1, "Q_p": float(scipy_stats.chi2.sf(Q, k - 1)), "q_hk": q,
            "pi_lo": mu - half_pi, "pi_hi": mu + half_pi,
            "k_zero_var": k_zero_var, "top1_weight_share": top1_weight_share}


def i2_stays_weighted(d, n):
    """Stay-weighted I squared (Q with stays as weights). The weight is not the inverse variance, so it is non-standard."""
    d = np.asarray(d, dtype=float)
    w = np.asarray(n, dtype=float)
    if len(d) < 3:
        return np.nan
    mu = float(np.average(d, weights=w))
    Q = float(((d - mu) ** 2 * w).sum())
    return max(0.0, (Q - (len(d) - 1)) / Q * 100) if Q > 0 else 0.0


# ---- actual MIMIC-IV year periods (A1) -----------------------------------------------
def read_mimic4_periods():
    """actual year period of a stay (MIMIC-IV). Two methods.
    (a) unit-transfer adjustment (M4_HAS_SHIFT): when patients has anchor_year and icustays has intime. anchor_year_group of patients is
        the actual 3-year period, and the period is shifted by the difference between the shifted year of the stay (intime) and anchor_year (PhysioNet MIMIC-IV documentation).
        if the stay falls entirely inside one 3-year period it is strict; if it straddles, it is dropped from strict. majority puts it in the period with most of its years.
    (b) patient-level approximation (copy without anchor_year): all stays of the same patient get the anchor_year_group period of that patient. There is no unit-transfer
        adjustment and intime is not used. MIMIC-IV shifts dates by a different amount per patient, so the intime order between patients is not the real calendar order.
        the patient_group column of the table is the period number (A1b uses only patient_group in this method)."""
    c_st = [_ci["stay_id"], _ci["subject_id"]] + ([_ci["intime"]] if M4_HAS_SHIFT else [])
    c_pt = [_cp["subject_id"], _cp["anchor_year_group"]] + ([_cp["anchor_year"]] if M4_HAS_SHIFT else [])
    # usecols returns columns in file order, so select again by name (naming by position would swap stay_id and subject_id)
    st = pd.read_csv(MIMIC4_ICUSTAYS_PATH, usecols=c_st)[c_st]
    st.columns = ["stay_id", "subject_id"] + (["intime"] if M4_HAS_SHIFT else [])
    pt = pd.read_csv(MIMIC4_PATIENTS_PATH, usecols=c_pt)[c_pt]
    pt.columns = ["subject_id", "anchor_year_group"] + (["anchor_year"] if M4_HAS_SHIFT else [])
    pt = pt.drop_duplicates("subject_id")
    m = st.merge(pt, on="subject_id", how="left")
    grp = m["anchor_year_group"].astype(str).str.extract(r"(\d{4})\s*-\s*(\d{4})")
    lo = pd.to_numeric(grp[0], errors="coerce")
    hi = pd.to_numeric(grp[1], errors="coerce")
    pairs = sorted({(int(a), int(b)) for a, b in zip(lo.dropna(), hi.dropna())})
    labels = [f"{a}-{b}" for a, b in pairs]
    if not M4_HAS_SHIFT:
        at = {p_: i_ for i_, p_ in enumerate(pairs)}
        pg = np.full(len(m), -1, dtype=int)
        for i, (a, b) in enumerate(zip(lo.to_numpy(dtype=float), hi.to_numpy(dtype=float))):
            if np.isfinite(a) and np.isfinite(b):
                pg[i] = at[(int(a), int(b))]
        out = pd.DataFrame({"stay_id": pd.to_numeric(m["stay_id"], errors="coerce").astype("int64"),
                            "shift": np.nan, "strict": pg, "majority": pg, "patient_group": pg})
        return out.set_index("stay_id"), labels
    shift = pd.to_datetime(m["intime"], errors="coerce").dt.year - pd.to_numeric(m["anchor_year"], errors="coerce")
    r_lo, r_hi = (lo + shift).to_numpy(dtype=float), (hi + shift).to_numpy(dtype=float)

    def which(y_):
        for gi_, (a, b) in enumerate(pairs):
            if a <= y_ <= b:
                return gi_
        return -1

    strict = np.full(len(m), -1, dtype=int)
    major = np.full(len(m), -1, dtype=int)
    for i in range(len(m)):
        if not (np.isfinite(r_lo[i]) and np.isfinite(r_hi[i])):
            continue
        yrs = range(int(r_lo[i]), int(r_hi[i]) + 1)
        gs = [which(y_) for y_ in yrs]
        if all(g_ == gs[0] for g_ in gs) and gs[0] >= 0:
            strict[i] = gs[0]
        cnt = {}
        for g_ in gs:
            if g_ >= 0:
                cnt[g_] = cnt.get(g_, 0) + 1
        if cnt:
            major[i] = sorted(cnt, key=lambda g_: (-cnt[g_], g_))[0]
    out = pd.DataFrame({"stay_id": pd.to_numeric(m["stay_id"], errors="coerce").astype("int64"),
                        "shift": shift.to_numpy(dtype=float), "strict": strict, "majority": major})
    return out.set_index("stay_id"), labels


# ---- temporal validation runner (A1) -------------------------------------------------------
def mean_abs_smd(XA, XB):
    """mean absolute standardized mean difference over columns of the standardized feature matrices of two periods."""
    ma, mb = XA.mean(0), XB.mean(0)
    sa, sb = XA.std(0, ddof=1), XB.std(0, ddof=1)
    sp = np.sqrt((sa ** 2 + sb ** 2) / 2.0)
    sp = np.where(sp < 1e-9, np.nan, sp)
    return float(np.nanmean(np.abs(ma - mb) / sp))


def run_temporal(C, past_mask, test_mask, name, outcome, banks, n_draws, groups):
    """splits the test-period patients into N_TIME_SPLITS folds stratified by outcome and tests fold by fold in turn (role alternation).
    P past bank (past period only) | S same-period bank (test period outside the test fold) | R (optional) everything outside the test fold (periods mixed).
    the three policies are scored on the same whole test patients, so differences between policies are paired. Compare at the same size only:
    if bank is "full", the policies are reduced to the largest size all can supply (n_max) and labeled "max"."""
    y = C["y"]
    test_ids = np.where(test_mask)[0]
    past_ids = np.where(past_mask)[0]
    notes = []
    if len(test_ids) < 4 * N_TIME_SPLITS or len(past_ids) < 50 or len(np.unique(y[test_ids])) < 2:
        notes.append({"analysis": name, "outcome": outcome, "note": f"insufficient sample (test period {len(test_ids)}, past {len(past_ids)})"})
        return [], [], notes
    folds = list(StratifiedKFold(n_splits=N_TIME_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(test_ids)), y[test_ids]))
    n_same = min(len(tr) for tr, _ in folds)                       # pool size of policy S (smallest fold)
    n_max = min(len(past_ids), n_same)
    rows, info_rows = [], []
    for bank_size in banks:
        size = n_max if bank_size == "full" else bank_size
        label = "max" if bank_size == "full" else str(bank_size)
        if size > n_max or size < 30:
            notes.append({"analysis": name, "outcome": outcome, "bank": label,
                          "note": f"insufficient pool (past {len(past_ids)} , same period {n_same} the smaller of {n_max} < bank {size})"})
            continue
        n_rep = 1 if bank_size == "full" else n_draws
        for pol in A1_POLICIES:
            for rep in range(n_rep):
                oof = {}
                for fi, (tr_i, te_i) in enumerate(folds):
                    test, other = test_ids[te_i], test_ids[tr_i]
                    pool = {"P": past_ids, "S": other, "R": np.concatenate([past_ids, other])}[pol]
                    key = f"a1|{name}|{pol}|{outcome}|{label}|{fi}|{rep}"
                    sel = np.arange(len(pool)) if size >= len(pool) else sample_bank(y[pool], size, key)
                    bank = pool[sel]
                    if np.intersect1d(bank, test).size or (pol == "P" and not past_mask[bank].all()):
                        raise AssertionError("invariant violation: the bank contains test patients or the P bank contains non-past patients " + key)
                    P, info = bank_predictions(C, bank, test, key, set(groups))
                    for m, v in P.items():
                        if m not in oof:
                            oof[m] = np.full(len(y), np.nan, dtype=np.float32)
                        oof[m][test] = v
                    info_rows.append({"analysis": name, "outcome": outcome, "bank": label, "policy": pol,
                                      "rep": rep, "fold": fi, "n_bank": int(len(bank)),
                                      "events_bank": int(y[bank].sum()), **info})
                for m, pm in oof.items():
                    pt = pm[test_ids].astype(float)
                    if np.isnan(pt).any():
                        continue
                    rows.append({"analysis": name, "outcome": outcome, "bank": label, "policy": pol,
                                 "rep": rep, "method": m, "n_test": int(len(test_ids)), "n_bank": int(size),
                                 **full_metrics(y[test_ids], pt)})
    return rows, info_rows, notes


def period_summary(C, past_mask, test_mask, name, outcome):
    """size, prevalence and feature shift of the past and test periods (for A1 description)."""
    y = C["y"]
    pa, te = np.where(past_mask)[0], np.where(test_mask)[0]
    smd = mean_abs_smd(C["Xg"][pa], C["Xg"][te]) if len(pa) > 2 and len(te) > 2 else np.nan
    return {"analysis": name, "outcome": outcome, "n_past": int(len(pa)), "n_test": int(len(te)),
            "prev_past": float(y[pa].mean()) if len(pa) else np.nan,
            "prev_test": float(y[te].mean()) if len(te) else np.nan, "mean_abs_smd": smd}


# ---- TabPFN connection (A9) ------------------------------------------------------------
TABPFN_INFO = {}                  # filled once when the first classifier is created: package version, v2 selection method, model_version, weight path argument


def tabpfn_classifier(device, seed):
    """creates the classifier by selecting the TabPFN v2 weights explicitly. Only the arguments accepted by the installed tabpfn are passed (the signature is read).
    behaviour by version (read from the sources of tabpfn 2.0.9, 2.1.3, 2.2.1, 6.0.0 and 9.1.0):
      - in 6.0.0 and 9.1.0, TabPFNClassifier.create_default_for_version(ModelVersion.V2, **kw) puts the v2 weight path in as model_path
        (classifier.py of 9.1.0 and 6.0.0. ModelVersion is in tabpfn.constants).
        versions between the two have not been tested. If the function is missing or fails, the fallback below is used and that fact is recorded in model_version.
        the default of 9.1.0 (when model_path is auto, settings.py, model_version=V3_5) is v3.5, not v2. So the default constructor is not used.
        the v2 weight file name has no v2.5, v2.6, v3 or v3.5 marker, so when 9.1.0 reads the version from the file name (model_loading.py) it reads V2.
      - 2.x has no selection function and only v2 weights, so the default constructor is v2 (ModelVersion in 2.2.1 model_loading.py has V2 only).
    if v2 cannot be selected, the run does not stop but continues with the default weights and records that fact in model_version."""
    from tabpfn import TabPFNClassifier
    params = inspect.signature(TabPFNClassifier.__init__).parameters
    kw = {}
    if "device" in params:
        kw["device"] = device
    if "n_estimators" in params:
        kw["n_estimators"] = TABPFN_N_ESTIMATORS
    if "random_state" in params:
        kw["random_state"] = seed
    factory = getattr(TabPFNClassifier, "create_default_for_version", None)
    clf, how, ver = None, "default constructor (this installation has no function to select the model version)", None
    if factory is not None:
        try:
            from tabpfn.constants import ModelVersion
            clf = factory(ModelVersion.V2, **kw)
            how, ver = "create_default_for_version(ModelVersion.V2)", "v2"
        except Exception as exc:  # noqa: BLE001
            how = f"v2 selection failed ({type(exc).__name__}: {str(exc)[:80]}), continuing with the default weights"
            ver = "unconfirmed: v2 selection failed, default weights"
    if clf is None:
        clf = TabPFNClassifier(**kw)
        if ver is None:
            ver = "v2 (tabpfn 2.x has v2 weights only)" if TABPFN_PKG_VERSION.startswith("2.") else "unconfirmed: no function to select the model version"
    if not TABPFN_INFO:
        TABPFN_INFO.update({"tabpfn_package_version": TABPFN_PKG_VERSION, "how_v2_selected": how, "model_version": ver,
                            "model_path_arg": str(getattr(clf, "model_path", "attribute absent")), "device": str(device),
                            "n_estimators": str(TABPFN_N_ESTIMATORS)})
    return clf


def tabpfn_info_rows():
    """installation and weight table printed by A9. Returns two sets (rows with full paths for the screen, rows with file names only for csv; the csv rows contain file names only, no folder paths)."""
    mp = TABPFN_INFO.get("model_path_arg", "")
    if mp in ("", "auto", "attribute absent"):
        wfile = "model_path is " + (mp if mp else "empty") + " (package default path, the file name is decided by this installation)"
    else:
        _f = Path(mp)
        wfile = _f.name + (" (present %.0f MB)" % (_f.stat().st_size / 1e6) if _f.is_file() else " (file not found)")
    items = [("tabpfn package version", TABPFN_INFO.get("tabpfn_package_version", "unknown")), ("v2 selection method", TABPFN_INFO.get("how_v2_selected", "unknown")),
             ("model_version", TABPFN_INFO.get("model_version", "unknown")), ("weight file", wfile),
             ("softmax_temperature (value after fitting)", TABPFN_INFO.get("softmax_temperature_after_fit", "before fitting")),
             ("device", TABPFN_INFO.get("device", "")), ("n_estimators", TABPFN_INFO.get("n_estimators", ""))]
    full = items + [("model_path argument (full path)", mp)]
    return full, items


def _is_oom(exc):
    """whether it is a GPU out-of-memory error (torch.OutOfMemoryError, TabPFNCUDAOutOfMemoryError, or out of memory in the message)."""
    return ("OutOfMemory" in type(exc).__name__) or ("out of memory" in str(exc).lower())


def tabpfn_predict(clf, X, batch=TABPFN_BATCH):
    """prediction in batches. On out of memory the batch is halved and the same batch is retried (minimum 8 rows). The predictions themselves do not depend on the batch size
    (TabPFN predicts each test row independently with the same context)."""
    out = np.empty(len(X), dtype=float)
    i, b = 0, max(int(batch), 1)
    while i < len(X):
        try:
            out[i:i + b] = clf.predict_proba(X[i:i + b])[:, 1]
            i += b
        except Exception as exc:  # noqa: BLE001
            if not _is_oom(exc) or b <= 8:
                raise
            b = max(b // 2, 8)
            try:
                import torch as _t
                _t.cuda.empty_cache()
            except Exception:  # noqa: BLE001
                pass
            print(f"  [A9] GPU out of memory , prediction batch reduced to {b} rows and retried", flush=True)
    return out


def tabpfn_fit_subsets(Xs, ys, n_ctx, n_sub, device, outcome):
    """fits TabPFN on n_sub stratified subsamples (n_ctx rows) of the development training data and freezes it (the model weights do not change)."""
    clfs = []
    for s in range(n_sub):
        idx = sample_bank(ys, n_ctx, f"tabpfn|{outcome}|{s}")
        clf = tabpfn_classifier(device, stable_seed("tabpfn", outcome, s))
        clf.fit(Xs[idx], ys[idx])
        TABPFN_INFO.setdefault("softmax_temperature_after_fit", str(getattr(clf, "softmax_temperature_", "attribute absent")))
        clfs.append(clf)
    return clfs


def tabpfn_frozen_probs(clfs, X):
    return np.mean([tabpfn_predict(c, X) for c in clfs], axis=0)


def stratified_cap(y, cap, key):
    """reduces the target cohort to cap rows (stratified). All rows if not reduced."""
    if cap is None or len(y) <= cap:
        return np.arange(len(y))
    return sample_bank(np.asarray(y), int(cap), key)


A9_RENAME = {"b1_base": "tabpfn_frozen", "b5_base": "logistic_frozen", "pram_b1": "pram_on_tabpfn",
             "pram_b5": "pram_on_logistic", "platt_b1": "platt_on_tabpfn", "recal_iso": "isotonic_on_tabpfn",
             "upd1_b1": "upd1_tabpfn", "upd2_b1": "upd2_tabpfn", "upd3_b1": "upd3_tabpfn", "ct_b1": "closed_test_tabpfn",
             "upd1_b5": "upd1_logistic", "upd2_b5": "upd2_logistic", "upd3_b5": "upd3_logistic", "ct_b5": "closed_test_logistic"}


# ---- reporting helpers (shared by the grid and hospital-level reporting cells) --------------------------------
def nbcol(t):
    return f"nb@{float(t)}"


def pick_report_banks(banks):
    pref = [b for b in (1000, "full") if b in banks]
    return pref if pref else [banks[0], banks[-1]]


def bank_sort_key(b):
    return 10 ** 9 if str(b) in ("full", "max") else int(b)


def draw_median_table(df, methods, extra_cols=()):
    """table of medians over draws (rep). Columns: AUROC, slope, intercept, ECE, net benefit at the representative threshold, net benefit at 20%."""
    d = df[df["method"].isin(methods)].copy()
    d["nb_site"] = [r[nbcol(T_SITE[oc])] for oc, r in zip(d["outcome"], d.to_dict("records"))]
    d["nb_20"] = d[nbcol(THRESH_MS)]
    cols = ["auroc", "slope", "intercept", "ece", "nb_site", "nb_20"] + list(extra_cols)
    g = d.groupby(["fs", "outcome", "target", "bank", "method"], sort=False)[cols].median().reset_index()
    g["n_draws"] = d.groupby(["fs", "outcome", "target", "bank", "method"], sort=False)["rep"].nunique().values
    return g


def paired_gap(df, method, base, col="auroc"):
    """summary of the distribution of the difference (col) between method and base in the same draws: median, 25%, 75%, minimum, fraction below 0."""
    keys = ["fs", "outcome", "target", "bank", "rep"]
    a = df[df["method"] == method].set_index(keys)[col]
    b = df[df["method"] == base].set_index(keys)[col]
    d = (a - b.reindex(a.index)).dropna()
    if len(d) == 0:
        return pd.DataFrame()
    g = d.groupby(["fs", "outcome", "target", "bank"], sort=False)
    out = g.agg(gap_median="median", gap_q25=lambda s: s.quantile(0.25), gap_q75=lambda s: s.quantile(0.75),
                gap_min="min", share_below_zero=lambda s: float((s < 0).mean()), n_draws="size").reset_index()
    out.insert(4, "method", method)
    out.insert(5, "base", base)
    return out


def hospital_time_unit(C, outcome, h, hid, year, y_past, y_test, groups):
    """A1 hospital level: the whole past year of hospital h is the bank and the whole next year is the test. Only hospitals above the criterion in both years."""
    y = C["y"]
    idx = np.where(hid == h)[0]
    bank, test = idx[year[idx] == y_past], idx[year[idx] == y_test]
    for part in (bank, test):
        ev = int(y[part].sum())
        if len(part) < HOSP_MIN_N_PER_YEAR or ev < HOSP_MIN_EVENTS_PER_YEAR or len(part) - ev < HOSP_MIN_EVENTS_PER_YEAR:
            return None
    P, info = bank_predictions(C, bank, test, f"a1h|{outcome}|{h}", set(groups))
    yt = y[test]
    rows = []
    for m, v in P.items():
        _a, _b, d1, v1 = delong_delta_var(yt, v, P["b1_base"])
        rows.append({"outcome": outcome, "hospitalid": int(h), "method": m, "n_bank": int(len(bank)), "n_test": int(len(test)),
                     "events_test": int(yt.sum()), **full_metrics(yt, v), "d_auroc_b1": d1, "var_b1": v1})
    return rows



def prepare_sites():
    """builds the hospital id array and the list of eligible hospitals (shared by the hospital-level cell and the network cell)."""
    hid, sites = {}, {}
    for oc in OUTCOMES:
        c = CTX[("F42", oc, "eICU")]
        hid[oc] = map_to_stays(EICU_TAB, "hospitalid", c["stay_ids"])
        s = eligible_sites(c["y"], hid[oc])
        sites[oc] = s[:SITE_LIMIT] if SITE_LIMIT else s
    return hid, sites

def as_table_frame(df, a="analysis", p="policy"):
    """temporarily renames the A1 result to the column names (fs, target) that draw_median_table and paired_gap read."""
    return df.rename(columns={a: "fs", p: "target"})


def policy_contrast(tab, method, base, col):
    """A1: puts the gains (col) of the same strategy side by side under policies P (past), S (same period) and R (if present), and gives P-S and P-R."""
    g = paired_gap(tab, method, base, col)
    if len(g) == 0:
        return pd.DataFrame()
    w = g.pivot_table(index=["fs", "outcome", "bank"], columns="target", values="gap_median", aggfunc="first")
    for pol in ("P", "S", "R"):
        if pol not in w.columns:
            w[pol] = np.nan
    w["P_minus_S"] = w["P"] - w["S"]
    w["P_minus_R"] = w["P"] - w["R"]
    w = w.reset_index().rename(columns={"fs": "analysis"})
    w.insert(3, "method", method)
    w.insert(4, "metric", col)
    return w


# ---- tail risk (A2): unit of (one bank, one test fold) -------------------------------------
def run_tail_unit(C, fs, outcome, target, bank_size, groups, keep, delong_pairs):
    """one cell (feature set, outcome, target, bank size). Makes N_FOLDS x N_DRAWS_TAIL // N_FOLDS (bank, test fold) pairs.
    one pair is one measurement in which one hospital receives one bank and measures on its own test patients."""
    y = C["y"]
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))
    per_fold = max(1, N_DRAWS_TAIL // N_FOLDS)
    rows = []
    for fi, (pool_idx, test_idx) in enumerate(folds):
        if bank_size > len(pool_idx):
            continue
        yt = y[test_idx]
        for dr in range(per_fold):
            key = f"{outcome}|{target}|{bank_size}|{fi}|{dr}"
            bank = pool_idx[sample_bank(y[pool_idx], bank_size, "tail|" + key)]
            if len(np.unique(y[bank])) < 2:
                continue
            P, _info = bank_predictions(C, bank, test_idx, key, set(groups))
            for m in keep:
                if m not in P:
                    continue
                row = {"fs": fs, "outcome": outcome, "target": target, "bank": str(bank_size), "fold": fi, "draw": dr,
                       "method": m, "n_events_bank": int(y[bank].sum()), **full_metrics(yt, P[m])}
                if delong_pairs.get(m) in P:
                    _a, _b, dd, vv = delong_delta_var(yt, P[m], P[delong_pairs[m]])
                    row["d_auroc_base"], row["var_auroc_base"] = dd, vv
                rows.append(row)
    return rows


def tail_gaps(df, pairs):
    """turns the tail rows (metrics per method) into a table of (method - reference) pair differences. Columns: gap_auroc gap_ece gap_slope gap_nb_site gap_nb_20 dl_lo dl_hi."""
    t = df.copy()
    t["nb_site"] = [r[nbcol(T_SITE[oc])] for oc, r in zip(t["outcome"], t.to_dict("records"))]
    t["nb_20"] = t[nbcol(THRESH_MS)]
    keys = ["fs", "outcome", "target", "bank", "fold", "draw"]
    cols = ["auroc", "ece", "slope", "nb_site", "nb_20"]
    out = []
    for m, b in pairs.items():
        a = t[t["method"] == m].set_index(keys)
        bb = t[t["method"] == b].set_index(keys)
        if len(a) == 0 or len(bb) == 0:
            continue
        g = (a[cols] - bb[cols].reindex(a.index)).add_prefix("gap_")
        g["method"] = m
        if "d_auroc_base" in a.columns:
            sd = np.sqrt(a["var_auroc_base"].astype(float))
            g["dl_lo"], g["dl_hi"] = a["d_auroc_base"] - 1.96 * sd, a["d_auroc_base"] + 1.96 * sd
        out.append(g.reset_index())
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def tail_summary(G):
    """tail of the gain distribution per (feature set, source, outcome, target, bank, method) cell. Cells without a metric (such as net benefit in the rows read from the reference file) are left empty."""
    rows = []
    for (fs, src, oc, tg, bk, m), s in G.groupby(["fs", "source", "outcome", "target", "bank", "method"], sort=False):
        a = s["gap_auroc"].dropna().to_numpy()
        if len(a) == 0:
            continue
        n = len(a)
        row = {"fs": fs, "source": src, "outcome": oc, "target": tg, "bank": bk, "method": m, "n_draws": n,
               "dAUROC_median": float(np.median(a)), "dAUROC_p05": float(np.quantile(a, 0.05)),
               "dAUROC_p10": float(np.quantile(a, 0.10)), "dAUROC_worst": float(a.min()), "share_below_zero": float((a < 0).mean())}
        for mg in NI_MARGINS_AUROC:
            row[f"auroc_share_lost_more_than_{mg}"] = float((a < -mg).mean())
        k = int((a < -NI_MARGIN_AUROC).sum())
        row["auroc_ni_share"] = k / n
        row["auroc_ni_cp_lo"], row["auroc_ni_cp_hi"] = cp_interval(k, n)
        if "dl_hi" in s.columns and s["dl_hi"].notna().any():
            dl = s.dropna(subset=["dl_lo", "dl_hi"])
            row["draws_ci_wholly_below_zero"] = float((dl["dl_hi"] < 0).mean())
            row["draws_ci_wholly_above_zero"] = float((dl["dl_lo"] > 0).mean())
        for col, tag in (("gap_ece", "ece"), ("gap_slope", "slope")):
            v = s[col].dropna()
            if len(v):
                row[f"d_{tag}_median"] = float(v.median())
        v = s["gap_ece"].dropna()
        if len(v):
            row["share_ece_worse"] = float((v > 0).mean())
        for col, tag in (("gap_nb_site", "nb_site"), ("gap_nb_20", "nb_20")):
            v = s[col].dropna().to_numpy()
            if len(v) == 0:
                continue
            row[f"d_{tag}_median"], row[f"d_{tag}_p05"], row[f"d_{tag}_worst"] = float(np.median(v)), float(np.quantile(v, 0.05)), float(v.min())
            for mg in NI_MARGINS_NB:
                row[f"{tag}_share_lost_more_than_{mg}"] = float((v < -mg).mean())
            kn = int((v < -NI_MARGIN_NB).sum())
            row[f"{tag}_ni_share"] = kn / len(v)
            row[f"{tag}_ni_cp_lo"], row[f"{tag}_ni_cp_hi"] = cp_interval(kn, len(v))
        rows.append(row)
    return pd.DataFrame(rows)


## Data loading, feature sets, frozen models

The six input files described in the repository README are read from `prepared_data/`. The 33-column set drops the variables whose maximum missingness over the three databases exceeds 30% (the 9 columns are removed before input and the development cache is rebuilt). The notebook prints whether the dropped variables are the nine expected ones (temperature, INR and weight; minimum, maximum and last).

For each feature set the development models (frozen logistic regression and CatBoost) are refitted on the MIMIC-IV training 56%.

| Set | Columns | Targets | Use |
|---|---|---|---|
| F42 | 42 inputs (43 after transform) | three cohorts | main analysis and update hierarchy; reference check |
| F33 | 33 inputs (34 after transform) | three cohorts | 33-column analyses |
| F42_noTemp, F42_noINR | only the three temperature columns or only the three INR columns removed | eICU | off (runs if added to `FS_RUN`) |

The printout of the number of inputs and of columns after the transform (42, 43, 33 and 34) is the feature count check. The column names are listed in Supplementary S1 Panel B.


In [ ]:
# ================================================================
# PRAM Cell 4. Data loading, feature sets, frozen models, A10
# ================================================================
DS, DEV, CTX, FS_COLS = {}, {}, {}, {}
EICU_TAB, SITE_CODE = None, {}
CAT_REF, NUM_REF, ALLVARS, SPARSE = [], [], [], []
MISS_TAB = pd.DataFrame()
STATUS = {}                                   # status per section. Read by the closing table (was it finished) of the last cell


def status(section, state, note=""):
    STATUS[section] = (state, note)


if not DATA_OK:
    emit("**[skipped] The main inputs are missing.** Look at the input check table of cell 1, fix `DATA_ROOT` and run again. All analysis cells below are skipped.")
    status("data loading", "not available", "main inputs absent")
else:
    t_load = time.time()
    for _oc in OUTCOMES:
        DS[("MIMIC-IV", _oc)] = load_mimic4(_oc)
        DS[("MIMIC-III", _oc)] = load_external("mimic3", _oc)
        DS[("eICU", _oc)] = load_external("eicu", _oc)
    X_REF, _ = _load_x("mimic4")
    CAT_REF, NUM_REF = _detect_cols(X_REF)
    ALLVARS = CAT_REF + NUM_REF
    _rows = []
    for _name in TARGETS:
        _Xall = X_REF if _name == "MIMIC-IV" else DS[(_name, "AKI_168h")]["X_df"]
        for _c in ALLVARS:
            _rows.append({"database": _name, "variable": _c, "pct_missing": 100.0 * float(_Xall[_c].isna().mean())})
    MISS_TAB = pd.DataFrame(_rows)
    _mm = MISS_TAB.groupby("variable")["pct_missing"].max()
    SPARSE = sorted([c for c in ALLVARS if _mm[c] > MISS_CUT_PCT])
    emit("### Cohort")
    _coh = [{"database": n, "outcome": oc, "n": len(DS[(n, oc)]["y"]), "prevalence": float(DS[(n, oc)]["y"].mean()),
             "n_inputs": DS[(n, oc)]["X_df"].shape[1]} for n in TARGETS for oc in OUTCOMES]
    emit_table("cohort size and prevalence", pd.DataFrame(_coh))
    emit(f"- number of feature inputs {len(ALLVARS)} (expected 42) | categorical {CAT_REF} | continuous {len(NUM_REF)} columns")
    emit(f"- maximum missingness over the three databases is {MISS_CUT_PCT}% are dropped {len(SPARSE)} columns: {SPARSE}")
    emit("- the nine expected variables and " + ("**equal**" if SPARSE == sorted(EXPECTED_SPARSE_9) else f"**differ** (expected {sorted(EXPECTED_SPARSE_9)})"))
    for _fs in FS_RUN:
        _sp = FS_SPEC[_fs]
        _cols = [c for c in ALLVARS if not (_sp["drop_rule"] == "miss" and c in SPARSE)
                 and not any(c.startswith(p) for p in _sp["drop_prefix"])]
        FS_COLS[_fs] = _cols
    emit("- number of feature-set columns: " + ", ".join(f"{k} {len(v)}" for k, v in FS_COLS.items()))
    status("data loading", "done", f"{time.time() - t_load:.0f} s")

    t_dev = time.time()
    for _fs in FS_RUN:
        for _oc in OUTCOMES:
            _d4 = DS[("MIMIC-IV", _oc)]
            _tr = _d4["idx_train"]
            DEV[(_fs, _oc)] = build_dev(_d4["X_df"].iloc[_tr], _d4["y"][_tr], FS_COLS[_fs], CAT_REF, NUM_REF, f"{_fs}|{_oc}")
            for _tg in FS_SPEC[_fs]["targets"]:
                _d = DS[(_tg, _oc)]
                if _tg == "MIMIC-IV":
                    _te = _d["idx_test"]
                    CTX[(_fs, _oc, _tg)] = make_ctx(DEV[(_fs, _oc)], _d["X_df"].iloc[_te], _d["y"][_te], _d["stay_ids"][_te], _oc)
                else:
                    CTX[(_fs, _oc, _tg)] = make_ctx(DEV[(_fs, _oc)], _d["X_df"], _d["y"], _d["stay_ids"], _oc)
            _log(f"frozen model fit {_fs} {_oc}")
    _chk = []
    for (_fs, _oc, _tg), _c in CTX.items():
        assert 0.0 <= _c["pb1"].min() and _c["pb1"].max() <= 1.0 and 0.0 <= _c["pb5"].min() and _c["pb5"].max() <= 1.0, "the frozen model output is not a probability"
        assert len(_c["y"]) == len(_c["Xg"]) == len(_c["Xc"]) == len(_c["stay_ids"]), "the context array lengths do not match"
        _a1 = roc_auc_score(_c["y"], _c["pb1"])
        _a5 = roc_auc_score(_c["y"], _c["pb5"])
        _chk.append({"fs": _fs, "outcome": _oc, "target": _tg, "n": len(_c["y"]), "prev": float(_c["y"].mean()),
                     "d": _c["d"], "auroc_logistic": _a1, "auroc_catboost": _a5})
    EICU_TAB = load_eicu_patient_table() if (HAVE_HOSPITALID or HAVE_EICU_YEAR) else None
    if EICU_TAB is not None and "hospitalid" in EICU_TAB.columns:        # only this code, not the hospital id, goes to the result csv
        SITE_CODE = {int(h): f"H{k + 1:03d}" for k, h in enumerate(sorted(EICU_TAB["hospitalid"].dropna().unique()))}
    emit_table("frozen model performance", pd.DataFrame(_chk), max_rows=60)
    save_csv(pd.DataFrame(_chk), "frozen_model_auroc")
    status("frozen model", "done", f"{time.time() - t_dev:.0f} s")
    for _fs in [f for f in ("F42", "F33") if f in FS_COLS]:                  # feature count consistency: number of inputs and number of columns after the transform
        _pg = DEV[(_fs, "AKI_168h")]["pg"]
        emit(f"- feature count consistency {_fs}: input {len(FS_COLS[_fs])} -> model columns after the transform {len(_pg.column_names())} (categorical {len(_pg.cat_cols)} expanded as one-hot)")


## Pre-check: finished units, units to run, expected time

This cell **does not compute.** It prints three things.

1. How many finished units each file in `out_pram/resume/` has (this run skips them).
2. The units this run will compute: 30 rescoring pooled-DeLong units (42 columns) and 6 development-bank starting-table units (`RESCORE_COLDSTART`). These two are all.
3. An expected time (approximate) from a short measurement of the cosine neighbor search speed of this machine.

`RESCORE_ONLY_GUARD` (default `False`) **stops the run at once, when set to `True`, if any section outside the rescoring would have to run a unit anew.** This prevents a computation of 12 to 16 hours from starting unnoticed when the working folder differs from the folder that holds the finished results (no resume) or the settings have changed. If result files `PRAM-summary.md` and `PRAM-results.md` already exist, they are copied to `PRAM-summary-backup.md` and `PRAM-results-backup.md` (not deleted) before the new result md files are written.


In [ ]:
# ================================================================
# PRAM pre-check . finished units | units to run | expected time | block new units outside the rescoring
# ================================================================
class RescoreStop(BaseException):
    """a unit outside the rescoring would run anew. Stops before computing. It is a BaseException, so `except Exception` does not swallow it."""


RS_SEEN_DONE, RS_NEW_OUTSIDE = set(), []         # (resume name, unit) that this run checked and skipped | units outside the rescoring that would have run anew (must be empty)
RS_PLAN, RS_C_CELL = pd.DataFrame(), float("nan")


def count_done_units(name):
    path = RESUME_DIR / f"{name}.units.jsonl"
    if not path.is_file():
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for ln in f if ln.strip())


def knn_seconds_per_cell(C):
    """cosine neighbor search speed of this machine (seconds / (queries x reference rows)). A short measurement for the time estimate."""
    X = C["Xg"]
    n_ref, n_q = min(4000, max(len(X) // 2, 10)), min(2000, max(len(X) // 4, 10))
    t0 = time.time()
    nn = NearestNeighbors(n_neighbors=min(K_NEIGHBORS, n_ref), metric="cosine", algorithm="brute", n_jobs=-1).fit(X[:n_ref])
    nn.kneighbors(X[n_ref:n_ref + n_q])
    return (time.time() - t0) / float(n_ref * n_q)


def cells_for(n_test, n_bank, n_inner_cv):
    """number of (queries x reference rows) of the cosine neighbor search: 1 test query + n_inner_cv inner cross-validation passes (per fold queries 1/N_INNER x reference (N_INNER-1)/N_INNER)."""
    return n_test * n_bank + n_inner_cv * ((N_INNER - 1.0) / N_INNER) * n_bank * n_bank


def pooled_unit_cells(n_rows, bank_size):
    n_test = n_rows / N_FOLDS
    pool = n_rows - n_test
    n_bank = pool if bank_size == "full" else min(bank_size, pool)
    per_call = cells_for(n_test, n_bank, 2)                                    # inner cross-validation of the two retrievals (b1 and b5)
    if bank_size == "full":
        per_call += cells_for(n_test, n_bank, 1)                               # key08 variant (diagnosis)
    return (1 if bank_size == "full" else N_DRAWS_MAIN) * N_FOLDS * per_call


def cold_unit_cells(n_rows, n_src):
    total = 0.0
    n_test = n_rows / N_FOLDS
    for bs in COLDSTART_BANKS:
        local = 0 if bs == 0 else ((n_rows - n_test) if bs == "full" else min(bs, n_rows - n_test))
        total += (1 if bs in (0, "full") else N_DRAWS_MAIN) * N_FOLDS * cells_for(n_test, n_src + local, 1)
    return total


if not (DATA_OK and RUN_RESCORE):
    status("rescoring pre-check", "off" if not RUN_RESCORE else "not available", "RUN_RESCORE=False or main inputs absent")
else:
    for _nm in ("PRAM-summary", "PRAM-results"):                      # keeps a backup of the result md without deleting it
        _src, _dst = OUT_DIR / (_nm + ".md"), OUT_DIR / (_nm + "-backup.md")
        if _src.is_file() and not _dst.exists():
            _dst.write_bytes(_src.read_bytes())
    _inv = pd.DataFrame([{"resume file": _p.name.replace(".units.jsonl", ""), "finished units": count_done_units(_p.name.replace(".units.jsonl", "")),
                          "last update": time.strftime("%Y-%m-%d %H:%M", time.localtime(_p.stat().st_mtime))}
                         for _p in sorted(RESUME_DIR.glob("*.units.jsonl"))])
    emit(f"### pre-check , resume folder `{RESUME_DIR.resolve()}`")
    emit_table("finished units (this run does not recompute the units below)", _inv, max_rows=80)
    _n_other = int(_inv.loc[~_inv["resume file"].isin(RESCORE_NAMES), "finished units"].sum()) if len(_inv) else 0
    _spec = FS_SPEC["F42"]
    _u_pool = [("F42", oc, tg, bk) for oc in OUTCOMES for tg in _spec["targets"] for bk in _spec["banks"]] if "F42" in FS_RUN else []
    _u_cold = [("p3", oc, tg) for oc in OUTCOMES for tg in TARGETS] if (RESCORE_COLDSTART and "F42" in FS_RUN) else []
    _Ra, _Rb = Resumable(RESCORE_NAMES[0]), Resumable(RESCORE_NAMES[1])
    _new_pool = [u for u in _u_pool if not _Ra.unit_done(u)]
    _new_cold = [u for u in _u_cold if not _Rb.unit_done(u)]
    _exp_m = {fs: len(OUTCOMES) * len(FS_SPEC[fs]["targets"]) * len(FS_SPEC[fs]["banks"]) for fs in FS_RUN}
    _have_m = {fs: count_done_units(f"matrix_{fs}") for fs in FS_RUN}
    _pre_bad = {fs: (_have_m[fs], _exp_m[fs]) for fs in FS_RUN if _have_m[fs] < _exp_m[fs]}
    if RESCORE_ONLY_GUARD and _pre_bad:
        raise RescoreStop("stop , the finished units of the strategy grid are insufficient (finished, needed) %s . This folder (%s) is not the folder of the stored results or its settings differ. "
                          "No new computation was started. Open this notebook in the same folder as the stored results (where out_pram is)." % (_pre_bad, RESUME_DIR.resolve()))
    _c_ref = CTX.get(("F42", OUTCOMES[0], "eICU"))
    RS_C_CELL = knn_seconds_per_cell(_c_ref) if _c_ref is not None else float("nan")
    _sec_pool = sum(pooled_unit_cells(len(CTX[(u[0], u[1], u[2])]["y"]), u[3]) for u in _new_pool) * RS_C_CELL * 1.3 if _new_pool else 0.0
    _sec_cold = sum(cold_unit_cells(len(CTX[("F42", u[1], u[2])]["y"]), len(DEV[("F42", u[1])]["ys_src"])) for u in _new_cold) * RS_C_CELL * 1.3 if _new_cold else 0.0

    def minutes(sec):
        return int(round(sec / 60.0)) if np.isfinite(sec) else -1                  # -1 = unknown because the neighbor search speed could not be measured
    RS_PLAN = pd.DataFrame([
        {"item": "rescoring pooled DeLong (42 columns, Table 2 q, S7 A and B)", "unit": len(_u_pool), "units to run": len(_new_pool), "units to skip": len(_u_pool) - len(_new_pool), "expected minutes (approximate)": minutes(_sec_pool)},
        {"item": "development-bank starting table", "unit": len(_u_cold), "units to run": len(_new_cold), "units to skip": len(_u_cold) - len(_new_cold), "expected minutes (approximate)": minutes(_sec_cold)},
        {"item": "all other sections (grid, hospital, A1, A2, A7, A9, etc.)", "unit": _n_other, "units to run": 0, "units to skip": _n_other, "expected minutes (approximate)": 0}])
    emit_table(f"what this run will compute (this machine neighbor search {RS_C_CELL * 1e9:.1f} counted as ns / (query x reference row). Recounted at the actual speed after the first unit)", RS_PLAN)
    emit(f"- units to run {len(_new_pool) + len(_new_cold)} | units to skip {_n_other + (len(_u_pool) - len(_new_pool)) + (len(_u_cold) - len(_new_cold))} | "
         f"expected about {minutes(_sec_pool + _sec_cold)} min (rough estimate) (data loading and frozen model fit about 75 seconds and a few minutes of aggregating stored results are separate)")
    if RESCORE_ONLY_GUARD:
        if not hasattr(Resumable, "_rs_unit_done_plain"):
            Resumable._rs_unit_done_plain = Resumable.unit_done

        def _guarded_unit_done(self, unit):
            done = Resumable._rs_unit_done_plain(self, unit)
            if done:
                if self.name not in RESCORE_NAMES:
                    RS_SEEN_DONE.add((self.name, self._ukey(unit)))
            elif self.name not in RESCORE_NAMES:
                RS_NEW_OUTSIDE.append((self.name, self._ukey(unit)))
                raise RescoreStop("stop , there are units outside the rescoring that would run anew: %s %s . This folder (%s) is not the folder of the stored results or its settings differ. "
                                  "No new computation was started. Open this notebook in the same folder as the stored results (where out_pram is)." % (self.name, self._ukey(unit), RESUME_DIR.resolve()))
            return done
        Resumable.unit_done = _guarded_unit_done
        emit("- guard on: in every section below, if a new unit outside the rescoring appears, it stops before computing.")
    status("rescoring pre-check", "done", f"new unit {len(_new_pool) + len(_new_cold)} (rescoring) | skipped {_n_other} columns")


## Reference check before the 42-column grid

Four units (a small bank and the full bank, three cohorts, two outcomes) are run first with the **same draw keys, seeds and splits** as the 42-column grid (`key_tag = "m23"`) and compared with reference values stored in the check cell: AUROC, calibration slope, and net benefit at 0.10 and 0.20, to four decimals.

- The reference values are constants in the check cell, so the check does not need any stored result file.
- Mismatching cells are printed as they are; a mismatch does not stop the run. Possible causes are different CatBoost and scikit-learn versions and a different preprocessing state.
- The results of these four units are reused as units of the grid (`matrix_F42`; same name and key). They are not recomputed.


In [ ]:
# ================================================================
# PRAM reference check of the 42-column grid (before the grid)
# ================================================================
REF_A23_CSV = """outcome,target,bank,method,auroc,slope,nb0.1,nb0.2
AKI_168h,MIMIC-III,1000,b1_base,0.7229,1.0177,0.0792,0.0435
AKI_168h,MIMIC-III,1000,b5_base,0.7485,0.9776,0.0855,0.0473
AKI_168h,MIMIC-III,1000,local_cb,0.6874,0.5953,0.0665,0.0290
AKI_168h,MIMIC-III,1000,local_lr,0.6918,0.6966,0.0734,0.0364
AKI_168h,MIMIC-III,1000,pooled_lr,0.7232,1.0129,0.0794,0.0436
AKI_168h,MIMIC-III,1000,pram_b1,0.7264,1.1881,0.0799,0.0448
AKI_168h,MIMIC-III,1000,pram_b5,0.7482,1.0337,0.0855,0.0473
AKI_168h,MIMIC-III,1000,recal_iso,0.7169,0.6511,0.0780,0.0426
AKI_168h,MIMIC-III,2000,b1_base,0.7229,1.0177,0.0792,0.0435
AKI_168h,MIMIC-III,2000,b5_base,0.7485,0.9776,0.0855,0.0473
AKI_168h,MIMIC-III,2000,local_cb,0.7064,0.7552,0.0760,0.0357
AKI_168h,MIMIC-III,2000,local_lr,0.7093,0.8604,0.0773,0.0397
AKI_168h,MIMIC-III,2000,pooled_lr,0.7234,1.0085,0.0794,0.0435
AKI_168h,MIMIC-III,2000,pram_b1,0.7282,1.1947,0.0801,0.0448
AKI_168h,MIMIC-III,2000,pram_b5,0.7478,1.0358,0.0853,0.0469
AKI_168h,MIMIC-III,2000,recal_iso,0.7172,0.7834,0.0785,0.0426
AKI_168h,MIMIC-III,500,b1_base,0.7229,1.0177,0.0792,0.0435
AKI_168h,MIMIC-III,500,b5_base,0.7485,0.9776,0.0855,0.0473
AKI_168h,MIMIC-III,500,local_cb,0.6719,0.4775,0.0537,0.0211
AKI_168h,MIMIC-III,500,local_lr,0.6664,0.4846,0.0696,0.0290
AKI_168h,MIMIC-III,500,pooled_lr,0.7230,1.0135,0.0793,0.0434
AKI_168h,MIMIC-III,500,pram_b1,0.7236,1.1611,0.0791,0.0438
AKI_168h,MIMIC-III,500,pram_b5,0.7474,1.0467,0.0851,0.0473
AKI_168h,MIMIC-III,500,recal_iso,0.7172,0.3969,0.0780,0.0421
AKI_168h,MIMIC-III,5000,b1_base,0.7229,1.0177,0.0792,0.0435
AKI_168h,MIMIC-III,5000,b5_base,0.7485,0.9776,0.0855,0.0473
AKI_168h,MIMIC-III,5000,local_cb,0.7345,0.9746,0.0828,0.0435
AKI_168h,MIMIC-III,5000,local_lr,0.7209,0.9561,0.0791,0.0429
AKI_168h,MIMIC-III,5000,pooled_lr,0.7244,1.0071,0.0795,0.0438
AKI_168h,MIMIC-III,5000,pram_b1,0.7300,1.2150,0.0807,0.0456
AKI_168h,MIMIC-III,5000,pram_b5,0.7484,1.0319,0.0854,0.0475
AKI_168h,MIMIC-III,5000,recal_iso,0.7195,0.9042,0.0792,0.0427
AKI_168h,MIMIC-III,full,b1_base,0.7229,1.0177,0.0792,0.0435
AKI_168h,MIMIC-III,full,b5_base,0.7485,0.9776,0.0855,0.0473
AKI_168h,MIMIC-III,full,local_cb,0.7521,1.0742,0.0871,0.0463
AKI_168h,MIMIC-III,full,local_lr,0.7260,0.9823,0.0795,0.0440
AKI_168h,MIMIC-III,full,pooled_lr,0.7258,1.0035,0.0796,0.0440
AKI_168h,MIMIC-III,full,pram_b1,0.7338,1.2405,0.0814,0.0467
AKI_168h,MIMIC-III,full,pram_b5,0.7492,1.0615,0.0859,0.0474
AKI_168h,MIMIC-III,full,recal_iso,0.7190,0.9592,0.0791,0.0434
AKI_168h,MIMIC-IV,1000,b1_base,0.7389,1.0338,0.0702,0.0398
AKI_168h,MIMIC-IV,1000,b5_base,0.7523,0.9838,0.0729,0.0398
AKI_168h,MIMIC-IV,1000,local_cb,0.6842,0.5892,0.0548,0.0201
AKI_168h,MIMIC-IV,1000,local_lr,0.6989,0.7376,0.0642,0.0311
AKI_168h,MIMIC-IV,1000,pooled_lr,0.7392,1.0325,0.0706,0.0399
AKI_168h,MIMIC-IV,1000,pram_b1,0.7383,1.1226,0.0700,0.0398
AKI_168h,MIMIC-IV,1000,pram_b5,0.7509,1.0506,0.0730,0.0395
AKI_168h,MIMIC-IV,1000,recal_iso,0.7315,0.6558,0.0699,0.0386
AKI_168h,MIMIC-IV,2000,b1_base,0.7389,1.0338,0.0702,0.0398
AKI_168h,MIMIC-IV,2000,b5_base,0.7523,0.9838,0.0729,0.0398
AKI_168h,MIMIC-IV,2000,local_cb,0.7023,0.7401,0.0632,0.0264
AKI_168h,MIMIC-IV,2000,local_lr,0.7213,0.8589,0.0681,0.0351
AKI_168h,MIMIC-IV,2000,pooled_lr,0.7396,1.0325,0.0708,0.0397
AKI_168h,MIMIC-IV,2000,pram_b1,0.7393,1.1455,0.0696,0.0397
AKI_168h,MIMIC-IV,2000,pram_b5,0.7512,1.0437,0.0730,0.0392
AKI_168h,MIMIC-IV,2000,recal_iso,0.7335,0.8142,0.0699,0.0384
AKI_168h,MIMIC-IV,500,b1_base,0.7389,1.0338,0.0702,0.0398
AKI_168h,MIMIC-IV,500,b5_base,0.7523,0.9838,0.0729,0.0398
AKI_168h,MIMIC-IV,500,local_cb,0.6644,0.4594,0.0409,0.0142
AKI_168h,MIMIC-IV,500,local_lr,0.6671,0.5050,0.0581,0.0236
AKI_168h,MIMIC-IV,500,pooled_lr,0.7390,1.0338,0.0704,0.0399
AKI_168h,MIMIC-IV,500,pram_b1,0.7373,1.1092,0.0696,0.0397
AKI_168h,MIMIC-IV,500,pram_b5,0.7506,1.0568,0.0728,0.0391
AKI_168h,MIMIC-IV,500,recal_iso,0.7285,0.5594,0.0692,0.0382
AKI_168h,MIMIC-IV,5000,b1_base,0.7389,1.0338,0.0702,0.0398
AKI_168h,MIMIC-IV,5000,b5_base,0.7523,0.9838,0.0729,0.0398
AKI_168h,MIMIC-IV,5000,local_cb,0.7318,0.9515,0.0705,0.0352
AKI_168h,MIMIC-IV,5000,local_lr,0.7340,0.9660,0.0701,0.0386
AKI_168h,MIMIC-IV,5000,pooled_lr,0.7403,1.0317,0.0708,0.0395
AKI_168h,MIMIC-IV,5000,pram_b1,0.7432,1.1861,0.0702,0.0409
AKI_168h,MIMIC-IV,5000,pram_b5,0.7530,1.0395,0.0734,0.0393
AKI_168h,MIMIC-IV,5000,recal_iso,0.7349,0.9312,0.0707,0.0389
AKI_168h,MIMIC-IV,full,b1_base,0.7389,1.0338,0.0702,0.0398
AKI_168h,MIMIC-IV,full,b5_base,0.7523,0.9838,0.0729,0.0398
AKI_168h,MIMIC-IV,full,local_cb,0.7446,1.0347,0.0727,0.0374
AKI_168h,MIMIC-IV,full,local_lr,0.7375,0.9833,0.0698,0.0394
AKI_168h,MIMIC-IV,full,pooled_lr,0.7409,1.0309,0.0716,0.0399
AKI_168h,MIMIC-IV,full,pram_b1,0.7442,1.1963,0.0702,0.0413
AKI_168h,MIMIC-IV,full,pram_b5,0.7533,1.0528,0.0735,0.0393
AKI_168h,MIMIC-IV,full,recal_iso,0.7343,0.9515,0.0709,0.0403
AKI_168h,eICU,1000,b1_base,0.6904,0.8339,0.0028,-0.0015
AKI_168h,eICU,1000,b5_base,0.6990,0.8448,0.0023,-0.0059
AKI_168h,eICU,1000,local_cb,0.6390,0.4722,0.0020,-0.0002
AKI_168h,eICU,1000,local_lr,0.6453,0.4606,0.0051,-0.0016
AKI_168h,eICU,1000,pooled_lr,0.6979,0.9726,0.0059,0.0012
AKI_168h,eICU,1000,pram_b1,0.6956,0.9646,0.0053,0.0006
AKI_168h,eICU,1000,pram_b5,0.7001,0.9201,0.0041,-0.0033
AKI_168h,eICU,1000,recal_iso,0.6808,0.4674,0.0079,0.0012
AKI_168h,eICU,2000,b1_base,0.6904,0.8339,0.0028,-0.0015
AKI_168h,eICU,2000,b5_base,0.6990,0.8448,0.0023,-0.0059
AKI_168h,eICU,2000,local_cb,0.6515,0.5939,0.0031,-0.0004
AKI_168h,eICU,2000,local_lr,0.6796,0.6841,0.0078,0.0010
AKI_168h,eICU,2000,pooled_lr,0.7022,1.0444,0.0076,0.0023
AKI_168h,eICU,2000,pram_b1,0.6981,1.0072,0.0058,0.0010
AKI_168h,eICU,2000,pram_b5,0.7018,0.9477,0.0044,-0.0027
AKI_168h,eICU,2000,recal_iso,0.6844,0.6778,0.0082,0.0012
AKI_168h,eICU,500,b1_base,0.6904,0.8339,0.0028,-0.0015
AKI_168h,eICU,500,b5_base,0.6990,0.8448,0.0023,-0.0059
AKI_168h,eICU,500,local_cb,0.6096,0.3724,0.0009,-0.0004
AKI_168h,eICU,500,local_lr,0.5954,0.2538,0.0016,-0.0047
AKI_168h,eICU,500,pooled_lr,0.6947,0.9204,0.0047,0.0001
AKI_168h,eICU,500,pram_b1,0.6929,0.9484,0.0052,0.0006
AKI_168h,eICU,500,pram_b5,0.6984,0.9005,0.0038,-0.0036
AKI_168h,eICU,500,recal_iso,0.6765,0.3398,0.0075,0.0008
AKI_168h,eICU,5000,b1_base,0.6904,0.8339,0.0028,-0.0015
AKI_168h,eICU,5000,b5_base,0.6990,0.8448,0.0023,-0.0059
AKI_168h,eICU,5000,local_cb,0.6855,0.8014,0.0055,-0.0000
AKI_168h,eICU,5000,local_lr,0.7094,0.8754,0.0105,0.0030
AKI_168h,eICU,5000,pooled_lr,0.7082,1.0964,0.0098,0.0034
AKI_168h,eICU,5000,pram_b1,0.7028,1.0509,0.0063,0.0013
AKI_168h,eICU,5000,pram_b5,0.7051,0.9838,0.0051,-0.0021
AKI_168h,eICU,5000,recal_iso,0.6866,0.8476,0.0085,0.0016
AKI_168h,eICU,full,b1_base,0.6904,0.8339,0.0028,-0.0015
AKI_168h,eICU,full,b5_base,0.6990,0.8448,0.0023,-0.0059
AKI_168h,eICU,full,local_cb,0.7402,1.1306,0.0113,0.0022
AKI_168h,eICU,full,local_lr,0.7315,0.9894,0.0118,0.0045
AKI_168h,eICU,full,pooled_lr,0.7264,1.0486,0.0117,0.0042
AKI_168h,eICU,full,pram_b1,0.7100,1.0952,0.0073,0.0018
AKI_168h,eICU,full,pram_b5,0.7102,1.0127,0.0059,-0.0014
AKI_168h,eICU,full,recal_iso,0.6892,0.9767,0.0090,0.0015
mortality_168h,MIMIC-III,1000,b1_base,0.8600,0.9070,0.0152,0.0058
mortality_168h,MIMIC-III,1000,b5_base,0.8779,0.8936,0.0166,0.0088
mortality_168h,MIMIC-III,1000,local_cb,0.8279,0.7557,0.0084,0.0032
mortality_168h,MIMIC-III,1000,local_lr,0.8036,0.6134,0.0109,0.0020
mortality_168h,MIMIC-III,1000,pooled_lr,0.8605,0.9123,0.0153,0.0056
mortality_168h,MIMIC-III,1000,pram_b1,0.8603,0.9975,0.0153,0.0056
mortality_168h,MIMIC-III,1000,pram_b5,0.8758,0.9259,0.0167,0.0087
mortality_168h,MIMIC-III,1000,recal_iso,0.8494,0.5909,0.0155,0.0056
mortality_168h,MIMIC-III,2000,b1_base,0.8600,0.9070,0.0152,0.0058
mortality_168h,MIMIC-III,2000,b5_base,0.8779,0.8936,0.0166,0.0088
mortality_168h,MIMIC-III,2000,local_cb,0.8468,0.8661,0.0123,0.0047
mortality_168h,MIMIC-III,2000,local_lr,0.8351,0.7918,0.0140,0.0037
mortality_168h,MIMIC-III,2000,pooled_lr,0.8609,0.9182,0.0155,0.0057
mortality_168h,MIMIC-III,2000,pram_b1,0.8618,0.9914,0.0155,0.0058
mortality_168h,MIMIC-III,2000,pram_b5,0.8773,0.9178,0.0168,0.0087
mortality_168h,MIMIC-III,2000,recal_iso,0.8519,0.7393,0.0157,0.0057
mortality_168h,MIMIC-III,500,b1_base,0.8600,0.9070,0.0152,0.0058
mortality_168h,MIMIC-III,500,b5_base,0.8779,0.8936,0.0166,0.0088
mortality_168h,MIMIC-III,500,local_cb,0.8062,0.6562,0.0060,0.0024
mortality_168h,MIMIC-III,500,local_lr,0.7671,0.4647,0.0092,0.0002
mortality_168h,MIMIC-III,500,pooled_lr,0.8602,0.9087,0.0153,0.0057
mortality_168h,MIMIC-III,500,pram_b1,0.8590,0.9820,0.0153,0.0055
mortality_168h,MIMIC-III,500,pram_b5,0.8750,0.9265,0.0166,0.0086
mortality_168h,MIMIC-III,500,recal_iso,0.8449,0.5185,0.0155,0.0050
mortality_168h,MIMIC-III,5000,b1_base,0.8600,0.9070,0.0152,0.0058
mortality_168h,MIMIC-III,5000,b5_base,0.8779,0.8936,0.0166,0.0088
mortality_168h,MIMIC-III,5000,local_cb,0.8626,0.9660,0.0157,0.0072
mortality_168h,MIMIC-III,5000,local_lr,0.8541,0.8949,0.0154,0.0048
mortality_168h,MIMIC-III,5000,pooled_lr,0.8620,0.9286,0.0157,0.0057
mortality_168h,MIMIC-III,5000,pram_b1,0.8640,1.0112,0.0157,0.0059
mortality_168h,MIMIC-III,5000,pram_b5,0.8776,0.9098,0.0167,0.0088
mortality_168h,MIMIC-III,5000,recal_iso,0.8558,0.8988,0.0156,0.0057
mortality_168h,MIMIC-III,full,b1_base,0.8600,0.9070,0.0152,0.0058
mortality_168h,MIMIC-III,full,b5_base,0.8779,0.8936,0.0166,0.0088
mortality_168h,MIMIC-III,full,local_cb,0.8756,1.0454,0.0177,0.0081
mortality_168h,MIMIC-III,full,local_lr,0.8611,0.9613,0.0162,0.0060
mortality_168h,MIMIC-III,full,pooled_lr,0.8631,0.9488,0.0160,0.0056
mortality_168h,MIMIC-III,full,pram_b1,0.8675,1.0297,0.0162,0.0061
mortality_168h,MIMIC-III,full,pram_b5,0.8782,0.9197,0.0166,0.0087
mortality_168h,MIMIC-III,full,recal_iso,0.8553,0.9450,0.0151,0.0061
mortality_168h,MIMIC-IV,1000,b1_base,0.8833,1.0064,0.0173,0.0077
mortality_168h,MIMIC-IV,1000,b5_base,0.8930,0.9809,0.0175,0.0096
mortality_168h,MIMIC-IV,1000,local_cb,0.8406,0.7939,0.0088,0.0041
mortality_168h,MIMIC-IV,1000,local_lr,0.8314,0.6291,0.0122,0.0046
mortality_168h,MIMIC-IV,1000,pooled_lr,0.8834,1.0043,0.0173,0.0078
mortality_168h,MIMIC-IV,1000,pram_b1,0.8834,1.0814,0.0171,0.0078
mortality_168h,MIMIC-IV,1000,pram_b5,0.8925,1.0126,0.0174,0.0093
mortality_168h,MIMIC-IV,1000,recal_iso,0.8744,0.6892,0.0168,0.0069
mortality_168h,MIMIC-IV,2000,b1_base,0.8833,1.0064,0.0173,0.0077
mortality_168h,MIMIC-IV,2000,b5_base,0.8930,0.9809,0.0175,0.0096
mortality_168h,MIMIC-IV,2000,local_cb,0.8613,0.8952,0.0115,0.0054
mortality_168h,MIMIC-IV,2000,local_lr,0.8539,0.7646,0.0137,0.0061
mortality_168h,MIMIC-IV,2000,pooled_lr,0.8834,1.0011,0.0172,0.0077
mortality_168h,MIMIC-IV,2000,pram_b1,0.8839,1.0789,0.0169,0.0076
mortality_168h,MIMIC-IV,2000,pram_b5,0.8919,1.0047,0.0175,0.0095
mortality_168h,MIMIC-IV,2000,recal_iso,0.8753,0.7433,0.0163,0.0070
mortality_168h,MIMIC-IV,500,b1_base,0.8833,1.0064,0.0173,0.0077
mortality_168h,MIMIC-IV,500,b5_base,0.8930,0.9809,0.0175,0.0096
mortality_168h,MIMIC-IV,500,local_cb,0.8170,0.6814,0.0062,0.0028
mortality_168h,MIMIC-IV,500,local_lr,0.8026,0.4668,0.0097,0.0023
mortality_168h,MIMIC-IV,500,pooled_lr,0.8833,1.0055,0.0172,0.0079
mortality_168h,MIMIC-IV,500,pram_b1,0.8830,1.0685,0.0171,0.0077
mortality_168h,MIMIC-IV,500,pram_b5,0.8903,1.0184,0.0175,0.0094
mortality_168h,MIMIC-IV,500,recal_iso,0.8648,0.3603,0.0164,0.0075
mortality_168h,MIMIC-IV,5000,b1_base,0.8833,1.0064,0.0173,0.0077
mortality_168h,MIMIC-IV,5000,b5_base,0.8930,0.9809,0.0175,0.0096
mortality_168h,MIMIC-IV,5000,local_cb,0.8804,0.9925,0.0151,0.0071
mortality_168h,MIMIC-IV,5000,local_lr,0.8683,0.8839,0.0154,0.0072
mortality_168h,MIMIC-IV,5000,pooled_lr,0.8829,0.9964,0.0169,0.0077
mortality_168h,MIMIC-IV,5000,pram_b1,0.8867,1.1056,0.0171,0.0079
mortality_168h,MIMIC-IV,5000,pram_b5,0.8931,1.0011,0.0175,0.0094
mortality_168h,MIMIC-IV,5000,recal_iso,0.8775,0.8782,0.0170,0.0069
mortality_168h,MIMIC-IV,full,b1_base,0.8833,1.0064,0.0173,0.0077
mortality_168h,MIMIC-IV,full,b5_base,0.8930,0.9809,0.0175,0.0096
mortality_168h,MIMIC-IV,full,local_cb,0.8870,1.0285,0.0162,0.0083
mortality_168h,MIMIC-IV,full,local_lr,0.8745,0.9162,0.0160,0.0074
mortality_168h,MIMIC-IV,full,pooled_lr,0.8828,0.9941,0.0171,0.0074
mortality_168h,MIMIC-IV,full,pram_b1,0.8866,1.0944,0.0172,0.0078
mortality_168h,MIMIC-IV,full,pram_b5,0.8927,1.0028,0.0173,0.0095
mortality_168h,MIMIC-IV,full,recal_iso,0.8764,0.9317,0.0171,0.0071
mortality_168h,eICU,1000,b1_base,0.7745,0.4063,0.0076,-0.0069
mortality_168h,eICU,1000,b5_base,0.8239,0.7494,0.0118,0.0013
mortality_168h,eICU,1000,local_cb,0.7868,0.7143,0.0056,0.0018
mortality_168h,eICU,1000,local_lr,0.7612,0.5492,0.0076,0.0005
mortality_168h,eICU,1000,pooled_lr,0.8070,0.7155,0.0108,0.0007
mortality_168h,eICU,1000,pram_b1,0.7944,0.6874,0.0085,-0.0031
mortality_168h,eICU,1000,pram_b5,0.8240,0.8134,0.0119,0.0020
mortality_168h,eICU,1000,recal_iso,0.7634,0.5605,0.0077,-0.0006
mortality_168h,eICU,2000,b1_base,0.7745,0.4063,0.0076,-0.0069
mortality_168h,eICU,2000,b5_base,0.8239,0.7494,0.0118,0.0013
mortality_168h,eICU,2000,local_cb,0.8028,0.8242,0.0081,0.0024
mortality_168h,eICU,2000,local_lr,0.7955,0.7489,0.0098,0.0024
mortality_168h,eICU,2000,pooled_lr,0.8114,0.7588,0.0113,0.0017
mortality_168h,eICU,2000,pram_b1,0.7990,0.7607,0.0088,-0.0023
mortality_168h,eICU,2000,pram_b5,0.8242,0.8417,0.0121,0.0023
mortality_168h,eICU,2000,recal_iso,0.7683,0.7667,0.0077,-0.0005
mortality_168h,eICU,500,b1_base,0.7745,0.4063,0.0076,-0.0069
mortality_168h,eICU,500,b5_base,0.8239,0.7494,0.0118,0.0013
mortality_168h,eICU,500,local_cb,0.7611,0.6268,0.0036,0.0010
mortality_168h,eICU,500,local_lr,0.7212,0.3933,0.0056,-0.0011
mortality_168h,eICU,500,pooled_lr,0.8021,0.6648,0.0103,-0.0004
mortality_168h,eICU,500,pram_b1,0.7913,0.6880,0.0083,-0.0032
mortality_168h,eICU,500,pram_b5,0.8225,0.8326,0.0119,0.0021
mortality_168h,eICU,500,recal_iso,0.7565,0.4112,0.0068,-0.0011
mortality_168h,eICU,5000,b1_base,0.7745,0.4063,0.0076,-0.0069
mortality_168h,eICU,5000,b5_base,0.8239,0.7494,0.0118,0.0013
mortality_168h,eICU,5000,local_cb,0.8238,0.9384,0.0111,0.0036
mortality_168h,eICU,5000,local_lr,0.8133,0.8759,0.0111,0.0037
mortality_168h,eICU,5000,pooled_lr,0.8147,0.8107,0.0116,0.0027
mortality_168h,eICU,5000,pram_b1,0.8029,0.7702,0.0089,-0.0020
mortality_168h,eICU,5000,pram_b5,0.8268,0.8499,0.0123,0.0024
mortality_168h,eICU,5000,recal_iso,0.7706,0.8475,0.0079,-0.0003
mortality_168h,eICU,full,b1_base,0.7745,0.4063,0.0076,-0.0069
mortality_168h,eICU,full,b5_base,0.8239,0.7494,0.0118,0.0013
mortality_168h,eICU,full,local_cb,0.8472,1.0484,0.0146,0.0067
mortality_168h,eICU,full,local_lr,0.8263,0.9862,0.0118,0.0047
mortality_168h,eICU,full,pooled_lr,0.8256,0.9583,0.0121,0.0045
mortality_168h,eICU,full,pram_b1,0.8103,0.7775,0.0095,-0.0014
mortality_168h,eICU,full,pram_b5,0.8310,0.9118,0.0127,0.0034
mortality_168h,eICU,full,recal_iso,0.7726,0.9799,0.0083,-0.0003"""      # reference values: AUROC, calibration slope and net benefit at 0.1 and 0.2, to four decimals
REPRO = pd.DataFrame()
if not (DATA_OK and RUN_MATRIX and "F42" in FS_RUN):
    status("reference check of the 42-column grid", "off" if not RUN_MATRIX else "not available", "main inputs absent, or RUN_MATRIX=False, or no F42")
else:
    _ref = pd.read_csv(io.StringIO(REF_A23_CSV), dtype={"bank": str})
    if N_DRAWS_MAIN != 10:
        emit(f"**Note** , the number of draws in this run is {N_DRAWS_MAIN} and the reference values use 10 (check profile, etc.). The medians differ, so a mismatch in the comparison below is normal. See the comparison for the paper in `RUN_PROFILE = \"full\"` .")
    # (1) frozen model AUROC (independent of the bank)
    _fz = []
    for (_oc, _tg), _s in _ref[_ref["method"].isin(["b1_base", "b5_base"])].groupby(["outcome", "target"]):
        _c = CTX[("F42", _oc, _tg)]
        for _m, _p in (("b1_base", _c["pb1"]), ("b5_base", _c["pb5"])):
            _r = float(_s[(_s["method"] == _m)]["auroc"].iloc[0])
            _o = float(roc_auc_score(_c["y"], _p))
            _fz.append({"outcome": _oc, "target": _tg, "method": _m, "ref": _r, "ours": _o, "abs_diff": abs(_r - _o), "match": abs(_r - _o) <= REPRO_TOL})
    _fz = pd.DataFrame(_fz)
    emit(f"- frozen model AUROC against the reference values: equal {int(_fz['match'].sum())}/{len(_fz)} (tolerance {REPRO_TOL})")
    # (2) four units
    _Rm, _Rmi = Resumable("matrix_F42"), Resumable("matrixinfo_F42")
    _specm = FS_SPEC["F42"]
    for _u4 in REPRO_UNITS:
        _u = ("F42",) + tuple(_u4)
        if _u4[2] not in _specm["banks"] and _u4[2] != "full":
            emit(f"  [skipped] comparison unit {_u4} bank is not in the bank list of this run")
            continue
        if _Rm.unit_done(_u):
            continue
        _t0 = time.time()
        _rows, _irows = run_matrix_unit(CTX[("F42", _u4[0], _u4[1])], "F42", _u4[0], _u4[1], _u4[2], N_DRAWS_MAIN, _specm["groups"], orig_slope=True)
        for _r in _rows:
            _Rm.add_row(_r)
        for _r in _irows:
            _Rmi.add_row(_r)
        _Rm.mark_unit(_u)
        _Rmi.mark_unit(_u)
        print(f"  [DONE] comparison unit {_u4} ({time.time() - _t0:.0f}s)", flush=True)
    _mdf = _Rm.df()
    _cmp = []
    _met = {"auroc": "auroc", "slope": "slope_orig" if (len(_mdf) and "slope_orig" in _mdf.columns) else "slope", "nb0.1": "nb@0.1", "nb0.2": "nb@0.2"}
    for _u4 in REPRO_UNITS:
        _sub = _mdf[(_mdf["outcome"] == _u4[0]) & (_mdf["target"] == _u4[1]) & (_mdf["bank"] == str(_u4[2]))] if len(_mdf) else _mdf
        if len(_sub) == 0:
            continue
        _med = _sub.groupby("method")[list(_met.values())].median()
        for _, _rr in _ref[(_ref["outcome"] == _u4[0]) & (_ref["target"] == _u4[1]) & (_ref["bank"] == str(_u4[2]))].iterrows():
            if _rr["method"] not in _med.index:
                continue
            for _k, _col in _met.items():
                _cmp.append({"outcome": _u4[0], "target": _u4[1], "bank": str(_u4[2]), "method": _rr["method"], "metric": _k,
                             "ref": float(_rr[_k]), "ours": float(_med.loc[_rr["method"], _col]),
                             "abs_diff": abs(float(_rr[_k]) - float(_med.loc[_rr["method"], _col]))})
    REPRO = pd.DataFrame(_cmp)
    if len(REPRO):
        REPRO["match"] = REPRO["abs_diff"] <= REPRO_TOL
        _sm = REPRO.groupby(["metric"]).agg(cells=("match", "size"), matched=("match", "sum"), max_abs_diff=("abs_diff", "max")).reset_index()
        emit_table(f"summary of the reference check of the 42-column grid (tolerance {REPRO_TOL}, the reference values have four decimals, so a rounding error is allowed)", _sm, digits=5)
        _sm2 = REPRO[REPRO["metric"] == "auroc"].groupby("method").agg(cells=("match", "size"), matched=("match", "sum"), max_abs_diff=("abs_diff", "max")).reset_index()
        emit_table("AUROC reproduction by method", _sm2, digits=5)
        _bad = REPRO[~REPRO["match"]].sort_values("abs_diff", ascending=False)
        if len(_bad):
            emit(f"**mismatched cells {len(_bad)} are printed as they are.** If these values mismatch, the 42-column values of this run should not be used for Table 3 and Supplementary S6 before the cause is understood.")
            emit_table("mismatched cells (largest first)", _bad, digits=5, max_rows=60)
        else:
            emit("- all compared cells equal the reference values to the fourth decimal.")
        save_csv(REPRO, "repro_check_vs_stored_grid")
    save_csv(_fz, "repro_check_frozen_auroc")
    status("reference check of the 42-column grid", "done" if len(REPRO) else "not available",
           (f"mismatch {int((~REPRO['match']).sum())}/{len(REPRO)}" if len(REPRO) else ""))


## Strategy grid: main analysis, update hierarchy, retrieval then Platt, 33-column set

The grid uses fixed draw keys (`m23`), so the same patients are drawn in every run. For each feature set (F42, F33) the notebook runs bank size x `N_DRAWS_MAIN` draws x 5 folds. One draw is one bank result that pools the 5-fold test predictions.

Strategies are computed as groups.

| Group | Strategy names | Note |
|---|---|---|
| frozen | `b1_base` (frozen logistic), `b5_base` (frozen CatBoost) | |
| pram | `pram_b1`, `pram_b5` | mixing weight chosen by 3 inner folds within the bank (blocks self-retrieval) |
| recal | `recal_iso` (isotonic on the frozen logistic), `platt_b1` (Platt scaling on the frozen logistic), `platt_b5` (Platt on CatBoost) | CatBoost gets Platt only |
| local | `local_lr`, `pooled_lr` | bank only; bank and development records |
| localcb | `local_cb` | bank only |
| anch | `anchored_ridge` | prediction of the model revision (step 3), shrunk toward the development coefficients, penalty chosen within the bank |
| hier | `upd1_b1` (intercept only), `upd2_b1` (intercept and slope, unpenalized), `upd3_b1` (unpenalized revision), `ct_b1` (closed-testing choice) | on the frozen logistic model only |
| combo | `pram_platt_b1`, `pram_platt_b5` (Platt after retrieval) | Platt fitted on within-bank OOF mixture predictions |
| sel (A11) | `sel_logloss`, `sel_nb` | off by default |

`platt_b1` is Platt scaling with the scikit-learn default regularization. `upd2_b1` is the same line fitted without regularization (needed for the likelihood ratio of closed testing). The two are expected to be nearly equal.


In [ ]:
# ================================================================
# PRAM run of the strategy grid (resumable)
# ================================================================
MX, MXINFO = {}, {}
if not (DATA_OK and RUN_MATRIX):
    status("strategy grid", "not available", "main inputs absent" if not DATA_OK else "RUN_MATRIX=False")
else:
    t_all = time.time()
    for _fs in FS_RUN:
        _spec = FS_SPEC[_fs]
        _R = Resumable(f"matrix_{_fs}")
        _RI = Resumable(f"matrixinfo_{_fs}")
        _units = [(_fs, oc, tg, bk) for oc in OUTCOMES for tg in _spec["targets"] for bk in _spec["banks"]]
        emit(f"### grid {_fs} , units {len(_units)} (completed {sum(_R.unit_done(u) for u in _units)} )")
        for u in _units:
            if _R.unit_done(u):
                print("  [SKIP]", u)
                continue
            t0 = time.time()
            rows, irows = run_matrix_unit(CTX[(u[0], u[1], u[2])], u[0], u[1], u[2], u[3], N_DRAWS_MAIN, _spec["groups"])
            for r in rows:
                _R.add_row(r)
            for r in irows:
                _RI.add_row(r)
            _R.mark_unit(u)
            _RI.mark_unit(u)
            print(f"  [DONE] {u[1]} / {u[2]} / bank={u[3]}  ({time.time() - t0:.0f}s, {len(rows)} rows)", flush=True)
        MX[_fs], MXINFO[_fs] = _R.df(), _RI.df()
        save_csv(MX[_fs], f"matrix_draws_{_fs}")
        save_csv(MXINFO[_fs], f"matrix_fold_info_{_fs}")
    status("strategy grid", "done", f"{(time.time() - t_all) / 60:.0f} min")


## Main analysis table (42-column set) and 33-column comparison

Two tables are produced.

1. **Main analysis table**: values for Table 3 and Supplementary Tables S6 and S8. Columns: AUROC, calibration slope, ECE, net benefit (outcome-specific representative threshold `T_SITE` and 20%), and the AUROC difference over the frozen logistic (median and quartiles of the paired differences between the same draws).
2. **33-column comparison**: the 42-column and 33-column sets side by side for the same draws and the same patients.

Below the two tables, the F42 values are compared with a reference result file of the 42-column grid when `reference/resume/a23_strategy_matrix.jsonl` exists in the working folder. If it is absent, the searched path is printed and the cell moves on.


In [ ]:
# ================================================================
# PRAM main analysis table | A8 comparison | check against the reference 42-column grid file
# ================================================================
T3, A8TAB = pd.DataFrame(), pd.DataFrame()
if not (DATA_OK and RUN_MATRIX and MX):
    status("main analysis table", "not available", "no grid")
    status("A8 33-column set", "not available", "no grid")
else:
    _t3 = []
    for _fs in [f for f in ("F42", "F33") if f in MX]:
        _banks = pick_report_banks([b for b in FS_SPEC[_fs]["banks"]])
        _d = MX[_fs]
        _d = _d[_d["bank"].isin([str(b) for b in _banks])]
        _g = draw_median_table(_d, M_EIGHT)
        _gaps = pd.concat([paired_gap(_d, m, "b1_base") for m in M_EIGHT if m != "b1_base"], ignore_index=True)
        _g = _g.merge(_gaps[["fs", "outcome", "target", "bank", "method", "gap_median", "gap_q25", "gap_q75"]],
                      on=["fs", "outcome", "target", "bank", "method"], how="left")
        _t3.append(_g)
    T3 = pd.concat(_t3, ignore_index=True)
    T3 = T3.rename(columns={"gap_median": "dAUROC_vs_b1", "gap_q25": "dAUROC_q25", "gap_q75": "dAUROC_q75"})
    for _fs in T3["fs"].unique():
        _show = T3[T3["fs"] == _fs].drop(columns=["fs"])
        emit_table(f"main analysis table , {_fs} (median over draws. Net benefit representative threshold = midpoint of the outcome-specific range {T_SITE})", _show, max_rows=200)
    save_csv(T3, "main_table_F42_F33")

    # ---- A8: 42-column set versus 33-column set ----
    if "F42" in MX and "F33" in MX:
        _rows = []
        _m8 = ("b1_base", "b5_base", "pram_b1", "pram_b5", "recal_iso", "local_lr", "local_cb", "pooled_lr")
        for _fs in ("F42", "F33"):
            _d = MX[_fs]
            _g = draw_median_table(_d, list(_m8))
            _gaps = pd.concat([paired_gap(_d, m, "b1_base") for m in _m8 if m != "b1_base"], ignore_index=True)
            _g = _g.merge(_gaps[["fs", "outcome", "target", "bank", "method", "gap_median"]],
                          on=["fs", "outcome", "target", "bank", "method"], how="left")
            _rows.append(_g)
        _all = pd.concat(_rows, ignore_index=True)
        if "gap_median" not in _all.columns:
            _all["gap_median"] = np.nan
        _w = _all.pivot_table(index=["outcome", "target", "bank", "method"], columns="fs",
                              values=["auroc", "gap_median", "nb_site", "slope"], aggfunc="first")
        _w.columns = [f"{a}_{b}" for a, b in _w.columns]
        A8TAB = _w.reset_index()
        A8TAB["dAUROC_gain_change_33_minus_42"] = A8TAB["gap_median_F33"] - A8TAB["gap_median_F42"]
        A8TAB["bank_order"] = [bank_sort_key(b) for b in A8TAB["bank"]]
        A8TAB = A8TAB.sort_values(["outcome", "target", "method", "bank_order"]).drop(columns=["bank_order"])
        emit_table("A8 , 42-column set versus 33-column set (gap = median over draws of the AUROC gain over the frozen logistic, the set is at the end of the column name)", A8TAB, max_rows=250)
        save_csv(A8TAB, "A8_compare_F42_F33")
        S2_REF = pd.DataFrame([
            ["AKI_168h", "1000", 0.6910, 0.0094, 0.7034, 0.0021], ["AKI_168h", "full", 0.6910, 0.0266, 0.7034, 0.0199],
            ["mortality_168h", "1000", 0.7785, 0.0205, 0.8113, -0.0004], ["mortality_168h", "full", 0.7785, 0.0357, 0.8113, 0.0120]],
            columns=["outcome", "bank", "S2_C0_frozen_auroc", "S2_C0_gain", "S2_C1_frozen_auroc", "S2_C1_gain"])
        emit_table("reference , the eICU values of Supplementary Table S2 (a different draw design, so not the same draws. Compare only direction and size)", S2_REF)
    for _fs in [f for f in ("F42_noTemp", "F42_noINR") if f in MX]:
        _d = MX[_fs]
        _g = draw_median_table(_d, ["b1_base", "pram_b1", "pram_b5"])
        _pg = paired_gap(_d, "pram_b1", "b1_base")
        if len(_pg):
            _g = _g.merge(_pg[["outcome", "target", "bank", "gap_median"]], on=["outcome", "target", "bank"], how="left")
        emit_table(f"A8 , {_fs} (eICU)", _g.drop(columns=["fs"]), max_rows=60)
    status("main analysis table", "done", "")
    status("A8 33-column set", "done" if ("F33" in MX) else "not available", "" if ("F33" in MX) else "no F33 in FS_RUN")

    # ---- check against the reference 42-column grid file ----
    _ref = Path("reference") / "resume" / "a23_strategy_matrix.jsonl"
    if _ref.is_file() and "F42" in MX:
        _old = pd.read_json(_ref, lines=True)
        _old["bank"] = _old["bank"].astype(str)
        _om = _old.groupby(["outcome", "target", "bank", "method"])["auroc"].median().rename("auroc_old")
        _nm = MX["F42"].groupby(["outcome", "target", "bank", "method"])["auroc"].median().rename("auroc_new")
        _cmp = pd.concat([_om, _nm], axis=1).dropna().reset_index()
        _cmp["abs_diff"] = (_cmp["auroc_old"] - _cmp["auroc_new"]).abs()
        emit(f"**check against the reference 42-column grid file** `{_ref.resolve()} ` , comparison {len(_cmp)} cells, maximum difference {_cmp['abs_diff'].max():.5f}, "
             f"within 0.002 {float((_cmp['abs_diff'] <= 0.002).mean()):.0%}")
        emit_table("top 10 cells with the largest difference", _cmp.sort_values("abs_diff", ascending=False).head(10))
        status("check against the reference 42-column grid file", "done", f"maximum difference {_cmp['abs_diff'].max():.4f}")
    else:
        emit(f"**check against the reference 42-column grid file skipped** , the reference 42-column grid file does not exist. Path searched `{_ref.resolve()} `. "
             "If the reference file is placed in reference/resume, the comparison is made automatically.")
        status("check against the reference 42-column grid file", "skipped", "no reference result file")


## A9. A modern model: TabPFN v2 as the frozen model

**Question**: does the retrieval layer remain harmless, and still gain, on top of a stronger frozen model? The relation between base-model strength and the gain from retrieval (rank correlations of -0.69 and -0.84 over 5 series and 10 development splits) is extended by one more series.

**Method**: on the MIMIC-IV training 56% with the same 42 features (43 columns after transform), TabPFN is fitted on stratified subsamples (10,000 rows on a GPU, 2,000 rows on a CPU) and **frozen** (no weight update; an ensemble of 3 subsamples on a GPU, 1 on a CPU). Targets are eICU and MIMIC-III, bank sizes 500, 1,000, 5,000 and full, with the same 5 folds and draw keys as the grid, with retrieval (`pram_on_tabpfn`) and recalibration on top. For comparison, the same layers on the frozen logistic model are given for the same patients.

**Without a GPU**: the target cohort is reduced by stratification (`TABPFN_CPU_TARGET_CAP`) and the time of the first 200 rows is measured to print an expected time first. If it exceeds `TABPFN_MAX_MINUTES` the sample is reduced further. A reduced run is marked as reduced in the table header, and reduced results are not merged with the main table.

**Fixing the v2 weights**: the code selects v2 explicitly with `TabPFNClassifier.create_default_for_version(ModelVersion.V2)` (read from the sources of tabpfn 6.0.0 and 9.1.0; versions 2.x have only v2 weights, so the default constructor gives v2; versions in between have not been tested). The default weights of the latest package (9.1.0) are v3.5, so the default constructor would not give v2. The installed version, selection method, weight file name and `model_version` are printed in the first table of this cell and saved in `A9_tabpfn_model_info.csv` and in the `model_version` column of the result table. If v2 cannot be selected, the run continues with the default weights, `model_version` is printed as unconfirmed, and the result must not be used as a v2 result. The context limit is the recommended 10,000 rows of v2. The pretrained weights are downloaded on the first run (internet access is needed).

**Reading**: `gap` is the AUROC difference between the layer added and the frozen model alone in the same draws. Retrieval on learned representations overlaps with other candidates, so only the probabilities of the tabular pretrained model are used here and not its representation.


In [ ]:
# ================================================================
# PRAM A9. TabPFN v2 (modern model)
# ================================================================
TPFN = pd.DataFrame()
_R9, _a9_todo = None, {}
if DATA_OK and RUN_A9 and "F42" in FS_RUN:
    # which cells remain is known even without tabpfn. If only finished cells exist, TabPFN is neither fitted nor used to predict (so that finished computations are not rerun)
    _R9 = Resumable("a9_tabpfn")
    _cap0 = None if HAVE_CUDA else TABPFN_CPU_TARGET_CAP
    for _oc in OUTCOMES:
        for _tg in TABPFN_TARGETS:
            _pool0 = int(len(stratified_cap(CTX[("F42", _oc, _tg)]["y"], _cap0, f"a9cap|{_oc}|{_tg}")) * (N_FOLDS - 1) / N_FOLDS)
            _a9_todo[(_oc, _tg)] = [_bk for _bk in TABPFN_BANKS if (_bk == "full" or _bk <= _pool0) and not _R9.unit_done(("F42", _oc, _tg, _bk))]
if _R9 is None:
    status("A9 TabPFN", "off" if not RUN_A9 else "not available", "main inputs absent or RUN_A9=False")
elif any(_a9_todo.values()) and not HAVE_TABPFN:
    emit("**A9 not available** , the tabpfn package is missing. Run this cell again after `pip install tabpfn` (see the TabPFN documentation for the license terms and the weight download).")
    status("A9 TabPFN", "not available", "tabpfn not installed")
else:
    _dev_name = "cuda" if HAVE_CUDA else "cpu"
    _n_ctx = TABPFN_CTX_GPU if HAVE_CUDA else TABPFN_CTX_CPU
    _n_sub = TABPFN_SUBSETS_GPU if HAVE_CUDA else TABPFN_SUBSETS_CPU
    _cap = None if HAVE_CUDA else TABPFN_CPU_TARGET_CAP
    if any(_a9_todo.values()):
        emit(f"- device {_dev_name} | development subsample {_n_ctx} rows x {_n_sub} | target cohort cap {_cap if _cap else 'absent'}")
    else:
        emit("- A9: all cells are finished. TabPFN is not fitted and the finished results are loaded.")
    _g9 = ("frozen", "pram", "recal") + (("hier",) if RUN_A4 else ())
    _shrunk = False
    _a9_fail, _info_shown = None, False
    for _oc in OUTCOMES:
        if not any(_a9_todo[(_oc, _tg)] for _tg in TABPFN_TARGETS):
            print("  [SKIP] A9", _oc, "all cells are finished (TabPFN fit and prediction are skipped)")
            continue
        _t0 = time.time()
        _dev = DEV[("F42", _oc)]
        _subs = {tg: stratified_cap(CTX[("F42", _oc, tg)]["y"], _cap, f"a9cap|{_oc}|{tg}") for tg in TABPFN_TARGETS}
        _fit_err = None
        try:
            _clfs = tabpfn_fit_subsets(_dev["Xs_src"], _dev["ys_src"], _n_ctx, _n_sub, _dev_name, _oc)
        except Exception as exc:  # noqa: BLE001
            _fit_err = exc
        if TABPFN_INFO and not _info_shown:
            _full, _csv_rows = tabpfn_info_rows()
            emit_table("A9 , TabPFN installation and weights (model_version must be v2 for the results to be used as v2 results)", pd.DataFrame(_full, columns=["item", "value"]))
            save_csv(pd.DataFrame(_csv_rows, columns=["item", "value"]), "A9_tabpfn_model_info")
            _info_shown = True
        if _fit_err is not None:
            emit(f"**A9 not available** , the TabPFN fit failed ({type(_fit_err).__name__}: {str(_fit_err)[:200]}). The weights may not have been downloaded (internet or license agreement) or the context limit may have been exceeded. The installed version and the selection method are in the table above. The other sections below continue.")
            _a9_fail = f"{type(_fit_err).__name__}: {str(_fit_err)[:120]}"
            break
        _probe_tg = TABPFN_TARGETS[-1]
        _probe = CTX[("F42", _oc, _probe_tg)]["Xg"][_subs[_probe_tg][:200]]
        _t1 = time.time()
        tabpfn_predict(_clfs[0], _probe)
        _per_row = (time.time() - _t1) / max(len(_probe), 1)
        _est = _per_row * sum(len(s) for s in _subs.values()) * _n_sub / 60.0
        emit(f"- {_oc}: development fit {time.time() - _t0:.0f} s | expected prediction about {_est:.1f} min (estimated from the 200-row time)")
        if _est > TABPFN_MAX_MINUTES:
            _newcap = max(1000, int(max(len(s) for s in _subs.values()) * TABPFN_MAX_MINUTES / _est))
            _subs = {tg: stratified_cap(CTX[("F42", _oc, tg)]["y"], _newcap, f"a9cap|{_oc}|{tg}") for tg in TABPFN_TARGETS}
            _shrunk = True
            emit(f"  expected {TABPFN_MAX_MINUTES} min exceeds the limit, so the target cohort is reduced to {_newcap} rows (marked reduced in the table)")
        for _tg in TABPFN_TARGETS:
            if not _a9_todo[(_oc, _tg)]:
                continue
            _c0 = CTX[("F42", _oc, _tg)]
            _sub = _subs[_tg]
            try:
                _p = tabpfn_frozen_probs(_clfs, _c0["Xg"][_sub])
            except Exception as exc:  # noqa: BLE001
                emit(f"**A9 not available ({_oc} {_tg})** , the prediction failed ({type(exc).__name__}: {str(exc)[:200]}). The other sections below continue.")
                _a9_fail = f"{type(exc).__name__}: {str(exc)[:120]}"
                break
            _c9 = {"Xg": _c0["Xg"][_sub], "y": _c0["y"][_sub], "pb1": _p, "pb5": _c0["pb1"][_sub],
                   "stay_ids": _c0["stay_ids"][_sub], "outcome": _oc, "d": _c0["d"]}
            _pool = int(len(_sub) * (N_FOLDS - 1) / N_FOLDS)
            for _bk in TABPFN_BANKS:
                _u = ("F42", _oc, _tg, _bk)
                if _R9.unit_done(_u):
                    continue
                if _bk != "full" and _bk > _pool:
                    emit(f"  [skipped] {_oc} {_tg} bank {_bk} > pool {_pool}")
                    continue
                _rows, _ = run_matrix_unit(_c9, "TABPFN", _oc, _tg, _bk, N_DRAWS_MAIN, _g9)
                for _r in _rows:
                    _r["reduced"] = bool(_cap or _shrunk)
                    _r["model_version"] = TABPFN_INFO.get("model_version", "unknown")
                    _R9.add_row(_r)
                _R9.mark_unit(_u)
                print(f"  [DONE] A9 {_oc} / {_tg} / bank={_bk}", flush=True)
    TPFN = _R9.df()
    # if this run did not fit TabPFN (only finished cells), the marker values are read from the stored rows
    _mv = TABPFN_INFO.get("model_version") or (str(TPFN["model_version"].iloc[0]) if len(TPFN) and "model_version" in TPFN.columns else "unknown")
    _reduced = bool(_cap or _shrunk or (len(TPFN) and "reduced" in TPFN.columns and TPFN["reduced"].any()))
    if len(TPFN):
        TPFN["method"] = TPFN["method"].map(A9_RENAME).fillna(TPFN["method"])
        _gt = draw_median_table(TPFN, ["tabpfn_frozen", "logistic_frozen", "pram_on_tabpfn", "pram_on_logistic",
                                       "platt_on_tabpfn", "isotonic_on_tabpfn"])
        _gp = pd.concat([paired_gap(TPFN, "pram_on_tabpfn", "tabpfn_frozen"), paired_gap(TPFN, "pram_on_logistic", "logistic_frozen")],
                        ignore_index=True)
        _gt["reduced"] = _reduced
        _gt["model_version"] = _mv
        _gp["model_version"] = _mv
        emit_table("A9 , TabPFN frozen model with retrieval and recalibration (median over draws)", _gt.drop(columns=["fs"]), max_rows=200)
        emit_table("A9 , gain with retrieval added (between the same draws, versus frozen as is)", _gp.drop(columns=["fs"]), max_rows=100)
        save_csv(TPFN, "A9_tabpfn_draws")
        save_csv(_gt, "A9_tabpfn_summary")
        save_csv(_gp, "A9_tabpfn_retrieval_gain")
        # does the gain shrink as the base model gets stronger: collect (base AUROC, gain) points at the full bank and compute the rank correlation
        _pts = []
        for _nm, _base, _m in (("tabpfn", "tabpfn_frozen", "pram_on_tabpfn"), ("logistic", "logistic_frozen", "pram_on_logistic")):
            _gg = paired_gap(TPFN[TPFN["bank"] == "full"], _m, _base)
            _bb = TPFN[(TPFN["bank"] == "full") & (TPFN["method"] == _base)].groupby(["outcome", "target"])["auroc"].median()
            for _r in _gg.itertuples():
                _pts.append({"base_model": _nm, "outcome": _r.outcome, "target": _r.target,
                             "base_auroc": float(_bb.loc[(_r.outcome, _r.target)]), "gain": float(_r.gap_median)})
        if "F42" in MX:
            _d5 = MX["F42"][MX["F42"]["bank"] == "full"]
            _gg = paired_gap(_d5, "pram_b5", "b5_base")
            _gg = _gg[_gg["target"].isin(TABPFN_TARGETS)]
            _bb = _d5[_d5["method"] == "b5_base"].groupby(["outcome", "target"])["auroc"].median()
            for _r in _gg.itertuples():
                _pts.append({"base_model": "catboost", "outcome": _r.outcome, "target": _r.target,
                             "base_auroc": float(_bb.loc[(_r.outcome, _r.target)]), "gain": float(_r.gap_median)})
        _pt = pd.DataFrame(_pts)
        if len(_pt) >= 4:
            _rho, _pv = scipy_stats.spearmanr(_pt["base_auroc"], _pt["gain"])
            emit(f"- rank correlation between base AUROC and retrieval gain (full bank, points {len(_pt)} = cohort x outcome x base model): rho = {_rho:.3f} (p = {_pv:.3f}). "
                 "the points share the same target cohorts, so they are not independent. Look at the direction only. The -0.69 and -0.84 reported for the main analysis are for 5 series and 10 development splits.")
            save_csv(_pt, "A9_base_strength_vs_gain")
    if _a9_fail:
        status("A9 TabPFN", "partly done" if len(TPFN) else "not available", "fit failed: " + _a9_fail)
    else:
        status("A9 TabPFN", "done" if len(TPFN) else "not available", ("sample reduced , " if _reduced else "") + "model_version " + _mv)


## N1. Temporal validation

The eICU patient table has the discharge year (`hospitaldischargeyear`, 2014 and 2015) and MIMIC-IV has the actual 3-year period (`anchor_year_group`). One more column of the same files is read; no preprocessing changes.

**Question**: do the main-analysis values hold when the bank is from the past and the test patients are later?

**Design**: the test-period patients are split into 2 folds stratified by outcome (`N_TIME_SPLITS`) and **the roles alternate**, one fold being the test. The bank is drawn at the same size under both policies, and both policies are scored on the same whole test patients, so the difference between policies is paired.

| Policy | Bank pool | Meaning |
|---|---|---|
| P | past period only | temporal validation |
| S | test period outside the test fold (the other fold) | same-period bank |
| R (off) | everything outside the test fold (periods mixed) | random bank as in the main analysis. Runs if added to `A1_POLICIES` |

Bank sizes are 500, 1,000, 2,000, 5,000 and max (the largest size that both P and S can supply; compared at the same size only), with `N_DRAWS_MAIN` draws per size. Five strategies are run: frozen logistic, retrieval on logistic, retrieval on CatBoost, Platt, local logistic (the needed frozen CatBoost and helper strategies come along).

| Analysis | Development | Bank (P) | Test |
|---|---|---|---|
| A1a | MIMIC-IV (the same frozen model as the main analysis) | eICU first discharge year | eICU last discharge year |
| A1b | MIMIC-III (CareVue 2001 to 2008, no year column) | all MIMIC-IV periods before the last cohort period (2008 to 2016) | the last MIMIC-IV period with cohort stays (2017 to 2019) |
| A1c (off) | model refitted on the earlier MIMIC-IV periods | MIMIC-IV periods before the last | the last MIMIC-IV period |

**MIMIC-IV period assignment** has two methods and the code chooses by the copy of the data (the period assignment row of the input check table in cell 1).

- Unit-transfer adjustment (when `patients` has `anchor_year`): actual year period = period lower and upper bounds + (shifted year of the stay - anchor_year). A stay is assigned as `strict` only when this 3-year period falls entirely inside one 3-year period, and dropped if it straddles. `majority` puts the stay in the period that holds most of its years (sensitivity).
- **Patient-level approximation (when `patients` has no `anchor_year`)**: all stays of the same patient get the `anchor_year_group` period of that patient. There is no unit-transfer adjustment. `intime` is not used to order patients (MIMIC-IV shifts dates by a different amount per patient, so the `intime` order between patients is not the calendar order). There is one rule, `patient_group`, so strict and majority are not run twice. The analysis name is `A1b_MIMIC3dev_patient_group`, and the output, the status table and the notes state that the patient-level approximation was used.

The period list is based on all `icustays`, but **the test period is the last period in which the cohort actually has stays**. Cohort stays and events per period are printed as a table (`A1_mimic4_period_counts`). If the test period is empty (fewer than 4 x `N_TIME_SPLITS` stays or only one outcome), the status table says not available: test period empty.

**Limits of the design**: the two years of eICU differ by one year, so a shift between them is expected to be small (standardized mean differences between blocks in Supplementary S11 are 0.013 to 0.057, against 0.174 between databases). A past bank has no advantage of being drawn from the same distribution as the test patients, so a difference between P and S is expected to be a disadvantage of P or none.


In [ ]:
# ================================================================
# PRAM A1a. eICU temporal validation (resumable)
# ================================================================
A1_RES, A1_INF, A1_NOTE, A1_PER = (Resumable("a1_time"), Resumable("a1_info"), Resumable("a1_notes"), Resumable("a1_period"))
A1_GROUPS = ("frozen", "pram", "recal", "local")     # the five strategies: frozen logistic | retrieval (on logistic and on CatBoost) | Platt | local logistic


def a1_run(unit, C, past_mask, test_mask, name, oc, groups):
    """runs one temporal validation unit and accumulates rows. Completed units are skipped."""
    if A1_RES.unit_done(unit):
        print("  [SKIP]", unit)
        return
    t0 = time.time()
    rows, irows, notes = run_temporal(C, past_mask, test_mask, name, oc, BANK_SIZES, N_DRAWS_MAIN, groups)
    for r in rows:
        A1_RES.add_row(r)
    for r in irows:
        A1_INF.add_row(r)
    for r in notes:
        A1_NOTE.add_row(r)
    A1_PER.add_row(period_summary(C, past_mask, test_mask, name, oc))
    A1_RES.mark_unit(unit)
    print(f"  [DONE] {name} {oc} ({time.time() - t0:.0f}s, {len(rows)} rows, note {len(notes)})", flush=True)


if not (DATA_OK and RUN_A1 and "F42" in FS_RUN):
    status("A1a eICU", "off" if not RUN_A1 else "not available", "main inputs absent or RUN_A1=False")
elif not (HAVE_EICU_YEAR and EICU_TAB is not None):
    emit("**A1a not available** , the `hospitaldischargeyear` column was not found in the eICU patient table. The paths searched are in the input check table of cell 1. Write the actual path in `EXTRA_RAW_PATHS` and run again from cell 1.")
    status("A1a eICU", "not available", "no discharge year column")
else:
    for _oc in OUTCOMES:
        _C = CTX[("F42", _oc, "eICU")]
        _yr = map_to_stays(EICU_TAB, "year", _C["stay_ids"])
        _ys = sorted(np.unique(_yr[np.isfinite(_yr)]))
        emit(f"- A1a {_oc}: discharge year {[int(v) for v in _ys]} | stays {len(_yr)} | no year {int(np.isnan(_yr).sum())} stays")
        if len(_ys) < 2:
            A1_NOTE.add_row({"analysis": "A1a", "outcome": _oc, "note": "only one year, so no temporal split is possible"})
            continue
        _name = f"A1a_eICU_{int(_ys[0])}_to_{int(_ys[-1])}"
        a1_run(("a1a", _oc), _C, _yr == _ys[0], _yr == _ys[-1], _name, _oc, A1_GROUPS)
    status("A1a eICU", "done", "")


In [ ]:
# ================================================================
# PRAM A1b and A1c. MIMIC-IV time chain (resumable)
# ================================================================
PER4, LABELS4 = None, []
A1B_FAIL, A1B_RAN = [], []                       # cells without a test period (rule, outcome, reason) | cells with a test period (rule, outcome)
if not (DATA_OK and RUN_A1 and "F42" in FS_RUN):
    status("A1b/c MIMIC-IV", "off" if not RUN_A1 else "not available", "main inputs absent or RUN_A1=False")
elif not HAVE_MIMIC4_YEAR:
    emit("**A1b and A1c not available** , " + " · ".join(M4_WHY) + ". The paths searched are in the input check table of cell 1. Write the actual path in `MIMIC4_ROOT` or `EXTRA_RAW_PATHS` and run again from cell 1.")
    status("A1b/c MIMIC-IV", "not available", " · ".join(M4_WHY))
else:
    PER4, LABELS4 = read_mimic4_periods()
    _n_g = len(LABELS4)
    _rules = list(YEAR_RULES) if M4_HAS_SHIFT else ["patient_group"]
    emit(f"- MIMIC-IV year periods {LABELS4} (based on all icustays stays, read from anchor_year_group of patients)")
    if M4_HAS_SHIFT:
        emit("- period assignment method: **unit-transfer adjusted** (the period is shifted by the difference between anchor_year of patients and intime of the stay)")
        _sh = pd.Series(DS[("MIMIC-IV", "AKI_168h")]["stay_ids"]).map(PER4["shift"])
        _cnt = pd.DataFrame({"shifted-year difference": ["0", "+-1", "+-2", "+-3", "others", "not in the table"],
                             "stays": [int((_sh == 0).sum()), int((_sh.abs() == 1).sum()), int((_sh.abs() == 2).sum()),
                                      int((_sh.abs() == 3).sum()), int((_sh.abs() > 3).sum()), int(_sh.isna().sum())]})
        emit_table("shifted year difference of the 33,455 cohort stays (= intime year - anchor_year). In strict, only multiples of 3 such as 0 and +-3 fall entirely inside one period", _cnt)
    else:
        emit(f"- period assignment method: **patient-level approximation: stays of the same patient in the same period, no anchor_year** (cause: {M4_APPROX_WHY}). "
             "there is no unit-transfer adjustment. Each stay uses the anchor_year_group of its patient as is, and `intime` is not used to order patients "
             "(MIMIC-IV shifts dates by a different amount per patient, so the `intime` order between patients is not the real calendar order). "
             "the period is the same even if a later stay of one patient falls in a year outside that period. Time order is interpreted at the period level only.")
    # the period list is based on all icustays. The test period is the last period in which the cohort actually has stays (the last period of icustays may be absent from the cohort).
    _rule_rows, _present = [], {}
    for _rule in _rules:
        for _oc in OUTCOMES:
            _d4 = DS[("MIMIC-IV", _oc)]
            _per = pd.Series(_d4["stay_ids"]).map(PER4[_rule]).fillna(-1).to_numpy(dtype=int)
            _present[(_rule, _oc)] = [g for g in range(_n_g) if (_per == g).any()]
            for _gi, _lab in enumerate(LABELS4):
                _m = _per == _gi
                _rule_rows.append({"assignment": M4_MODE, "rule": _rule, "outcome": _oc, "period": _lab, "cohort_stays": int(_m.sum()),
                                   "cohort_events": int(_d4["y"][_m].sum()), "in_cohort": bool(_m.any())})
            _m = _per < 0
            _rule_rows.append({"assignment": M4_MODE, "rule": _rule, "outcome": _oc, "period": "not assigned", "cohort_stays": int(_m.sum()),
                               "cohort_events": int(_d4["y"][_m].sum()), "in_cohort": False})
    _rr = pd.DataFrame(_rule_rows)
    emit_table("cohort stays and events per period (the period list is based on all icustays. A period without cohort stays has in_cohort=False and cannot be a test period)", _rr, max_rows=100)
    save_csv(_rr, "A1_mimic4_period_counts")
    for _rule in _rules:
        for _oc in OUTCOMES:
            _d4 = DS[("MIMIC-IV", _oc)]
            _per = pd.Series(_d4["stay_ids"]).map(PER4[_rule]).fillna(-1).to_numpy(dtype=int)
            _pres = _present[(_rule, _oc)]
            _why, _last, _n_te, _ev_te, _n_pa = None, None, 0, 0, 0
            if len(_pres) < 3:
                _why = f"periods with cohort stays are {len(_pres)} (fewer than 3)"
            else:
                _last = _pres[-1]
                _n_te = int((_per == _last).sum())
                _ev_te = int(_d4["y"][_per == _last].sum())
                _n_pa = int(((_per >= 0) & (_per < _last)).sum())
                if _n_te < 4 * N_TIME_SPLITS or _ev_te == 0 or _ev_te == _n_te:
                    _why = f"test period empty (last cohort period {LABELS4[_last]} stays {_n_te} | events {_ev_te})"
                elif _n_pa < 50:
                    _why = f"past bank pool insufficient (past period stays {_n_pa} < 50)"
            if _why:
                emit(f"**A1b not available: {_why}** , rule {_rule} outcome {_oc}")
                A1B_FAIL.append((_rule, _oc, _why))
                continue
            A1B_RAN.append((_rule, _oc))
            emit(f"- A1b {_rule} {_oc}: test period {LABELS4[_last]} (stays {_n_te} | events {_ev_te}), past bank period {[LABELS4[g] for g in _pres[:-1]]} (stays {_n_pa})")
            # A1b: development MIMIC-III, target the whole MIMIC-IV cohort
            _ub = ("a1b", _rule, _oc)
            if not A1_RES.unit_done(_ub):
                if not M4_HAS_SHIFT:
                    A1_NOTE.add_row({"analysis": f"A1b_MIMIC3dev_{_rule}", "outcome": _oc,
                                     "note": "patient-level approximation: stays of the same patient in the same period, no anchor_year (intime is not used to order patients)"})
                _d3 = DS[("MIMIC-III", _oc)]
                _tr3, _, _ = split_56_14_30(_d3["y"])
                _dev3 = build_dev(_d3["X_df"].iloc[_tr3], _d3["y"][_tr3], FS_COLS["F42"], CAT_REF, NUM_REF, f"A1b|{_oc}")
                _Cb = make_ctx(_dev3, _d4["X_df"], _d4["y"], _d4["stay_ids"], _oc)
                a1_run(_ub, _Cb, (_per >= 0) & (_per < _last), _per == _last, f"A1b_MIMIC3dev_{_rule}", _oc, A1_GROUPS)
            else:
                print("  [SKIP]", _ub)
            # A1c: development on the earlier MIMIC-IV periods, target the last two MIMIC-IV periods (only when RUN_A1C is on)
            _uc = ("a1c", _rule, _oc)
            if not RUN_A1C:
                continue
            if not A1_RES.unit_done(_uc):
                _prev = _pres[-2]
                _early = np.where((_per >= 0) & (_per < _prev))[0]
                _late = np.where(_per >= _prev)[0]
                _devc = build_dev(_d4["X_df"].iloc[_early], _d4["y"][_early], FS_COLS["F42"], CAT_REF, NUM_REF, f"A1c|{_rule}|{_oc}")
                _Cc = make_ctx(_devc, _d4["X_df"].iloc[_late], _d4["y"][_late], _d4["stay_ids"][_late], _oc)
                _pl = _per[_late]
                a1_run(_uc, _Cc, _pl == _prev, _pl == _last, f"A1c_MIMIC4early_{_rule}", _oc, A1_GROUPS)
            else:
                print("  [SKIP]", _uc)
    _fails = "; ".join(f"{r_}/{o_}: {w_}" for r_, o_, w_ in A1B_FAIL)
    if A1B_FAIL and not A1B_RAN:
        status("A1b/c MIMIC-IV", "not available", _fails)
    elif A1B_FAIL:
        status("A1b/c MIMIC-IV", "partly done", "cells not available: " + _fails)
    else:
        status("A1b/c MIMIC-IV", "done", M4_MODE + (" (" + M4_APPROX_WHY + ")" if M4_APPROX_WHY else ""))


In [ ]:
# ================================================================
# PRAM A1 report (table | policy comparison | judgment)
# ================================================================
A1DF, A1CONTRAST = pd.DataFrame(), pd.DataFrame()
if not (DATA_OK and RUN_A1):
    status("A1 report", "off" if not RUN_A1 else "not available", "main inputs absent or RUN_A1=False")
else:
    A1DF = A1_RES.df()
    if len(A1DF) == 0:
        emit("**There is no A1 result.** A1a, A1b and A1c all failed (see the reasons in the cells above).")
        status("A1 report", "not available", "no result")
    else:
        _per = A1_PER.df()
        if len(_per):
            emit_table("A1 size, prevalence and feature shift per period (past period versus test period)", _per.drop_duplicates(["analysis", "outcome"]))
            save_csv(_per.drop_duplicates(["analysis", "outcome"]), "A1_period_summary")
        _nt = A1_NOTE.df()
        if len(_nt):
            emit_table("A1 skipped | note (cells not available because of an insufficient pool, etc.)", _nt, max_rows=60)
        _tab = as_table_frame(A1DF)
        _tab["nb_site"] = [r[nbcol(T_SITE[oc])] for oc, r in zip(_tab["outcome"], _tab.to_dict("records"))]
        _shown = ["b1_base", "b5_base", "pram_b1", "pram_b5", "recal_iso", "platt_b1", "local_lr", "pooled_lr", "local_cb"]
        _mt = draw_median_table(_tab, _shown).rename(columns={"fs": "analysis", "target": "policy"})
        _mt["bank_order"] = [bank_sort_key(b) for b in _mt["bank"]]
        _mt = _mt.sort_values(["analysis", "outcome", "policy", "method", "bank_order"]).drop(columns=["bank_order"])
        save_csv(_mt, "A1_time_summary")
        _rb = [str(b) for b in pick_report_banks(BANK_SIZES)]
        _rb = ["max" if b == "full" else b for b in _rb]
        _mt_show = _mt[_mt["bank"].isin(_rb)]
        emit_table("A1 , median over draws per policy (P past bank | S same-period bank | R random bank if present)", _mt_show, max_rows=250)
        _parts = []
        for _m, _b in (("pram_b1", "b1_base"), ("pram_b5", "b5_base"), ("recal_iso", "b1_base"), ("platt_b1", "b1_base"), ("local_lr", "b1_base")):
            for _col in ("auroc", "nb_site", "ece"):
                _pc = policy_contrast(_tab, _m, _b, _col)
                if len(_pc):
                    _parts.append(_pc)
        A1CONTRAST = pd.concat(_parts, ignore_index=True) if _parts else pd.DataFrame()
        if len(A1CONTRAST):
            A1CONTRAST["bank_order"] = [bank_sort_key(b) for b in A1CONTRAST["bank"]]
            A1CONTRAST = A1CONTRAST.sort_values(["analysis", "outcome", "method", "metric", "bank_order"]).drop(columns=["bank_order"])
            save_csv(A1CONTRAST, "A1_policy_contrast")
            _core = A1CONTRAST[(A1CONTRAST["method"].isin(["pram_b1", "pram_b5"])) & (A1CONTRAST["metric"] == "auroc")]
            emit_table("A1 , policy comparison of the retrieval gain (AUROC). If P_minus_S is negative, the past bank is worse than the same-period bank (if policy R is on, P_minus_R is also given)",
                       _core.dropna(axis=1, how="all"), max_rows=120)
            _bad = _core[_core["P_minus_S"] < -0.002]
            if len(_bad):
                emit(f"**Note** , the number of cells in which the past bank has an AUROC gain lower than the same-period bank by more than 0.002 is {len(_bad)} . These cells bear on the statement of Supplementary S11 Panel B that the sample bank is not an optimistic proxy.")
            else:
                emit("- there is no cell in which the past bank has an AUROC gain lower than the same-period bank by more than 0.002.")
        status("A1 report", "done", "")


## A2. Tail risk

A hospital has **one bank**, not the mean bank. So **(one bank, one test fold of the hospital that received the bank)** is treated as one point and the tail of its distribution is counted. Each cell has 5 folds x 20 draws = 100 draws.

| Item | Definition |
|---|---|
| Setting | bank sizes 500, 1,000, 2,000, 5,000; `N_DRAWS_TAIL` (100) draws per cell; one draw is scored on the test patients of one fold |
| 42-column set | if the result file `reference/resume/a54_alpha_stability.jsonl` exists it is read from it (no model fit, AUROC gain only); otherwise it is computed |
| 33-column set | computed (eICU) |
| Gain | AUROC, ECE, slope and net benefit (outcome-specific representative threshold and 20%) of retrieval added minus frozen alone, in the same draw |
| Non-inferiority margin | AUROC 0.01, net benefit 0.001 (one net true positive per 1,000 patients). This is a **post hoc definition**, not a margin set beforehand. Tables are also produced for margins 0, 0.005, 0.01 and 0.02 |
| Tail | 5th, 10th and 50th percentiles, worst, fraction below 0, fraction of draws that lost more than the margin with a Clopper-Pearson 95% interval |
| Per-draw DeLong | the paired DeLong gives a 95% interval of the difference, and the fraction of draws whose whole interval is below 0 is counted (new computation only) |
| Smallest bank | the first size at which the upper bound of the 95% interval of the probability of losing more than the margin is at or below 5% |

Compared strategies are two retrievals (`pram_b1` on the frozen logistic and `pram_b5` on frozen CatBoost) and, as alternatives in the same bank and the same draw, Platt recalibration (`platt_b1`, `platt_b5`) and the closed-testing update (`ct_b1`). Measured by net benefit, the tail may be longer than by AUROC.


In [ ]:
# ================================================================
# PRAM A2. tail risk run (resumable; the 42-column tail is read from the reference file if it exists)
# ================================================================
A2_KEEP = ["b1_base", "b5_base", "pram_b1", "pram_b5", "platt_b1", "platt_b5", "ct_b1"]
A2_PAIRS = {"pram_b1": "b1_base", "pram_b5": "b5_base", "platt_b1": "b1_base", "platt_b5": "b5_base", "ct_b1": "b1_base"}
A2_GROUPS = ("frozen", "pram", "recal") + (("anch", "hier") if RUN_A4 else ())
A2GAP = pd.DataFrame()
if not (DATA_OK and RUN_A2):
    status("A2 tail risk", "off" if not RUN_A2 else "not available", "main inputs absent or RUN_A2=False")
else:
    _gaps = []
    _use_a54 = A54_PATH.is_file() and (not A2_RECOMPUTE_F42) and ("F42" in FS_RUN)
    if _use_a54:
        _a = pd.read_json(A54_PATH, lines=True).drop_duplicates(["outcome", "target", "bank", "fold", "draw"])
        _a = _a[_a["bank"].isin(TAIL_BANKS)]
        _g = pd.DataFrame({"fs": "F42", "outcome": _a["outcome"], "target": _a["target"], "bank": _a["bank"].astype(str),
                           "fold": _a["fold"], "draw": _a["draw"], "method": "pram_b1", "gap_auroc": _a["d_auroc"]})
        _g["source"] = "read from the reference file (AUROC gain only; no model fit)"
        _gaps.append(_g)
        emit(f"- 42-column tail: `{A54_PATH.resolve()} ` re-aggregated, rows {len(_a)} (about 100 per cell = 5 folds x 20 draws). Only the AUROC gain (retrieval on logistic) is present.")
    elif "F42" in FS_RUN:
        emit(f"- 42-column tail: no re-aggregation file (path searched `{A54_PATH.resolve()} `), so it is computed anew.")
    for _fs in [f for f in ("F42", "F33") if f in FS_RUN and (f == "F33" or not _use_a54)]:
        _R2 = Resumable(f"a2_tail_{_fs}")
        _tgs = TARGETS if _fs == "F42" else A2_TARGETS_F33
        _u2 = [(oc, tg, bk) for oc in OUTCOMES for tg in _tgs for bk in TAIL_BANKS]
        emit(f"- A2 {_fs} unit {len(_u2)} (completed {sum(_R2.unit_done((_fs,) + u) for u in _u2)} ) | draws per cell {N_DRAWS_TAIL} | bank {TAIL_BANKS}")
        for _u in _u2:
            _uu = (_fs,) + _u
            if _R2.unit_done(_uu):
                print("  [SKIP]", _uu)
                continue
            _t0 = time.time()
            for _r in run_tail_unit(CTX[(_fs, _u[0], _u[1])], _fs, _u[0], _u[1], _u[2], A2_GROUPS, A2_KEEP, A2_PAIRS):
                _R2.add_row(_r)
            _R2.mark_unit(_uu)
            print(f"  [DONE] A2 {_fs} {_u[0]} / {_u[1]} / bank={_u[2]} ({time.time() - _t0:.0f}s)", flush=True)
        _raw = _R2.df()
        if len(_raw):
            save_csv(_raw, f"A2_tail_draws_{_fs}")
            _g = tail_gaps(_raw, A2_PAIRS)
            _g["source"] = "new computation"
            _gaps.append(_g)
    A2GAP = pd.concat(_gaps, ignore_index=True) if _gaps else pd.DataFrame()
    status("A2 tail risk", "done" if len(A2GAP) else "not available", "")


In [ ]:
# ================================================================
# PRAM A2. tail risk report
# ================================================================
A2SUM, A2SAFE = pd.DataFrame(), pd.DataFrame()
if not (DATA_OK and RUN_A2) or len(A2GAP) == 0:
    status("A2 report", "not available", "no result")
else:
    A2SUM = tail_summary(A2GAP)
    A2SUM["bank_order"] = [bank_sort_key(b) for b in A2SUM["bank"]]
    A2SUM = A2SUM.sort_values(["fs", "outcome", "target", "method", "bank_order"]).drop(columns=["bank_order"])
    save_csv(A2SUM, "A2_tail_summary")
    _core = ["fs", "source", "outcome", "target", "bank", "method", "n_draws", "dAUROC_median", "dAUROC_p05", "dAUROC_worst",
             "share_below_zero", "auroc_ni_share", "auroc_ni_cp_lo", "auroc_ni_cp_hi", "draws_ci_wholly_below_zero"]
    emit_table(f"A2 , distribution of the AUROC gain of one bank (margin {NI_MARGIN_AUROC} post hoc definition, ni_share = fraction of draws that lost more than the margin, cp = 95% interval)",
               A2SUM[[c for c in _core if c in A2SUM.columns]], max_rows=250)
    _nbc = ["fs", "outcome", "target", "bank", "method", "d_nb_site_median", "d_nb_site_p05", "d_nb_site_worst", "nb_site_ni_share",
            "nb_site_ni_cp_hi", "d_nb_20_median", "d_nb_20_p05", "nb_20_ni_share", "nb_20_ni_cp_hi"]
    if "nb_site_ni_share" in A2SUM.columns:
        emit_table(f"A2 , net benefit gain distribution (new computation only, representative threshold {T_SITE} | 20%, margin {NI_MARGIN_NB} post hoc definition)",
                   A2SUM[[c for c in _nbc if c in A2SUM.columns]].dropna(subset=["nb_site_ni_share"]), max_rows=250)
    _sf = []
    for (_fs, _oc, _tg, _m), _s in A2SUM.groupby(["fs", "outcome", "target", "method"], sort=False):
        _s = _s.copy()
        _s["o"] = [bank_sort_key(b) for b in _s["bank"]]
        _s = _s.sort_values("o")
        _ok_a = _s[_s["auroc_ni_cp_hi"] <= SAFE_UPPER_BOUND]
        _row = {"fs": _fs, "outcome": _oc, "target": _tg, "method": _m,
                "safe_from_bank_AUROC": str(_ok_a["bank"].iloc[0]) if len(_ok_a) else f"{TAIL_BANKS[-1]} or below: none"}
        if "nb_site_ni_cp_hi" in _s.columns and _s["nb_site_ni_cp_hi"].notna().any():
            _ok_n = _s[_s["nb_site_ni_cp_hi"] <= SAFE_UPPER_BOUND]
            _row["safe_from_bank_NB_site"] = str(_ok_n["bank"].iloc[0]) if len(_ok_n) else f"{TAIL_BANKS[-1]} or below: none"
        _sf.append(_row)
    A2SAFE = pd.DataFrame(_sf)
    save_csv(A2SAFE, "A2_safe_bank")
    emit_table(f"A2 , safest minimum bank size (first size at which the upper bound of the 95% interval of the probability of losing more than the margin is {SAFE_UPPER_BOUND:.0%} or below)", A2SAFE, max_rows=80)
    emit("- reference values (100 draws): at 500 patients the worst -0.023, at most 39% below 0, 4.7% of all 2,400 draws lost more than 0.0029, at 5,000 patients the worst -0.0025. The unit is the same as the 42-column rows read from the reference file, so they are compared directly.")
    status("A2 report", "done", "")


## Hospital-level computation

The hospital parts of A3, A6 and A8 are derived from this computation. With the eICU hospital id, hospitals with at least 300 stays and at least 15 events and 15 non-events are selected (67 hospitals for AKI, 59 for death). Within a hospital the 3 folds split the bank (2/3 of the hospital stays) and the test (1/3), and all strategies are fitted. The metrics come from the set of test predictions collected per hospital (OOF). The strategies are the eight strategies of the main analysis plus Platt, the update hierarchy (on the frozen logistic model, including anchored ridge) and Platt after retrieval.

Per hospital, the AUROC difference versus the frozen logistic and versus the frozen CatBoost and its **paired DeLong variance (standard implementation)** are stored together. The tau squared and the prediction interval of A6 use this variance. The event counts per hospital x test fold are also stored.

The result csv files contain only pseudonymous codes (`H001` sequence) in place of hospital ids. The hospital id is used only inside this notebook.


In [ ]:
# ================================================================
# PRAM hospital-level computation (resumable)
# ================================================================
HOSP, HFOLD, HID, SITES = {}, {}, {}, {}
if not (DATA_OK and RUN_HOSPITAL):
    status("hospital level", "off" if not RUN_HOSPITAL else "not available", "main inputs absent or RUN_HOSPITAL=False")
elif not (HAVE_HOSPITALID and EICU_TAB is not None):
    emit("**hospital level not available** , the `hospitalid` column was not found in the eICU patient table. The paths searched are in the input check table of cell 1. Write the actual path in `EXTRA_RAW_PATHS` and run again from cell 1.")
    status("hospital level", "not available", "no hospital id column")
else:
    HID, SITES = prepare_sites()
    for _oc in OUTCOMES:
        emit(f"- {_oc}: stays linked to a hospital id {int(np.isfinite(HID[_oc]).sum())}/{len(HID[_oc])} | eligible hospitals {len(SITES[_oc])} hospitals"
             f"{' (check profile, so limited to ' + str(SITE_LIMIT) + ' hospitals)' if SITE_LIMIT else ''} (expected: AKI 67 | death 59)")
    for _fs in [f for f in FS_RUN if FS_SPEC[f]["hospital"]]:
        _R, _RF = Resumable(f"hospital_{_fs}"), Resumable(f"hospital_folds_{_fs}")
        _t0 = time.time()
        for _oc in OUTCOMES:
            for _i, _h in enumerate(SITES[_oc]):
                _u = (_fs, _oc, int(_h))
                if _R.unit_done(_u):
                    continue
                _rows, _frows = hospital_unit(CTX[(_fs, _oc, "eICU")], _oc, _h, HID[_oc], GROUPS_HOSP)
                for _r in (_rows or []):
                    _R.add_row(_r)
                for _r in (_frows or []):
                    _RF.add_row(_r)
                _R.mark_unit(_u)
                if _i % 10 == 0:
                    print(f"  [{_fs}] {_oc} {_i + 1}/{len(SITES[_oc])} ({(time.time() - _t0) / 60:.1f} min)", flush=True)
        HOSP[_fs], HFOLD[_fs] = _R.df(), _RF.df()
        for _nm, _df in ((f"hospital_results_{_fs}", HOSP[_fs]), (f"hospital_fold_counts_{_fs}", HFOLD[_fs])):
            if len(_df):
                _x = _df.copy()
                _x.insert(1, "site", _x["hospitalid"].map(SITE_CODE))
                save_csv(_x.drop(columns=["hospitalid"]), _nm)
    status("hospital level", "done", f"{len(HOSP)} feature sets")


In [ ]:
# ================================================================
# PRAM hospital-level summary | hospital part of A8 | event counts per test fold
# ================================================================
HSUM, HSUM_A8, HEVENTS = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()


def hosp_summary_table(df, fs):
    out = []
    for (oc, m), s in df.groupby(["outcome", "method"], sort=False):
        d = s["d_auroc_b1"].to_numpy(float)
        w = s["n_site"].to_numpy(float)
        v = s["var_b1"].to_numpy(float)
        meta = meta_reml_hk(d, v)
        meta_o = meta_reml_hk(d, v, exclude_zero_var=False)       # values of the original function without dropping zero-variance hospitals (diagnostic column)
        out.append({"fs": fs, "outcome": oc, "method": m, "k": int(len(s)),
                    "delta_stays_weighted": float(np.average(d, weights=w)), "delta_median": float(np.median(d)),
                    "delta_p10": float(np.quantile(d, 0.10)), "delta_p90": float(np.quantile(d, 0.90)),
                    "pct_better": float(100 * (d > 0).mean()),
                    "I2_stays_weighted": i2_stays_weighted(d, w),
                    "I2_invvar": meta["I2_invvar"] if meta else np.nan,
                    "k_zero_var": meta["k_zero_var"] if meta else np.nan,
                    "I2_invvar_orig": meta_o["I2_invvar"] if meta_o else np.nan,
                    "slope_median": float(s["slope"].median()), "ece_median": float(s["ece"].median()),
                    "nb_20_median": float(s[nbcol(THRESH_MS)].median()),
                    "nb_site_median": float(np.median([r[nbcol(T_SITE[oc])] for r in s.to_dict("records")]))})
    return pd.DataFrame(out)


if not HOSP:
    status("hospital summary", "not available", "no hospital-level result")
    status("A8 hospital", "not available", "no hospital-level result")
else:
    HSUM = pd.concat([hosp_summary_table(HOSP[fs], fs) for fs in HOSP], ignore_index=True)
    save_csv(HSUM, "hospital_summary")
    for _fs in HOSP:
        emit_table(f"hospital-level summary, {_fs} , the eight strategies of the main analysis (versus frozen logistic, Fig. 3 and S14 Panel A). Reference values: retrieval +0.0092 and +0.0201, hospitals that gained 61% and 76%",
                   HSUM[(HSUM["fs"] == _fs) & (HSUM["method"].isin(M_EIGHT))].drop(columns=["fs"]), max_rows=40)
    emit("- `I2_stays_weighted` is the stay-weighted method (Q with stays as weights). The weight is not the inverse variance, so it is non-standard and is pulled toward 0 as the estimates get smaller. `I2_invvar` is the standard (inverse-variance Higgins). Both are placed side by side.")
    if "F42" in HOSP and "F33" in HOSP:
        _a = HSUM[HSUM["fs"] == "F42"].set_index(["outcome", "method"])
        _b = HSUM[HSUM["fs"] == "F33"].set_index(["outcome", "method"])
        _cols = ["delta_stays_weighted", "pct_better", "I2_invvar", "slope_median", "nb_20_median"]
        HSUM_A8 = _a[_cols].join(_b[_cols], lsuffix="_F42", rsuffix="_F33", how="inner").reset_index()
        HSUM_A8 = HSUM_A8[HSUM_A8["method"].isin(M_EIGHT)]
        emit_table("A8 , hospital-level 42-column set versus 33-column set (prediction: the hospital-level gain +0.0201 and the 76% of hospitals helped by retrieval are likely to decrease)", HSUM_A8, max_rows=40)
        save_csv(HSUM_A8, "A8_hospital_F42_F33")
    _src = HFOLD.get("F42", next(iter(HFOLD.values())))
    if len(_src):
        _ev = _src.groupby("outcome").agg(hospitals=("hospitalid", "nunique"), test_folds=("fold", "size"),
                                         events_min=("events_test", "min"), events_q25=("events_test", lambda s: s.quantile(0.25)),
                                         events_median=("events_test", "median"), events_q75=("events_test", lambda s: s.quantile(0.75)),
                                         events_max=("events_test", "max"),
                                         share_folds_below_10=("events_test", lambda s: float((s < 10).mean()))).reset_index()
        HEVENTS = _ev
        emit_table("summary of event counts per hospital x test fold (full table in the `hospital_fold_counts_*` csv)", _ev)
        save_csv(_ev, "hospital_events_summary")
    status("hospital summary", "done", "")
    status("A8 hospital", "done" if "F33" in HOSP else "not available", "" if "F33" in HOSP else "no F33 hospital results")


## A1d. Hospital-level temporal validation (off)

Off by default (`RUN_A1D = False`). If switched on, for each hospital **all first-year stays are the bank and all last-year stays are the test**. Only hospitals with at least `HOSP_MIN_N_PER_YEAR` stays and at least `HOSP_MIN_EVENTS_PER_YEAR` events and non-events in each of the two years are used. If fewer than `HOSP_TIME_MIN_K` hospitals qualify, only per-hospital values are given without a meta-analysis. The test patient set differs from the main hospital-level analysis (random 3 folds), so differences between the two are read for direction only.


In [ ]:
# ================================================================
# PRAM A1d. hospital-level temporal validation (resumable)
# ================================================================
A1H = pd.DataFrame()
if not (DATA_OK and RUN_A1 and RUN_A1D and "F42" in FS_RUN):
    status("A1d hospital time", "off", "RUN_A1D=False (turned off) or main inputs absent")
elif not (HAVE_HOSPITALID and HAVE_EICU_YEAR and EICU_TAB is not None):
    emit("**A1d not available** , neither `hospitalid` nor `hospitaldischargeyear` was found in the eICU patient table (see the input check table of cell 1).")
    status("A1d hospital time", "not available", "no hospital id or discharge year column")
else:
    _Rh = Resumable("a1_hospital_time")
    _g_h = A1_GROUPS
    _elig = {}
    for _oc in OUTCOMES:
        _C = CTX[("F42", _oc, "eICU")]
        _hid = map_to_stays(EICU_TAB, "hospitalid", _C["stay_ids"])
        _yr = map_to_stays(EICU_TAB, "year", _C["stay_ids"])
        _ys = sorted(np.unique(_yr[np.isfinite(_yr)]))
        if len(_ys) < 2:
            emit(f"- A1d {_oc}: skipped because there is only one year")
            continue
        _n_ok = 0
        for _h in sorted(int(v) for v in pd.unique(_hid[np.isfinite(_hid)])):
            _u = ("a1d", _oc, _h)
            if _Rh.unit_done(_u):
                continue
            _rows = hospital_time_unit(_C, _oc, _h, _hid, _yr, _ys[0], _ys[-1], _g_h)
            for _r in (_rows or []):
                _Rh.add_row(_r)
            _Rh.mark_unit(_u)
    A1H = _Rh.df()
    if len(A1H) == 0:
        emit("**no A1d result** , no hospital exceeds the criterion in both years.")
        status("A1d hospital time", "not available", "no eligible hospital")
    else:
        _kh = A1H.groupby("outcome")["hospitalid"].nunique()
        emit(f"- A1d eligible hospitals (criterion met in both years): {_kh.to_dict()} (hospital-level analysis of the main text: AKI 67 | death 59)")
        _out = []
        for (_oc, _m), _s in A1H.groupby(["outcome", "method"], sort=False):
            if _m == "b1_base":
                continue
            _d, _v, _w = _s["d_auroc_b1"].to_numpy(float), _s["var_b1"].to_numpy(float), _s["n_test"].to_numpy(float)
            _row = {"outcome": _oc, "method": _m, "k": int(len(_s)), "delta_median": float(np.median(_d)),
                    "delta_stays_weighted": float(np.average(_d, weights=_w)), "pct_better": float(100 * (_d > 0).mean())}
            if len(_s) >= HOSP_TIME_MIN_K:
                _mt = meta_reml_hk(_d, _v)
                if _mt:
                    _row.update({"meta_mu": _mt["mu_random"], "ci_hk_lo": _mt["ci_hk_lo"], "ci_hk_hi": _mt["ci_hk_hi"],
                                 "tau": _mt["tau"], "pi_lo": _mt["pi_lo"], "pi_hi": _mt["pi_hi"], "I2_invvar": _mt["I2_invvar"]})
            if "F42" in HOSP:
                _r = HOSP["F42"][(HOSP["F42"]["outcome"] == _oc) & (HOSP["F42"]["method"] == _m)][["hospitalid", "d_auroc_b1"]]
                _j = _s[["hospitalid", "d_auroc_b1"]].merge(_r, on="hospitalid", suffixes=("_time", "_random"))
                if len(_j):
                    _row["same_hospitals_in_random_design"] = int(len(_j))
                    _row["delta_median_random_design"] = float(_j["d_auroc_b1_random"].median())
                    _row["time_minus_random_median"] = float((_j["d_auroc_b1_time"] - _j["d_auroc_b1_random"]).median())
            _out.append(_row)
        A1HSUM = pd.DataFrame(_out)
        emit_table("A1d , hospital-level temporal validation (AUROC gain over the frozen logistic)", A1HSUM, max_rows=60)
        _x = A1H.copy()
        _x.insert(1, "site", _x["hospitalid"].map(SITE_CODE))
        save_csv(_x.drop(columns=["hospitalid"]), "A1d_hospital_time_results")
        save_csv(A1HSUM, "A1d_hospital_time_summary")
        status("A1d hospital time", "done", f"hospital {_kh.to_dict()}")


## A3. Decision-curve thresholds

The decision-curve threshold of the main analysis, 20%, is high for an AKI prevalence of 6 to 16% and a death prevalence of about 4.5%. At 20%, the AUROC winner and the net benefit winner differ in 78% of hospitals for AKI and 68% for death; this is a value at a single threshold.

This section uses the **net benefit over the full 1 to 50% threshold grid** stored by the computations above, with no new fit, to give the following.

1. Hospital level: per threshold, the fraction of hospitals whose AUROC winner and net benefit winner differ (including the value at 20%).
2. Within the outcome-specific range (`THRESH_RANGE`, set in `code/pram_config.py`) and within the range 2 to 30%: mean disagreement, number of times the net benefit winner changes, and the fraction in which the winner by area under the curve equals the AUROC winner.
3. Grid: median over draws of the net benefit curve per bank size, and the net benefit difference retrieval minus isotonic by threshold.

At low thresholds the net benefit of the frozen model is less negative, so the disagreement may rise or fall with the threshold.


In [ ]:
# ================================================================
# PRAM A3. decision-curve thresholds (hospital level + grid)
# ================================================================
A3DIS, A3RANGE, A3AREA, A3CURVE = pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
_trap = getattr(np, "trapezoid", None) or np.trapz
if not RUN_A3:
    status("A3 decision curve", "off" if not RUN_A3 else "not available", "RUN_A3=False")
elif not (HOSP or MX):
    status("A3 decision curve", "not available", "neither hospital-level nor grid results exist")
else:
    _dis_rows, _rng_rows, _area_rows = [], [], []
    for _fs in HOSP:
        _d8 = HOSP[_fs][HOSP[_fs]["method"].isin(M_EIGHT)]
        for _oc in OUTCOMES:
            _s = _d8[_d8["outcome"] == _oc]
            if len(_s) == 0:
                continue
            _auc = _s.pivot(index="hospitalid", columns="method", values="auroc")
            _win_auc = _auc.idxmax(axis=1)
            _win_nb = pd.DataFrame({float(t): _s.pivot(index="hospitalid", columns="method", values=nbcol(t)).idxmax(axis=1)
                                    for t in THRESH_GRID})
            _dis = _win_nb.ne(_win_auc, axis=0).mean()
            for _t, _v in _dis.items():
                _dis_rows.append({"fs": _fs, "outcome": _oc, "threshold": float(_t), "share_disagree": float(_v), "n_hospitals": int(len(_win_auc))})
            _nb = _s.set_index(["hospitalid", "method"])[NB_COLS]
            for _rn, (_lo, _hi) in (("single range 2 to 30 percent", RANGE_ORIGINAL), ("outcome-specific range", THRESH_RANGE[_oc])):
                _ts = [float(t) for t in THRESH_GRID if _lo - 1e-9 <= t <= _hi + 1e-9]
                _cols = [nbcol(t) for t in _ts]
                _ar = pd.Series(_trap(_nb[_cols].to_numpy(), x=_ts, axis=1), index=_nb.index).unstack("method")
                _win_area = _ar.idxmax(axis=1)
                _chg = _win_nb[_ts].apply(lambda r: int((r.values[1:] != r.values[:-1]).sum()), axis=1)
                _rng_rows.append({"fs": _fs, "outcome": _oc, "range": _rn, "lo": _lo, "hi": _hi, "n_hospitals": int(len(_win_auc)),
                                  "disagree_at_20": float(_dis.get(0.2, np.nan)), "disagree_mean_in_range": float(_dis[_ts].mean()),
                                  "winner_changes_median": float(_chg.median()), "winner_changes_max": int(_chg.max()),
                                  "area_winner_equals_auroc_winner": float((_win_area == _win_auc).mean()),
                                  "pct_hospitals_pram_b1_area_gt_frozen": float(100 * (_ar["pram_b1"] > _ar["b1_base"]).mean()) if {"pram_b1", "b1_base"} <= set(_ar.columns) else np.nan})
                for _m, _n in _win_area.value_counts().items():
                    _area_rows.append({"fs": _fs, "outcome": _oc, "range": _rn, "area_winner": _m, "hospitals": int(_n)})
    A3DIS, A3RANGE, A3AREA = pd.DataFrame(_dis_rows), pd.DataFrame(_rng_rows), pd.DataFrame(_area_rows)
    if len(A3DIS):
        save_csv(A3DIS, "A3_disagreement_by_threshold")
        save_csv(A3RANGE, "A3_range_summary")
        save_csv(A3AREA, "A3_area_winners")
        _pick = A3DIS[A3DIS["threshold"].isin([0.02, 0.05, 0.10, 0.20, 0.30, 0.40])].pivot_table(index=["fs", "outcome"], columns="threshold", values="share_disagree")
        emit_table("A3 , fraction of hospitals where the AUROC winner and the net benefit winner differ (value at 20%: AKI 78% | death 68%)", _pick.reset_index(), digits=3)
        emit_table(f"A3 , summary by range (outcome-specific range {THRESH_RANGE}, range 2 to 30% {RANGE_ORIGINAL})", A3RANGE, digits=3, max_rows=40)
        emit_table("A3 , winner by area under the curve ", A3AREA, max_rows=80)
        status("A3 hospital decision curve", "done", "")
    else:
        status("A3 hospital decision curve", "not available", "no hospital-level result")

if RUN_A3 and MX:
    _cm = ["b1_base", "b5_base", "pram_b1", "pram_b5", "recal_iso", "platt_b1", "platt_b5", "local_lr", "pooled_lr", "local_cb",
           "upd1_b1", "upd2_b1", "anchored_ridge", "ct_b1", "pram_platt_b1", "pram_platt_b5"]
    _curves = []
    for _fs in [f for f in ("F42", "F33") if f in MX]:
        _d = MX[_fs][MX[_fs]["method"].isin(_cm)]
        _med = _d.groupby(["fs", "outcome", "target", "bank", "method"], sort=False)[NB_COLS + ["obs_rate"]].median().reset_index()
        _long = _med.melt(id_vars=["fs", "outcome", "target", "bank", "method", "obs_rate"], value_vars=NB_COLS,
                          var_name="threshold", value_name="nb_median")
        _long["threshold"] = _long["threshold"].str.replace("nb@", "", regex=False).astype(float)
        _long["nb_treat_all"] = _long["obs_rate"] - (1 - _long["obs_rate"]) * _long["threshold"] / (1 - _long["threshold"])
        _curves.append(_long.drop(columns=["obs_rate"]))
    A3CURVE = pd.concat(_curves, ignore_index=True)
    save_csv(A3CURVE, "A3_dca_curves_matrix")
    _gap = A3CURVE[A3CURVE["method"].isin(["pram_b1", "recal_iso"])].pivot_table(
        index=["fs", "outcome", "target", "bank", "threshold"], columns="method", values="nb_median").reset_index()
    _gap["pram_b1_minus_recal_iso"] = _gap["pram_b1"] - _gap["recal_iso"]
    _gap = _gap[_gap["threshold"].isin([0.02, 0.05, 0.10, 0.20, 0.30]) & _gap["bank"].isin([str(b) for b in pick_report_banks(BANK_SIZES)])]
    emit_table("A3 , net benefit difference by threshold (retrieval minus isotonic recalibration, median over draws). If the sign changes with the threshold, the order of recalibration versus retrieval depends on the threshold",
               _gap.pivot_table(index=["fs", "outcome", "target", "bank"], columns="threshold", values="pram_b1_minus_recal_iso").reset_index(), digits=4, max_rows=80)
    _rng = []
    for _oc in OUTCOMES:
        _lo, _hi = THRESH_RANGE[_oc]
        _w = A3CURVE[(A3CURVE["outcome"] == _oc) & A3CURVE["threshold"].between(_lo - 1e-9, _hi + 1e-9)
                     & A3CURVE["bank"].isin([str(b) for b in pick_report_banks(BANK_SIZES)]) & A3CURVE["method"].isin(["b1_base", "pram_b1", "recal_iso", "platt_b1", "local_lr", "upd1_b1", "upd2_b1", "anchored_ridge", "ct_b1", "pram_platt_b1"])]
        _g = _w.groupby(["fs", "outcome", "target", "bank", "method"], sort=False)["nb_median"].mean().reset_index()
        _g["range"] = f"{_lo:.2f}~{_hi:.2f}"
        _rng.append(_g.rename(columns={"nb_median": "mean_nb_in_range"}))
    emit_table("A3 , mean net benefit within the outcome-specific range (area under the curve / range width)", pd.concat(_rng, ignore_index=True), digits=4, max_rows=150)
    status("A3 grid curve", "done", "")


## Re-aggregation of the reference hospital files and comparison with the hospital computation above

If the reference result files `a29_real_sites` (per-hospital AUROC), `a51_hospital_thresholds` (per-hospital net benefit by threshold) and `a59_hospital_diag` (per-hospital gains and variances) are in `reference/resume/`, they are read. If they are absent, the searched paths are printed and the section is skipped. If they are read, three things are done.

1. **Disagreement by threshold**: from the per-hospital AUROC (`a29`) and the per-hospital net benefit by threshold (`a51`, 2 to 30%, 2% steps), the fraction of hospitals whose AUROC winner and net benefit winner differ is counted, including the value at 20%.
2. **Tau and prediction interval**: meta-analysis with the per-hospital gains and variances of `a59` (back-calculated from the DeLong z), compared with tau = 0 back-calculated from the summary values.
3. **Hospital computation of this notebook versus the reference `a29`**: whether the per-hospital AUROC gain computed here with the same hospital, fold and key equals the reference value (42 columns). A mismatch is printed as it is.


In [ ]:
# ================================================================
# PRAM re-aggregation of the reference hospital files and comparison with the hospital computation of this notebook
# ================================================================
REAGG = {}
if not (DATA_OK and RUN_REAGG):
    status("re-aggregation of existing results", "off" if not RUN_REAGG else "not available", "RUN_REAGG=False or main inputs absent")
else:
    _have = {n: p.is_file() for n, p in (("a29", A29_PATH), ("a51", A51_PATH), ("a59", A59_PATH))}
    if not any(_have.values()):
        emit("**re-aggregation of existing results skipped** , the reference files a29, a51 and a59 are not in `reference/resume/`. Paths searched: "
             + ", ".join("`%s`" % p.resolve() for p in (A29_PATH, A51_PATH, A59_PATH)) + ". If the reference files are placed in `reference/resume/`, they are read automatically.")
        status("re-aggregation of existing results", "skipped", "no reference result file")
    else:
        _a29 = pd.read_json(A29_PATH, lines=True).drop_duplicates(["outcome", "hospitalid", "method"]) if _have["a29"] else pd.DataFrame()
        if _have["a29"] and _have["a51"]:
            _a51 = pd.read_json(A51_PATH, lines=True).drop_duplicates(["outcome", "hospitalid", "method", "threshold"])
            _rows, _rng = [], []
            for _oc in OUTCOMES:
                _s = _a29[(_a29["outcome"] == _oc) & _a29["method"].isin(M_EIGHT)]
                _n = _a51[(_a51["outcome"] == _oc) & _a51["method"].isin(M_EIGHT)]
                _hs = sorted(set(_s["hospitalid"]) & set(_n["hospitalid"]))
                if not _hs:
                    continue
                _win_auc = _s[_s["hospitalid"].isin(_hs)].pivot(index="hospitalid", columns="method", values="auroc").idxmax(axis=1)
                _ths = sorted(_n["threshold"].unique())
                _wn = pd.DataFrame({float(t): _n[(_n["threshold"] == t) & _n["hospitalid"].isin(_hs)].pivot(index="hospitalid", columns="method", values="nb").idxmax(axis=1)
                                    for t in _ths})
                _dis = _wn.ne(_win_auc, axis=0).mean()
                for _t, _v in _dis.items():
                    _rows.append({"outcome": _oc, "threshold": float(_t), "share_disagree": float(_v), "n_hospitals": len(_hs)})
                for _rn, (_lo, _hi) in (("single range 2 to 30 percent", RANGE_ORIGINAL), ("outcome-specific range", THRESH_RANGE[_oc])):
                    _ts = [float(t) for t in _ths if _lo - 1e-9 <= t <= _hi + 1e-9]
                    if _ts:
                        _chg = _wn[_ts].apply(lambda r: int((r.values[1:] != r.values[:-1]).sum()), axis=1)
                        _rng.append({"outcome": _oc, "range": _rn, "n_thresholds_in_grid": len(_ts), "disagree_mean": float(_dis[_ts].mean()),
                                     "winner_changes_median": float(_chg.median()), "n_hospitals": len(_hs)})
            REAGG["thr"], REAGG["rng"] = pd.DataFrame(_rows), pd.DataFrame(_rng)
            if len(REAGG["thr"]):
                _pk = REAGG["thr"][REAGG["thr"]["threshold"].round(2).isin([0.02, 0.06, 0.10, 0.20, 0.30])]
                emit_table("re-aggregation of thresholds , fraction of hospitals where the AUROC winner and the net benefit winner differ (a29 x a51, 2% steps. Value at 20%: AKI 78% | death 68%)",
                           _pk.pivot_table(index="outcome", columns="threshold", values="share_disagree").reset_index(), digits=3)
                emit_table("re-aggregation of thresholds , summary by range (the grid has 2% steps, so the outcome-specific range has few points)", REAGG["rng"], digits=3)
                save_csv(REAGG["thr"], "REAGG_disagreement_a29_a51")
        if _have["a59"]:
            _a59 = pd.read_json(A59_PATH, lines=True).drop_duplicates(["outcome", "hospitalid"]).dropna(subset=["var"])
            _out = []
            for _oc, _s in _a59.groupby("outcome"):
                _mt = meta_reml_hk(_s["delta"], _s["var"], exclude_zero_var=False)       # for the reference file the original function is used as is (no exclusion of zero-variance hospitals)
                if _mt:
                    _out.append({"outcome": _oc, "k": _mt["k"], "df": _mt["df"], "Q": _mt["Q"], "I2_invvar": _mt["I2_invvar"], "tau": _mt["tau"],
                                 "mu_random": _mt["mu_random"], "ci_hk_lo": _mt["ci_hk_lo"], "ci_hk_hi": _mt["ci_hk_hi"],
                                 "ci_hk_trunc_lo": _mt["ci_hk_trunc_lo"], "ci_hk_trunc_hi": _mt["ci_hk_trunc_hi"], "pi_lo": _mt["pi_lo"], "pi_hi": _mt["pi_hi"],
                                 "I2_stays_weighted": i2_stays_weighted(_s["delta"].to_numpy(float), _s["n"].to_numpy(float))})
            REAGG["meta"] = pd.DataFrame(_out)
            if len(REAGG["meta"]):
                emit_table("re-aggregation of tau , meta-analysis with the per-hospital gains (retrieval on logistic) and variances of the reference file a59 (back-calculation of the reference summary: tau = 0, prediction interval AKI -0.0071 to +0.0129 | death -0.0072 to +0.0176)", REAGG["meta"], digits=4)
                save_csv(REAGG["meta"], "REAGG_meta_a59")
        if _have["a29"] and "F42" in HOSP:
            _new = HOSP["F42"][HOSP["F42"]["method"].isin(M_EIGHT)][["outcome", "hospitalid", "method", "auroc", "d_auroc_b1"]]
            _old = _a29[_a29["method"].isin(M_EIGHT)][["outcome", "hospitalid", "method", "auroc", "delta_auroc"]]
            _j = _new.merge(_old, on=["outcome", "hospitalid", "method"], suffixes=("_new", "_old"))
            if len(_j):
                _j["abs_diff_auroc"] = (_j["auroc_new"] - _j["auroc_old"]).abs()
                _g = _j.groupby(["outcome", "method"]).agg(hospitals=("hospitalid", "nunique"), max_abs_diff=("abs_diff_auroc", "max"),
                                                           median_abs_diff=("abs_diff_auroc", "median"),
                                                           share_within_5e5=("abs_diff_auroc", lambda s: float((s <= 0.00005).mean()))).reset_index()
                emit_table("comparison of the hospital computation of this notebook with the reference a29 (hospital AUROC, same hospital | same fold | same key)", _g, digits=5, max_rows=40)
                save_csv(_g, "REAGG_repro_hospital_vs_a29")
                REAGG["repro"] = _g
        status("re-aggregation of existing results", "done", ",".join(k for k, v in _have.items() if v))


## A6. Hospital-level heterogeneity: tau, prediction interval, event counts

A random-effects meta-analysis of the per-hospital AUROC gain (versus the frozen logistic) with its paired DeLong variance (**standard fastDeLong**). It gives: REML tau squared and tau, **Q and degrees of freedom**, the Hartung-Knapp 95% interval (**untruncated and truncated forms**: the truncated form lower-bounds the variance correction q at 1), the 95% prediction interval (t distribution, k-2 degrees of freedom), the inverse-variance Higgins I squared, and the stay-weighted I squared.

1. **The stay-weighted I squared** uses a Q with stays as weights. This weight is not the inverse variance, so it is non-standard; if the between-hospital spread of the gain is small, any such method is pulled toward 0. The inverse-variance I squared is reported next to it.
2. **The original DeLong core overestimates the variance.** `_delong_core_original` uses `np.arange` in place of the within-class midranks of the positives and negatives. When the scores within a class are not in sorted order, the variance of the paired AUROC difference is overestimated relative to the standard implementation, the O(mn) definition and the bootstrap. The diagnostic cell below gives the actual ratio on the hospital data.

All DeLong variances in this section use the standard implementation. The values of the original core are computed only in the diagnostic cell, for comparison.

**Handling of zero-variance hospitals**: a hospital whose mixing weight is 0 in all three folds has a retrieval prediction equal to the frozen prediction (gain 0, variance 0). A variance floor of 1e-12 would give such a hospital 99.9998% of the inverse-variance weight and collapse the pooled value, tau and prediction interval. Hospitals with a variance of 1e-12 or less are therefore excluded, and the columns `k_zero_var` (number of hospitals dropped) and `top1_weight_share` (largest weight share of one hospital) are added. The values without dropping hospitals are in the `*_orig` columns. Weight can still pile up on a hospital with a variance below 1e-10 (check `top1_weight_share`). This pooling is a secondary report; the main report is the stay-weighted mean and the hospital-level bootstrap interval.


In [ ]:
# ================================================================
# PRAM A6. meta-analysis
# ================================================================
A6TAB = pd.DataFrame()
if not (RUN_A6 and HOSP):
    status("A6 meta-analysis", "off" if not RUN_A6 else "not available", "RUN_A6=False or no hospital-level result")
else:
    _rows, _skip = [], []
    for _fs in HOSP:
        for (_oc, _m), _s in HOSP[_fs].groupby(["outcome", "method"], sort=False):
            if _m == "b1_base":
                continue
            _meta = meta_reml_hk(_s["d_auroc_b1"], _s["var_b1"])
            if _meta is None:
                _skip.append(f"{_fs}/{_oc}/{_m}")
                continue
            _orig = meta_reml_hk(_s["d_auroc_b1"], _s["var_b1"], exclude_zero_var=False)       # value of the original function (zero-variance hospitals entered with weight 1e12)
            _rows.append({"fs": _fs, "outcome": _oc, "method": _m, **_meta,
                          "I2_stays_weighted": i2_stays_weighted(_s["d_auroc_b1"].to_numpy(float), _s["n_site"].to_numpy(float)),
                          **{f"{_k}_orig": _orig[_k] for _k in ("k", "mu_random", "tau", "I2_invvar", "pi_lo", "pi_hi", "top1_weight_share")}})
    A6TAB = pd.DataFrame(_rows)
    if _skip:
        emit(f"- A6 cells with no row {len(_skip)} (fewer than 3 hospitals with nonzero variance): {_skip}")
    if len(A6TAB):
        save_csv(A6TAB, "A6_meta_reml_hk")
        _c = ["fs", "outcome", "method", "k", "df", "Q", "Q_p", "mu_random", "ci_hk_lo", "ci_hk_hi", "ci_hk_trunc_lo", "ci_hk_trunc_hi",
              "tau", "pi_lo", "pi_hi", "I2_invvar", "I2_reml", "I2_stays_weighted",
              "k_zero_var", "top1_weight_share", "k_orig", "mu_random_orig", "tau_orig", "I2_invvar_orig", "top1_weight_share_orig"]
        _mm = ["pram_b1", "pram_b5", "platt_b1", "recal_iso", "local_lr", "local_cb", "pooled_lr", "ct_b1", "pram_platt_b1", "platt_pram_b1"]
        emit("- hospitals with zero variance (k_zero_var hospitals) were dropped from this table. For such a hospital the retrieval prediction equals the frozen prediction, so the gain is also 0, and a variance floor of 1e-12 would give weight 1e12 and take over the pooled value. "
             "`top1_weight_share` is the largest share of weight held by one hospital. The `*_orig` columns are values of the original function without dropping hospitals . Inverse-variance pooling is a secondary report; the main report is the stay-weighted mean and the hospital-level bootstrap interval.")
        emit_table("A6 , hospital-level random-effects meta-analysis (AUROC gain over frozen logistic. hk = Hartung-Knapp original form, hk_trunc = truncated form with q lower bound 1, Q, df and Q_p = heterogeneity test)",
                   A6TAB[A6TAB["method"].isin(_mm)][_c], max_rows=120)
        for _r in A6TAB[(A6TAB["method"].isin(["pram_b1", "pram_b5"])) & (A6TAB["fs"] == "F42")].itertuples():
            _inc = "includes" if _r.pi_lo <= 0 <= _r.pi_hi else "does not include"
            emit(f"- {_r.outcome} {_r.method}: gain {_r.mu_random:+.4f}, prediction interval [{_r.pi_lo:+.4f}, {_r.pi_hi:+.4f}] with 0 in it: {_inc}. τ = {_r.tau:.4f}, inverse-variance I squared = {_r.I2_invvar:.0f}% "
                 f"(zero-variance hospitals {int(_r.k_zero_var)} excluded, largest weight of one hospital {_r.top1_weight_share:.0%} , gain of the original function without exclusion {_r.mu_random_orig:+.4f} τ {_r.tau_orig:.4f})")
        _ev = HEVENTS
        if len(_ev):
            emit_table("event counts per hospital x test fold (the main text gives minimum, median and quartiles)", _ev)
    status("A6 meta-analysis", "done" if len(A6TAB) else "not available", "")


### A6 diagnosis: was the per-hospital variance overestimated compared with the actual spread?

An I squared of 0% means that Q is far smaller than the degrees of freedom. There are two explanations. (1) The retrieval gain really is almost the same across hospitals. (2) The variance of the per-hospital gain was set larger than the actual within-hospital variation, so that the I squared of 0% is a product of the method.

The **three variances** are compared directly for the same hospitals and the same test predictions: (a) the original DeLong core, (b) the standard fastDeLong, (c) the within-hospital bootstrap (test patients resampled with replacement `N_BOOT_WITHIN` times, recomputing the AUROC difference between retrieval and frozen). A ratio below 1 means underestimation and above 1 overestimation. The meta-analysis is repeated with each of the three variances to show how Q, I squared, tau and the prediction interval change.

The bootstrap resamples the set of test predictions of a hospital (the 3 folds pooled) at the patient level. Its random stream is `np.random.default_rng([BOOT_SEED, outcome index, hospital id])`, so it does not depend on the hospital order. The bootstrap does not reflect the within-hospital fold structure or the variation of bank draws; including that variation would enlarge the variance. Notebook 03 (cells 80 to 83) computes the same quantities with the same stream.


In [ ]:
# ================================================================
# PRAM A6 diagnosis. original DeLong variance | standard DeLong variance | within-hospital bootstrap variance
# ================================================================
A6DIAG, A6DIAG_H = pd.DataFrame(), pd.DataFrame()
# This section has no resume unit, so it runs anew every time. If the two csv files of this diagnosis already exist in the csv folder, it is not recomputed.
_dg_h, _dg_m = CSV_DIR / "A6_variance_diagnosis_by_hospital.csv", CSV_DIR / "A6_variance_diagnosis_meta.csv"
_dg_ready = False
if _dg_h.is_file() and _dg_m.is_file():
    try:
        _dg_ready = ({"var_boot", "n_boot_used", "var_delong_standard"} <= set(pd.read_csv(_dg_h, nrows=2).columns)
                     and {"k_zero_var", "top1_weight_share", "variance_source"} <= set(pd.read_csv(_dg_m, nrows=2).columns))
    except (OSError, ValueError, pd.errors.ParserError) as _exc:
        emit(f"- the existing A6 diagnosis csv could not be read ({type(_exc).__name__}). Recomputing.")
if not (RUN_A6 and HOSP and "F42" in HOSP and HID):
    status("A6 variance diagnosis", "off" if not RUN_A6 else "not available", "RUN_A6=False or no hospital-level result")
elif _dg_ready:
    emit(f"- A6 variance diagnosis: csv files of this diagnosis already exist, so it is not recomputed (`{_dg_h.name}` · `{_dg_m.name}`, within-hospital bootstrap {N_BOOT_WITHIN} repetitions setting unchanged).")
    status("A6 variance diagnosis", "done", "existing csv kept as is (not recomputed)")
else:
    _hrows = []
    for _oc in OUTCOMES:
        _C = CTX[("F42", _oc, "eICU")]
        for _h in SITES[_oc]:
            _ys, _oof, _ = hospital_oof(_C, _oc, _h, HID[_oc], ("frozen", "pram"))
            if _ys is None or "pram_b1" not in _oof:
                continue
            _pa, _pb = _oof["pram_b1"], _oof["b1_base"]
            _a1, _a2, _d, _v_std = delong_delta_var(_ys, _pa, _pb)
            _v_org = delong_delta_var(_ys, _pa, _pb, original=True)[3]
            _rs = np.random.default_rng([int(BOOT_SEED), OUTCOMES.index(_oc), int(_h)])
            _bs = []
            for _ in range(N_BOOT_WITHIN):
                _i = _rs.integers(0, len(_ys), len(_ys))
                if len(np.unique(_ys[_i])) > 1:
                    _bs.append(roc_auc_score(_ys[_i], _pa[_i]) - roc_auc_score(_ys[_i], _pb[_i]))
            _hrows.append({"outcome": _oc, "hospitalid": int(_h), "n_site": int(len(_ys)), "d_auroc": _d, "var_delong_original": _v_org,
                           "var_delong_standard": _v_std, "var_boot": float(np.var(_bs, ddof=1)) if len(_bs) > 10 else np.nan, "n_boot_used": len(_bs),
                           "n_boot": int(N_BOOT_WITHIN), "boot_seed": int(BOOT_SEED)})
    A6DIAG_H = pd.DataFrame(_hrows)
    if len(A6DIAG_H):
        A6DIAG_H["ratio_original_over_boot"] = A6DIAG_H["var_delong_original"] / A6DIAG_H["var_boot"]
        A6DIAG_H["ratio_standard_over_boot"] = A6DIAG_H["var_delong_standard"] / A6DIAG_H["var_boot"]
        _x = A6DIAG_H.copy()
        _x.insert(1, "site", _x["hospitalid"].map(SITE_CODE))
        save_csv(_x.drop(columns=["hospitalid"])[["outcome", "site", "n_site", "d_auroc", "var_delong_original", "var_delong_standard", "var_boot", "n_boot_used",
                                                  "ratio_original_over_boot", "ratio_standard_over_boot", "n_boot", "boot_seed"]], "A6_variance_diagnosis_by_hospital")
        _out = []
        for _oc, _s in A6DIAG_H.dropna(subset=["var_boot"]).groupby("outcome"):
            for _src, _col in (("DeLong original implementation", "var_delong_original"), ("DeLong standard", "var_delong_standard"), ("bootstrap", "var_boot")):
                _mt = meta_reml_hk(_s["d_auroc"], _s[_col])
                if _mt:
                    _out.append({"outcome": _oc, "variance_source": _src, "k": _mt["k"], "k_zero_var": _mt["k_zero_var"],
                                 "top1_weight_share": _mt["top1_weight_share"], "df": _mt["df"], "Q": _mt["Q"], "Q_p": _mt["Q_p"],
                                 "I2_invvar": _mt["I2_invvar"], "tau": _mt["tau"], "mu_random": _mt["mu_random"],
                                 "ci_hk_lo": _mt["ci_hk_lo"], "ci_hk_hi": _mt["ci_hk_hi"], "pi_lo": _mt["pi_lo"], "pi_hi": _mt["pi_hi"],
                                 "ratio_orig_over_boot_median": float(_s["ratio_original_over_boot"].median()),
                                 "ratio_std_over_boot_median": float(_s["ratio_standard_over_boot"].median())})
        A6DIAG = pd.DataFrame(_out)
        emit_table("A6 diagnosis , meta-analysis using each of the three variances on the same per-hospital gains (ratio = hospital median of each DeLong variance / bootstrap variance)", A6DIAG, digits=4)
        save_csv(A6DIAG, "A6_variance_diagnosis_meta")
        for _oc, _s in A6DIAG[A6DIAG["variance_source"] == "bootstrap"].groupby("outcome"):
            _ro, _rs_ = float(_s["ratio_orig_over_boot_median"].iloc[0]), float(_s["ratio_std_over_boot_median"].iloc[0])
            if _ro > 1.5 and 0.8 <= _rs_ <= 1.25:
                _verdict = "The original DeLong implementation overestimates the variance (the standard implementation agrees with the bootstrap). The Q and the per-draw DeLong significance fractions based on the original implementation were conservative"
            elif _rs_ > 1.25:
                _verdict = "The standard DeLong is also larger than the bootstrap (this may mean the within-hospital variation is smaller than the independent-sample assumption of DeLong)"
            elif _ro <= 1.5:
                _verdict = "The variances of the original and standard implementations are similar (the effect of the defect is small in these hospitals)"
            else:
                _verdict = "The standard DeLong is smaller than the bootstrap (a bootstrap that does not reflect the within-hospital fold structure may be too small)"
            emit(f"- {_oc}: median variance ratio original/bootstrap {_ro:.2f} , standard/bootstrap {_rs_:.2f} , {_verdict} (the decision thresholds 1.5, 0.8 and 1.25 are post hoc)")
    status("A6 variance diagnosis", "done" if len(A6DIAG_H) else "not available", "")


## A7. Unlabeled hospital network and partner bank prevalence, and the network part of A8

Switched on by default (`RUN_A7 = True`) for both the 42-column and the 33-column sets. To switch off, set `RUN_A7 = False`.

**Setting**: a hospital without any local labels queries partner hospitals (at most 20) and mixes the mean of the returned probabilities into the frozen output. The mixing weight is fixed because there are no labels (0.30 in the main analysis; the full grid of 0.05 to 0.50, and up to 1.0, is a sensitivity analysis).

**Question**: how does calibration at the querying hospital change if the prevalence of the partner banks differs from that of the querying hospital?

**Design**: for each partner the bank is redrawn at **prevalence 0.5x, 1x and 2x** (to raise: all events and some non-events; to lower: all non-events and some events). 1x is the whole partner cohort. The same queries and the same partner selection are used for every factor. For each querying hospital, the relation between (mean partner bank prevalence minus own prevalence) and the calibration intercept, slope and mean predicted risk after mixing is read as a rank correlation across hospitals.

**Limitation**: the redrawn banks also shrink in size (non-events when raising, events when lowering). With the k = 50 neighbor search the effect of size is expected to be small, but it was not separated.


In [ ]:
# ================================================================
# PRAM A7. unlabeled network and partner prevalence (resumable)
# ================================================================
NET, A7COR = {}, pd.DataFrame()
if not (DATA_OK and RUN_A7):
    status("A7 network", "off" if not RUN_A7 else "not available", "main inputs absent or RUN_A7=False")
elif not (HAVE_HOSPITALID and EICU_TAB is not None):
    emit("**A7 not available** , the `hospitalid` column was not found in the eICU patient table (see the input check table of cell 1).")
    status("A7 network", "not available", "no hospital id column")
else:
    if not SITES:
        HID, SITES = prepare_sites()
    for _fs in [f for f in FS_RUN if FS_SPEC[f]["network"]]:
        _R = Resumable(f"network_{_fs}")
        _t0 = time.time()
        for _oc in OUTCOMES:
            for _i, _h in enumerate(SITES[_oc]):
                _u = (_fs, _oc, int(_h))
                if _R.unit_done(_u):
                    continue
                _rows = network_unit(CTX[(_fs, _oc, "eICU")], _oc, _h, HID[_oc], SITES[_oc], PARTNER_PREV_FACTORS, NET_ALPHA_GRID)
                for _r in (_rows or []):
                    _R.add_row(_r)
                _R.mark_unit(_u)
                if _i % 10 == 0:
                    print(f"  [network {_fs}] {_oc} {_i + 1}/{len(SITES[_oc])} ({(time.time() - _t0) / 60:.1f} min)", flush=True)
        NET[_fs] = _R.df()
        if len(NET[_fs]):
            _x = NET[_fs].copy()
            _x.insert(1, "site", _x["hospitalid"].map(SITE_CODE))
            _x["calib_gap_mean_pred_minus_obs"] = _x["mean_pred"] - _x["obs_rate"]
            save_csv(_x.drop(columns=["hospitalid"]), f"A7_network_{_fs}")
    _alphas = [0.05, 0.10, 0.30, 0.50, 0.70, 1.0]
    for _fs, _n in NET.items():
        if len(_n) == 0:
            continue
        _b = _n[(_n["factor"] == 1.0) & _n["alpha"].isin(_alphas)]
        _g = _b.groupby(["outcome", "alpha"]).agg(n_hospitals=("hospitalid", "nunique"), d_auroc_median=("d_auroc", "median"),
                                                 pct_improved=("d_auroc", lambda s: float(100 * (s > 0).mean())),
                                                 slope_median=("slope", "median"), base_slope_median=("base_slope", "median"),
                                                 ece_median=("ece", "median"), base_ece_median=("base_ece", "median"), d_ece_median=("d_ece", "median"),
                                                 pct_ece_improved=("d_ece", lambda s: float(100 * (s < 0).mean())),
                                                 nb_rep_median=("nb_rep", "median"), base_nb_rep_median=("base_nb_rep", "median"), d_nb_rep_median=("d_nb_rep", "median"),
                                                 pct_nb_improved=("d_nb_rep", lambda s: float(100 * (s > 0).mean()))).reset_index()
        emit_table(f"network {_fs} , whole partner cohort (factor 1), by mixing weight. Reference values: at 0.30 median +0.017, improved hospitals 84% and 88%, slope AKI 0.86 to 1.06 and death 0.48 to 0.76. "
                   "calibration columns: base_ = frozen model as is (the value an unlabeled hospital uses without a partner query), d_ece and d_nb_rep = after mixing minus frozen (for ECE negative is improvement, for net benefit positive is improvement), net benefit at the outcome-specific representative threshold", _g, max_rows=60)
        save_csv(_g.assign(fs=_fs), f"A7_summary_{_fs}")
    _cor = []
    for _fs, _n in NET.items():
        if len(_n) == 0:
            continue
        for _oc in OUTCOMES:
            for _f in PARTNER_PREV_FACTORS + ["all"]:
                _s = _n[(_n["outcome"] == _oc) & (np.isclose(_n["alpha"], NET_ALPHA_FIXED))]
                if _f != "all":
                    _s = _s[_s["factor"] == _f]
                if len(_s) < 5:
                    continue
                _r1, _p1 = scipy_stats.spearmanr(_s["prev_diff"], _s["intercept"])
                _r2, _p2 = scipy_stats.spearmanr(_s["prev_diff"], _s["slope"])
                _cor.append({"fs": _fs, "outcome": _oc, "partner_prev_factor": _f, "n_points": int(len(_s)),
                             "rho_prevdiff_intercept": float(_r1), "p_intercept": float(_p1),
                             "rho_prevdiff_slope": float(_r2), "p_slope": float(_p2),
                             "intercept_median": float(_s["intercept"].median()), "slope_median": float(_s["slope"].median()),
                             "d_auroc_median": float(_s["d_auroc"].median())})
    A7COR = pd.DataFrame(_cor)
    if len(A7COR):
        save_csv(A7COR, "A7_correlations")
        emit_table(f"A7 , rank correlation across hospitals between the partner prevalence difference (partner mean - own) and the calibration, mixing weight {NET_ALPHA_FIXED} (all = the points of the three multipliers combined)", A7COR, digits=3, max_rows=60)
    if "F42" in NET and "F33" in NET and len(NET["F42"]) and len(NET["F33"]):
        _k = ["outcome"]
        _a = NET["F42"][(NET["F42"]["factor"] == 1.0) & np.isclose(NET["F42"]["alpha"], NET_ALPHA_FIXED)].groupby(_k).agg(
            d_auroc_median=("d_auroc", "median"), pct_improved=("d_auroc", lambda s: float(100 * (s > 0).mean())), slope_median=("slope", "median"),
            d_ece_median=("d_ece", "median"), d_nb_rep_median=("d_nb_rep", "median"))
        _b = NET["F33"][(NET["F33"]["factor"] == 1.0) & np.isclose(NET["F33"]["alpha"], NET_ALPHA_FIXED)].groupby(_k).agg(
            d_auroc_median=("d_auroc", "median"), pct_improved=("d_auroc", lambda s: float(100 * (s > 0).mean())), slope_median=("slope", "median"),
            d_ece_median=("d_ece", "median"), d_nb_rep_median=("d_nb_rep", "median"))
        _c = _a.join(_b, lsuffix="_F42", rsuffix="_F33").reset_index()
        emit_table("A8 , network 42-column set versus 33-column set (mixing weight 0.30, whole partner cohort)", _c, digits=4)
        save_csv(_c, "A8_network_F42_F33")
    status("A7 network", "done" if NET else "not available", "")


## N4. Update hierarchy and closed testing on the frozen logistic model

The update steps recalibration-in-the-large, logistic recalibration and model revision, with closed testing, are computed on the same banks and the same draws.

Step 2 (intercept and slope) is mathematically the Platt scaling (an intercept and a slope on the logit). Step 1 is fitted by maximum likelihood. Step 3, model revision, is a continuous revision by anchored ridge. **The discrimination axis follows from the construction**: steps 1 and 2 are monotone increasing transforms of the logit, so the AUROC is unchanged and recalibration cannot beat retrieval in discrimination.

| Name | What |
|---|---|
| `upd1_b1` | re-estimate only the intercept (recalibration-in-the-large) |
| `upd2_b1` | intercept and slope (logistic recalibration, unpenalized) |
| `upd3_b1` | unpenalized model revision: the frozen linear predictor is an offset and all coefficients are re-estimated |
| `anchored_ridge` | the actual prediction of step 3: shrunk toward the development coefficients, penalty chosen by the 3 inner folds within the bank |
| `ct_b1` | one of steps 0 to 3 chosen per bank by closed testing. The test uses the likelihood of `upd3_b1`, and when step 3 is chosen the prediction is `anchored_ridge` |
| `platt_b5` | Platt on CatBoost (the only update for CatBoost) |

The decision cells below flag cells where the `ct` choice is not lower than retrieval by more than 0.0005 net benefit at the representative threshold, has lower ECE than retrieval, and is not lower than frozen by more than 0.002 AUROC. These criterion values are post hoc.

**Limitation**: in a small bank the likelihood-ratio test of the revision (d + 1 = 44 parameters) is unreliable, because there are fewer events than parameters.


In [ ]:
# ================================================================
# PRAM A4. update hierarchy and closed testing report
# ================================================================
A4TAB, A4STEP, A4FLAG = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
if not (RUN_A4 and MX and "F42" in MX) or "upd2_b1" not in set(MX["F42"]["method"]):
    status("A4 update hierarchy", "off" if not RUN_A4 else "not available", "RUN_A4=False or no grid results")
else:
    for _fs in [f for f in ("F42", "F33") if f in MX and "ct_b1" in set(MX[f]["method"])]:
        _d = MX[_fs]
        _info = MXINFO[_fs]
        _st = []
        for _tag in ("b1",):
            _c = f"ct_step_{_tag}"
            if _c in _info.columns:
                _t = _info.dropna(subset=[_c]).groupby(["outcome", "target", "bank"])[_c].value_counts(normalize=True).unstack(fill_value=0.0).reset_index()
                for _k in (0, 1, 2, 3):
                    if _k not in _t.columns:
                        _t[_k] = 0.0
                _t = _t.rename(columns={0: "share_step0_none", 1: "share_step1_intercept", 2: "share_step2_slope", 3: "share_step3_revision"})
                _t.insert(0, "base", _tag)
                _t.insert(0, "fs", _fs)
                _st.append(_t)
        if _st:
            _s = pd.concat(_st, ignore_index=True)
            _s["o"] = [bank_sort_key(b) for b in _s["bank"]]
            _s = _s.sort_values(["fs", "base", "outcome", "target", "o"]).drop(columns=["o"])
            A4STEP = pd.concat([A4STEP, _s], ignore_index=True)
        _keym = ["b1_base", "upd1_b1", "upd2_b1", "upd3_b1", "anchored_ridge", "ct_b1", "platt_b1", "platt_b5", "recal_iso", "pram_b1", "local_lr"]
        _g = draw_median_table(_d, _keym)
        _gaps = pd.concat([paired_gap(_d, m, "b1_base") for m in _keym if m != "b1_base"], ignore_index=True)
        _g = _g.merge(_gaps[["fs", "outcome", "target", "bank", "method", "gap_median", "share_below_zero"]],
                      on=["fs", "outcome", "target", "bank", "method"], how="left").rename(columns={"gap_median": "dAUROC_vs_b1"})
        _g["o"] = [bank_sort_key(b) for b in _g["bank"]]
        A4TAB = pd.concat([A4TAB, _g.sort_values(["fs", "outcome", "target", "o", "method"]).drop(columns=["o"])], ignore_index=True)
        # decision cells: cells where the closed-testing choice is equal or better than retrieval in calibration and net benefit and does not lose discrimination
        _w = _g.pivot_table(index=["fs", "outcome", "target", "bank"], columns="method", values=["auroc", "ece", "nb_site"], aggfunc="first")
        _w.columns = [f"{a}_{b}" for a, b in _w.columns]
        _w = _w.reset_index()
        _w["ct_vs_pram_flag"] = ((_w["nb_site_ct_b1"] >= _w["nb_site_pram_b1"] - 0.0005) & (_w["ece_ct_b1"] < _w["ece_pram_b1"])
                                 & (_w["auroc_ct_b1"] >= _w["auroc_b1_base"] - 0.002))
        A4FLAG = pd.concat([A4FLAG, _w[["fs", "outcome", "target", "bank", "ct_vs_pram_flag", "auroc_b1_base", "auroc_ct_b1", "auroc_pram_b1",
                                         "ece_ct_b1", "ece_pram_b1", "nb_site_ct_b1", "nb_site_pram_b1"]]], ignore_index=True)
    save_csv(A4STEP, "A4_closed_test_step_shares")
    save_csv(A4TAB, "A4_update_hierarchy_table")
    save_csv(A4FLAG, "A4_ct_vs_pram_flag")
    emit_table("A4 , fraction of steps chosen by closed testing (by bank size, per fold x draw)", A4STEP, digits=3, max_rows=120)
    emit_table("A4 , update hierarchy | retrieval | recalibration comparison (median over draws, dAUROC_vs_b1 = median of the pair difference versus the frozen logistic)", A4TAB, max_rows=300)
    emit_table("A4 , decision cells (is the ct choice equal or better than retrieval in calibration and net benefit and not losing discrimination)", A4FLAG, digits=4, max_rows=100)
    for _fs, _s in A4FLAG.groupby("fs"):
        emit(f"- {_fs}: decision cells {int(_s['ct_vs_pram_flag'].sum())}/{len(_s)}. Read separately for small banks (at least two sizes) and large banks.")
    for _fs in [f for f in HOSP if f in ("F42", "F33")]:
        _hs = HSUM[(HSUM["fs"] == _fs) & HSUM["method"].isin(["b1_base", "upd1_b1", "upd2_b1", "upd3_b1", "ct_b1", "platt_b1", "pram_b1", "recal_iso"])] if len(HSUM) else pd.DataFrame()
        if len(_hs):
            emit_table(f"A4 , hospital level {_fs} (versus frozen logistic)", _hs.drop(columns=["fs"]), max_rows=40)
    status("A4 update hierarchy", "done", "")


## N4. Platt after retrieval

Retrieval followed by Platt scaling is computed on the same banks and the same draws. Only this order is computed.

| Name | Order |
|---|---|
| `pram_platt_b1` | the retrieval mixture prediction is made first; Platt is then fitted on the within-bank cross-validation OOF mixture predictions and applied to the test mixture prediction (on the frozen logistic) |
| `pram_platt_b5` | the same order on frozen CatBoost |

The AUROC of the combination equals the AUROC of retrieval (mathematically, when the Platt slope is positive). Calibration is expected at the Platt level; the double-fit noise in small banks depends on the data.


In [ ]:
# ================================================================
# PRAM A5. retrieval + Platt combination report
# ================================================================
A5TAB = pd.DataFrame()
if not (RUN_A5 and MX and "F42" in MX) or "pram_platt_b1" not in set(MX["F42"]["method"]):
    status("A5 combination", "off" if not RUN_A5 else "not available", "RUN_A5=False or no grid results")
else:
    _parts = []
    for _fs in [f for f in ("F42", "F33") if f in MX and "pram_platt_b1" in set(MX[f]["method"])]:
        _d = MX[_fs]
        for _base, _single in (("b1_base", ["pram_b1", "platt_b1", "pram_platt_b1"]),
                               ("b5_base", ["pram_b5", "platt_b5", "pram_platt_b5"])):
            _ms = [_base] + _single
            _g = draw_median_table(_d, _ms)
            _gaps = pd.concat([paired_gap(_d, m, _base) for m in _single], ignore_index=True)
            _g = _g.merge(_gaps[["fs", "outcome", "target", "bank", "method", "gap_median", "share_below_zero", "gap_min"]],
                          on=["fs", "outcome", "target", "bank", "method"], how="left").rename(
                columns={"gap_median": "dAUROC_vs_base", "gap_min": "dAUROC_worst_draw"})
            _g.insert(1, "base", _base)
            _parts.append(_g)
    A5TAB = pd.concat(_parts, ignore_index=True)
    A5TAB["o"] = [bank_sort_key(b) for b in A5TAB["bank"]]
    A5TAB = A5TAB.sort_values(["fs", "base", "outcome", "target", "o", "method"]).drop(columns=["o"])
    save_csv(A5TAB, "A5_combination_table")
    emit_table("A5 , alone and combined (median over draws. dAUROC_worst_draw is the worst of one bank)", A5TAB[A5TAB["bank"].isin([str(b) for b in pick_report_banks(BANK_SIZES)])], max_rows=250)
    _w = A5TAB[A5TAB["base"] == "b1_base"].pivot_table(index=["fs", "outcome", "target", "bank"], columns="method", values=["auroc", "ece", "nb_site"], aggfunc="first")
    _w.columns = [f"{a}_{b}" for a, b in _w.columns]
    _w = _w.reset_index()
    _w["combo_matches_pram_auroc_and_platt_ece"] = ((_w["auroc_pram_platt_b1"] >= _w["auroc_pram_b1"] - 0.001) & (_w["ece_pram_platt_b1"] <= _w["ece_platt_b1"] + 0.002))
    _w["combo_beats_both_nb"] = (_w["nb_site_pram_platt_b1"] > np.maximum(_w["nb_site_pram_b1"], _w["nb_site_platt_b1"]))
    emit(f"- cells where the combination (retrieval then Platt) has both the AUROC of retrieval (within 0.001) and the ECE of Platt (within 0.002) {int(_w['combo_matches_pram_auroc_and_platt_ece'].sum())}/{len(_w)}, "
         f"cells where the net benefit is higher than both of the singles {int(_w['combo_beats_both_nb'].sum())}/{len(_w)} (the criterion numbers are post hoc)")
    save_csv(_w, "A5_combo_flags")
    status("A5 combination", "done", "")


## A11. Strategy selection rule within a bank (optional)

This section measures the regret of **rules that a hospital can apply using only its own bank**.

| Item | Definition |
|---|---|
| Candidates | seven: frozen logistic, frozen CatBoost, Platt, isotonic, retrieval (on logistic and on CatBoost), local logistic. The pooled strategy needs the development records and cannot be chosen by a hospital; local CatBoost and anchored ridge are left out because their inner cross-validation within a bank is costly |
| Rules | `sel_logloss` minimum log loss of the 3-fold bank predictions, `sel_nb` maximum net benefit at the representative threshold of the same predictions |
| Regret | in the same draw, (best test metric among the candidates, a post hoc oracle) minus (test metric of the strategy chosen by the rule) |
| Baselines | the regret of always retrieval, always Platt and always frozen is given too |

The within-bank retrieval predictions (OOF) are slightly optimistic because the mixing weight was chosen by the same inner cross-validation. Among the seven candidates, some use different inner splits for their within-bank predictions (the two retrievals).


In [ ]:
# ================================================================
# PRAM A11. strategy selection rule report
# ================================================================
A11TAB, A11FREQ = pd.DataFrame(), pd.DataFrame()
if not (RUN_A11 and MX and "F42" in MX) or "sel_logloss" not in set(MX["F42"]["method"]):
    status("A11 strategy selection", "off" if not RUN_A11 else "not available", "RUN_A11=False or no grid results")
else:
    _out, _fr = [], []
    for _fs in [f for f in ("F42", "F33") if f in MX and "sel_logloss" in set(MX[f]["method"])]:
        _cand = [c for c in SEL_CANDIDATES if c in set(MX[_fs]["method"])]
        _d = MX[_fs][MX[_fs]["method"].isin(_cand + ["sel_logloss", "sel_nb"])].copy()
        _d["nb_site"] = [r[nbcol(T_SITE[oc])] for oc, r in zip(_d["outcome"], _d.to_dict("records"))]
        _key = ["outcome", "target", "bank", "rep"]
        for _metric, _mg in (("auroc", NI_MARGIN_AUROC), ("nb_site", NI_MARGIN_NB)):
            _pv = _d.pivot_table(index=_key, columns="method", values=_metric)
            _orc = _pv[_cand].max(axis=1)
            for _rule in ("sel_logloss", "sel_nb", "pram_b1", "platt_b1", "b1_base"):
                if _rule not in _pv.columns:
                    continue
                _rg = (_orc - _pv[_rule]).dropna()
                _gg = _rg.groupby(level=["outcome", "target", "bank"])
                _t = _gg.agg(regret_median="median", regret_q75=lambda s: s.quantile(0.75), regret_max="max",
                             share_regret_over_margin=lambda s: float((s > _mg).mean()), n_draws="size").reset_index()
                _t.insert(0, "metric", _metric)
                _t.insert(1, "rule", _rule if _rule.startswith("sel_") else "always " + _rule)
                _t.insert(0, "fs", _fs)
                _out.append(_t)
        _inf = MXINFO[_fs]
        for _c in ("sel_logloss", "sel_nb"):
            if _c in _inf.columns:
                _f = _inf.dropna(subset=[_c]).groupby(["outcome", "target", "bank"])[_c].value_counts(normalize=True).unstack(fill_value=0.0).reset_index()
                _f.insert(0, "rule", _c)
                _f.insert(0, "fs", _fs)
                _fr.append(_f)
    A11TAB = pd.concat(_out, ignore_index=True)
    A11FREQ = pd.concat(_fr, ignore_index=True) if _fr else pd.DataFrame()
    A11TAB["o"] = [bank_sort_key(b) for b in A11TAB["bank"]]
    A11TAB = A11TAB.sort_values(["fs", "metric", "outcome", "target", "o", "rule"]).drop(columns=["o"])
    save_csv(A11TAB, "A11_selection_regret")
    if len(A11FREQ):
        save_csv(A11FREQ, "A11_selection_frequency")
    _show = A11TAB[A11TAB["bank"].isin([str(b) for b in pick_report_banks(BANK_SIZES)])]
    emit_table(f"A11 , regret (post hoc oracle minus the strategy chosen by the rule). AUROC margin {NI_MARGIN_AUROC} | net benefit margin {NI_MARGIN_NB} exceeded, the fraction of draws is share_regret_over_margin", _show, max_rows=300)
    if len(A11FREQ):
        emit_table("A11 , fraction of strategies chosen by the rule (per fold x draw)", A11FREQ[A11FREQ["bank"].isin([str(b) for b in pick_report_banks(BANK_SIZES)])], digits=3, max_rows=80)
    status("A11 strategy selection", "done", "")


## Rescoring 1. Per-draw paired DeLong with the standard implementation (42 columns: Table 2 q column and Supplementary S7 Panels A and B)

**Why**: the p values of the paired AUROC difference depend on the DeLong core. `_delong_core_original` inflates the variance of the paired AUROC difference on data with mixed scores, so p values from it are conservative. The grid csv (`matrix_draws_F42.csv`) holds only metrics, with no predictions or variances, so p cannot be recomputed from it.

**What**: with the **same split, the same draw key (`m23`) and the same `bank_predictions`** as the grid (`run_matrix_unit`), the 5-fold test predictions are collected per draw and the paired DeLong of retrieval (`pram_b1`) and frozen logistic (`b1_base`) is computed. One draw is one bank result pooling the 5-fold predictions, and the full bank is one draw. Only retrieval (the `frozen` and `pram` groups) is fitted; recalibration, local retraining, CatBoost, anchored ridge and closed testing are not part of this section. The frozen model is already in `CTX`.

| Paper position | Value of this section | csv, column |
|---|---|---|
| Table 2 q column (full bank, six cells) | p of one full-bank draw with BH adjustment over six cells | `RESCORE_table2_q.csv`, `q_standard` (q with the original implementation in `q_original_implementation`) |
| Supplementary S7 Panel A (median p over draws, q, verdict) | median p per bank size, BH adjustment over six cells | `RESCORE_S7_panelA.csv`, `p_standard_median`, `q_standard`, `verdict_standard` |
| Supplementary S7 Panel B (fraction of significant draws, 42 columns) | fraction of draws with p < 0.05 | `RESCORE_S7_panelB.csv`, `share_sig_standard` |
| for comparison | p and q using the original implementation on the same predictions | `*_original` columns of the same csv |

**Number of draws and keys.** The grid uses 10 draws per bank size (`N_DRAWS_MAIN`), and S7 is computed with the same number, so the fractions of Panel B come in units of one tenth. The full bank has no draws, so the patients are the same and only the key of the mixing-weight selection (inner 3 folds) is a free choice. To separate its effect, **only for the full bank** the retrieval prediction made with the alternative key `key08` is also produced. Whether the original implementation on the same predictions reproduces the stored q is shown in `RESCORE_check_original_core.csv`.

**Verification**: the AUROC of this section must equal the value of the same draw key of the grid (`matrix_F42`) to the ninth decimal (`RESCORE_check_vs_matrix.csv`). If it is different, the status becomes comparison mismatch and the value must not be used.


In [ ]:
# ================================================================
# PRAM rescoring 1 . per-draw paired DeLong (standard implementation), 42 columns: Table 2 q column | Supplementary S7 Panel A and B
# ================================================================
RS_DRAWS, RS_T2Q, RS_S7A, RS_S7B, RS_KEY08, RS_CHECK = (pd.DataFrame() for _ in range(6))
RS_GROUPS = ("frozen", "pram")                         # only one retrieval is needed (the other groups do not affect the retrieval predictions)
REF_T2_Q = {("AKI_168h", "MIMIC-IV"): 0.631, ("AKI_168h", "MIMIC-III"): 0.161, ("AKI_168h", "eICU"): 0.129,
            ("mortality_168h", "MIMIC-IV"): 0.829, ("mortality_168h", "MIMIC-III"): 0.701, ("mortality_168h", "eICU"): 0.015}   # stored q column of Table 2 (original implementation)
REF_Q_TOL = 0.0006                                     # q is printed to three decimals
DRAW_KEYS = ["outcome", "target", "bank", "rep"]


def delong_p(d, var):
    """p rule of the DeLong paired test: p = 1 if the variance is below 1e-15, otherwise a two-sided normal test."""
    if not (np.isfinite(d) and np.isfinite(var)):
        return float("nan")
    if var < 1e-15:
        return 1.0
    return float(2.0 * scipy_stats.norm.sf(abs(d / np.sqrt(var))))


def bh_adjust(p):
    """Benjamini-Hochberg adjusted p (q). The monotone form that keeps the order. nan is skipped and kept as nan."""
    p = np.asarray(p, dtype=float)
    ok = np.isfinite(p)
    q = np.full(len(p), np.nan)
    if ok.sum() == 0:
        return q
    pv = p[ok]
    order = np.argsort(pv)
    ranked = pv[order] * len(pv) / (np.arange(len(pv)) + 1.0)
    q_sorted = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(len(pv))
    out[order] = np.minimum(q_sorted, 1.0)
    q[ok] = out
    return q


def rescore_unit(C, outcome, target, bank_size, n_draws):
    """one unit (outcome, target, bank size). With the same split, the same draw key (m23) and the same bank_predictions as run_matrix_unit, collects the 5-fold test predictions per draw (rep)
    and gives the paired DeLong of retrieval (pram_b1) and frozen logistic (b1_base) (standard implementation; the variance and p of the original implementation are given alongside for explanation). For the full bank, the retrieval prediction variant key08 made with the alternative mixing-weight selection key
    (no prefix b1|) is also given (a diagnosis to see whether using the original implementation on the same predictions reproduces the stored q)."""
    y = C["y"]
    n = len(y)
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(n), y))
    full = bank_size == "full"
    rows = []
    for rep in range(1 if full else n_draws):
        oof = {m: np.full(n, np.nan, dtype=np.float32) for m in ("pram_b1", "b1_base")}
        oof08 = np.full(n, np.nan, dtype=np.float32)
        for fi, (pool_idx, test_idx) in enumerate(folds):
            key = f"{outcome}|{target}|{bank_size}|{fi}|{rep}"
            bank = pool_idx[np.arange(len(pool_idx)) if full else sample_bank(y[pool_idx], bank_size, "m23|" + key)]
            P, _info = bank_predictions(C, bank, test_idx, key, set(RS_GROUPS))
            for m in oof:
                oof[m][test_idx] = P[m]
            if full:
                Xb, yb, p1b = C["Xg"][bank], y[bank], C["pb1"][bank]
                Xt, yt, p1t = C["Xg"][test_idx], y[test_idx], C["pb1"][test_idx]
                p_retr = RETRIEVAL_FN({"X_tr_s": Xb, "y_tr": yb, "X_te_s": Xt, "y_te": yt, "mi": None, "mi_prob": None})
                icv08 = inner_cv_bank(Xb, yb, p1b, RETRIEVAL_FN, key=key)
                oof08[test_idx] = ((1 - icv08["alpha"]) * p1t + icv08["alpha"] * p_retr).astype(np.float32)
        variants = [("main", oof["pram_b1"].astype(float), oof["b1_base"].astype(float))]
        if full:
            variants.append(("key08", oof08.astype(float), np.asarray(C["pb1"], dtype=float)))
        for variant, pm, base in variants:
            if np.isnan(pm).any():
                continue
            a_p, a_b, dd, v_std = delong_delta_var(y, pm, base)
            v_org = delong_delta_var(y, pm, base, original=True)[3]
            rows.append({"outcome": outcome, "target": target, "bank": str(bank_size), "rep": rep, "variant": variant,
                         "auroc_pram": a_p, "auroc_frozen": a_b, "d_auroc": dd, "var_standard": v_std, "var_original": v_org,
                         "p_standard": delong_p(dd, v_std), "p_original": delong_p(dd, v_org)})
    return rows


_bad_n = None
if not (DATA_OK and RUN_RESCORE and "F42" in FS_RUN):
    status("rescoring pooled DeLong", "off" if not RUN_RESCORE else "not available", "RUN_RESCORE=False or main inputs absent or no F42")
else:
    _RR = Resumable(RESCORE_NAMES[0])
    _spec = FS_SPEC["F42"]
    _units = [("F42", oc, tg, bk) for oc in OUTCOMES for tg in _spec["targets"] for bk in _spec["banks"]]
    emit(f"### rescoring 1 , pooled DeLong units {len(_units)} (completed {sum(_RR.unit_done(u) for u in _units)} ) | draws {N_DRAWS_MAIN} (1 for the full bank) | 5 folds")
    if RESCORE_ONLY_GUARD:
        emit(f"- pre-check: units this run has skipped so far {len(RS_SEEN_DONE)} , units outside the rescoring that would run anew {len(RS_NEW_OUTSIDE)} (must be 0).")
    _w_all = {u: pooled_unit_cells(len(CTX[(u[0], u[1], u[2])]["y"]), u[3]) for u in _units}
    _w_left = sum(w for u, w in _w_all.items() if not _RR.unit_done(u))
    _t_all, _w_done = time.time(), 0.0
    for _u in _units:
        if _RR.unit_done(_u):
            print("  [SKIP]", _u)
            continue
        _t0 = time.time()
        for _r in rescore_unit(CTX[(_u[0], _u[1], _u[2])], _u[1], _u[2], _u[3], N_DRAWS_MAIN):
            _RR.add_row(_r)
        _RR.mark_unit(_u)
        _w_done += _w_all[_u]
        _eta = (time.time() - _t_all) * (_w_left - _w_done) / max(_w_done, 1.0)
        print(f"  [DONE] rescoring {_u[1]} / {_u[2]} / bank={_u[3]} ({time.time() - _t0:.0f} s) remaining time about {_eta / 60.0:.0f} min (estimated by cost ratio)", flush=True)
    RS_DRAWS = _RR.df()
    _main = RS_DRAWS[RS_DRAWS["variant"] == "main"] if len(RS_DRAWS) else RS_DRAWS
    if len(_main):
        # (1) is the AUROC of the same draw key as the grid equal
        _mx = MX.get("F42", pd.DataFrame())
        if len(_mx):
            _a = _mx[_mx["method"] == "pram_b1"].set_index(DRAW_KEYS)["auroc"]
            _b = _mx[_mx["method"] == "b1_base"].set_index(DRAW_KEYS)["auroc"]
            _m = _main.set_index(DRAW_KEYS)
            _j = pd.DataFrame({"auroc_pram_new": _m["auroc_pram"], "auroc_pram_matrix": _a, "auroc_frozen_new": _m["auroc_frozen"], "auroc_frozen_matrix": _b})
            _j = _j.dropna().reset_index()
            _j["abs_diff_pram"] = (_j["auroc_pram_new"] - _j["auroc_pram_matrix"]).abs()
            _j["abs_diff_frozen"] = (_j["auroc_frozen_new"] - _j["auroc_frozen_matrix"]).abs()
            RS_CHECK = _j
            _bad_n = int(((_j["abs_diff_pram"] > 1e-9) | (_j["abs_diff_frozen"] > 1e-9)).sum()) + (len(_main) - len(_j))
            emit(f"**rescoring versus grid comparison** , same draw key {len(_j)}/{len(_main)} compared, mismatches {_bad_n} , maximum difference of retrieval AUROC {_j['abs_diff_pram'].max():.2e}, maximum difference of frozen AUROC {_j['abs_diff_frozen'].max():.2e} (tolerance 1e-9)")
            save_csv(_j, "RESCORE_check_vs_matrix")
        else:
            emit("**rescoring versus grid comparison not possible** , the grid result (MX) does not exist.")
        # (2) Supplementary S7 Panel A and B, Table 2 q column
        _g = []
        for _bk in sorted(_main["bank"].unique(), key=bank_sort_key):
            _t = _main[_main["bank"] == _bk].groupby(["outcome", "target"], sort=False).agg(
                n_draws=("rep", "nunique"), d_auroc_median=("d_auroc", "median"),
                p_standard_median=("p_standard", "median"), p_original_median=("p_original", "median"),
                share_sig_standard=("p_standard", lambda x: float((x < SIG_LEVEL).mean())),
                share_sig_original=("p_original", lambda x: float((x < SIG_LEVEL).mean()))).reset_index()
            _t["q_standard"] = bh_adjust(_t["p_standard_median"].to_numpy())
            _t["q_original"] = bh_adjust(_t["p_original_median"].to_numpy())
            _t["verdict_standard"] = np.where(_t["q_standard"] < SIG_LEVEL, "Significant", "Not significant")
            _t.insert(2, "bank", _bk)
            _g.append(_t)
        RS_S7A = pd.concat(_g, ignore_index=True)
        RS_S7B = RS_S7A[["outcome", "target", "bank", "n_draws", "share_sig_standard", "share_sig_original"]].copy()
        RS_T2Q = RS_S7A[RS_S7A["bank"] == "full"][["outcome", "target", "bank", "d_auroc_median", "p_standard_median", "q_standard", "p_original_median", "q_original"]].copy()
        RS_T2Q["q_original_implementation"] = [REF_T2_Q.get((o, t), float("nan")) for o, t in zip(RS_T2Q["outcome"], RS_T2Q["target"])]
        emit_table("values to put in the Table 2 q column (paired DeLong of one draw of the full bank, standard implementation, BH adjustment over six cells. *_original is the value using the original implementation on the same predictions, q_original_implementation is the q printed in Table 2)", RS_T2Q, digits=4)
        emit_table("Supplementary S7 Panel A , median p over draws and BH-adjusted q over six cells (per bank size, standard implementation)",
                   RS_S7A[["outcome", "target", "bank", "n_draws", "d_auroc_median", "p_standard_median", "q_standard", "verdict_standard", "p_original_median", "q_original"]], digits=4, max_rows=60)
        emit_table(f"Supplementary S7 Panel B , fraction of draws with p < {SIG_LEVEL} (standard | original implementation). Draws {N_DRAWS_MAIN} , so units of 1/10; the full bank has 1 draw, so 0 or 1", RS_S7B, digits=3, max_rows=60)
        save_csv(RS_DRAWS, "RESCORE_draws_F42")
        save_csv(RS_T2Q, "RESCORE_table2_q")
        save_csv(RS_S7A, "RESCORE_S7_panelA")
        save_csv(RS_S7B, "RESCORE_S7_panelB")
        # (3) does the original implementation on the same predictions reproduce the stored q
        _k8 = RS_DRAWS[RS_DRAWS["variant"] == "key08"]
        if len(_k8):
            _k = _k8.groupby(["outcome", "target"], sort=False).agg(d_auroc=("d_auroc", "median"), p_original=("p_original", "median"), p_standard=("p_standard", "median")).reset_index()
            _k["q_original"] = bh_adjust(_k["p_original"].to_numpy())
            _k["q_standard"] = bh_adjust(_k["p_standard"].to_numpy())
            _k["q_original_implementation"] = [REF_T2_Q.get((o, t), float("nan")) for o, t in zip(_k["outcome"], _k["target"])]
            _k["abs_diff_q_original_vs_table2"] = (_k["q_original"] - _k["q_original_implementation"]).abs()
            _k["reproduces_table2"] = _k["abs_diff_q_original_vs_table2"] <= REF_Q_TOL
            RS_KEY08 = _k
            emit_table(f"reproduction comparison with the original implementation. If the q from the original implementation is within the Table 2 value {REF_Q_TOL} then the difference is only the DeLong implementation", RS_KEY08, digits=4)
            emit(f"- cells where the original-implementation q reproduced the q printed in Table 2 {int(_k['reproduces_table2'].sum())}/{len(_k)}. In reproduced cells, the difference from the standard-implementation q is the effect of the DeLong correction.")
            save_csv(RS_KEY08, "RESCORE_check_original_core")
    _done_all = len(_main) > 0 and set(_main["bank"].unique()) == {str(b) for b in _spec["banks"]}
    if _done_all and len(RS_CHECK) and _bad_n == 0:
        status("rescoring pooled DeLong", "done", f"unit {len(_units)} | AUROC mismatch with the grid 0")
    elif _done_all:
        status("rescoring pooled DeLong", "comparison mismatch", f"mismatch with the grid {_bad_n} or no grid. This value must not be used")
    else:
        status("rescoring pooled DeLong", "not available", "no result")


## Rescoring 2. Development-bank starting table (p column of Supplementary Table S14)

**Question**: if the development split (MIMIC-IV training 56%) is shipped with the model and used as a starting bank (source bank), what are the AUROC gain and its p as the local bank grows from 0 patients to all?

**Design**: the source bank is all development training data, with base predictions by 3-fold cross-validation (OOF). Local bank sizes `COLDSTART_BANKS` are 0, 100, 300, 500, 1,000, 2,000, 5,000 and full. Size 0 is the source bank alone; the others are the source bank plus a stratified draw of the local bank (`N_DRAWS_MAIN` draws, 1 draw for 0 and full). The paired DeLong against the frozen logistic uses the result pooling the 5-fold test predictions. The draw keys and the inner 3-fold keys are the `p3|...` keys of the code, so the same patients are drawn in every run.

**Comparison with reference values**: the AUROC columns (frozen, with source bank, change) do not depend on the DeLong implementation and are compared with the values of Supplementary Table S14 to four decimals. For p, the value using the original implementation on the same predictions (`p_original`) is compared with the p of S20, and the value from the standard implementation (`p_standard`) is the p of this analysis (`RESCORE_coldstart_check_vs_manuscript.csv`).

**This section is slow.** The source bank has over 18,000 rows, so each call runs the cosine neighbor search (inner 3 folds and the test query) on a large matrix. Look at the expected minutes of the pre-check table; if it is too slow, stop, set `RESCORE_COLDSTART = False` in cell 1 and run again: only this section is left out (finished units are skipped).


In [ ]:
# ================================================================
# PRAM rescoring 2 . development-bank starting table , p column of Supplementary Table S14
# ================================================================
RS_COLD, RS_S20, RS_S20_CHECK = (pd.DataFrame() for _ in range(3))
REF_S20_CSV = """outcome,target,local,frozen,with_source,change,p
Acute kidney injury,MIMIC-IV,none,0.7389,0.7401,0.0012,0.877
Acute kidney injury,MIMIC-IV,100,0.7389,0.7402,0.0013,0.868
Acute kidney injury,MIMIC-IV,300,0.7389,0.7401,0.0012,0.883
Acute kidney injury,MIMIC-IV,500,0.7389,0.7406,0.0017,0.831
Acute kidney injury,MIMIC-IV,1000,0.7389,0.7408,0.0019,0.813
Acute kidney injury,MIMIC-IV,2000,0.7389,0.7413,0.0024,0.758
Acute kidney injury,MIMIC-IV,5000,0.7389,0.7435,0.0046,0.551
Acute kidney injury,MIMIC-IV,full pool,0.7389,0.7445,0.0056,0.444
Acute kidney injury,MIMIC-III,none,0.7229,0.7285,0.0056,0.363
Acute kidney injury,MIMIC-III,100,0.7229,0.7288,0.0059,0.339
Acute kidney injury,MIMIC-III,300,0.7229,0.7290,0.0062,0.306
Acute kidney injury,MIMIC-III,500,0.7229,0.7292,0.0064,0.296
Acute kidney injury,MIMIC-III,1000,0.7229,0.7294,0.0065,0.293
Acute kidney injury,MIMIC-III,2000,0.7229,0.7301,0.0072,0.234
Acute kidney injury,MIMIC-III,5000,0.7229,0.7324,0.0095,0.119
Acute kidney injury,MIMIC-III,full pool,0.7229,0.7344,0.0116,0.055
Acute kidney injury,eICU-CRD,none,0.6904,0.7009,0.0105,0.286
Acute kidney injury,eICU-CRD,100,0.6904,0.7014,0.0110,0.264
Acute kidney injury,eICU-CRD,300,0.6904,0.7017,0.0113,0.243
Acute kidney injury,eICU-CRD,500,0.6904,0.7019,0.0115,0.231
Acute kidney injury,eICU-CRD,1000,0.6904,0.7022,0.0119,0.218
Acute kidney injury,eICU-CRD,2000,0.6904,0.7018,0.0114,0.213
Acute kidney injury,eICU-CRD,5000,0.6904,0.7039,0.0135,0.137
Acute kidney injury,eICU-CRD,full pool,0.6904,0.7088,0.0185,0.034
168 h mortality,MIMIC-IV,none,0.8833,0.8858,0.0024,0.914
168 h mortality,MIMIC-IV,100,0.8833,0.8859,0.0025,0.910
168 h mortality,MIMIC-IV,300,0.8833,0.8856,0.0022,0.920
168 h mortality,MIMIC-IV,500,0.8833,0.8857,0.0024,0.916
168 h mortality,MIMIC-IV,1000,0.8833,0.8856,0.0023,0.918
168 h mortality,MIMIC-IV,2000,0.8833,0.8855,0.0022,0.920
168 h mortality,MIMIC-IV,5000,0.8833,0.8864,0.0031,0.881
168 h mortality,MIMIC-IV,full pool,0.8833,0.8867,0.0033,0.867
168 h mortality,MIMIC-III,none,0.8600,0.8626,0.0026,0.881
168 h mortality,MIMIC-III,100,0.8600,0.8626,0.0026,0.884
168 h mortality,MIMIC-III,300,0.8600,0.8629,0.0029,0.869
168 h mortality,MIMIC-III,500,0.8600,0.8632,0.0032,0.854
168 h mortality,MIMIC-III,1000,0.8600,0.8638,0.0038,0.823
168 h mortality,MIMIC-III,2000,0.8600,0.8644,0.0043,0.805
168 h mortality,MIMIC-III,5000,0.8600,0.8659,0.0059,0.738
168 h mortality,MIMIC-III,full pool,0.8600,0.8672,0.0072,0.662
168 h mortality,eICU-CRD,none,0.7745,0.7980,0.0236,0.009
168 h mortality,eICU-CRD,100,0.7745,0.7974,0.0229,0.009
168 h mortality,eICU-CRD,300,0.7745,0.7975,0.0230,0.009
168 h mortality,eICU-CRD,500,0.7745,0.7979,0.0235,0.008
168 h mortality,eICU-CRD,1000,0.7745,0.7983,0.0238,0.009
168 h mortality,eICU-CRD,2000,0.7745,0.7992,0.0248,0.008
168 h mortality,eICU-CRD,5000,0.7745,0.8022,0.0277,0.006
168 h mortality,eICU-CRD,full pool,0.7745,0.8106,0.0361,0.002"""     # Supplementary Table S14 reference values: outcome, target, local patients, frozen, with source bank, change, p
S20_NAMES = {"outcome": {"AKI_168h": "Acute kidney injury", "mortality_168h": "168 h mortality"},
             "target": {"MIMIC-IV": "MIMIC-IV", "MIMIC-III": "MIMIC-III", "eICU": "eICU-CRD"},
             "bank": {"0": "none", "full": "full pool"}}
REF_P_TOL = 0.0006                        # p is printed to three decimals


def coldstart_source(dev):
    """_source_bank: the development training data as the source bank, base predictions by cross-validation (OOF)."""
    Xs, ys = dev["Xs_src"], dev["ys_src"]
    oof = np.zeros(len(ys))
    skf = StratifiedKFold(n_splits=COLDSTART_SRC_OOF_SPLITS, shuffle=True, random_state=SEED)
    for itr, iva in skf.split(np.zeros(len(ys)), ys):
        m = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xs[itr], ys[itr])
        oof[iva] = m.predict_proba(Xs[iva])[:, 1]
    return Xs, ys, oof


def coldstart_unit(C, src, outcome, target, banks, n_draws):
    """run_source_unit: one unit (outcome, target). Collects the 5-fold test predictions per bank size and gives paired DeLong against the frozen logistic (standard implementation, original for explanation)."""
    Xs, ys, ps = src
    y, Xg, pb1 = C["y"], C["Xg"], C["pb1"]
    n = len(y)
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(n), y))
    rows = []
    for bs in banks:
        n_rep = 1 if bs in (0, "full") else n_draws
        oof = np.full((n_rep, n), np.nan, dtype=np.float32)
        for fi, (pool_idx, test_idx) in enumerate(folds):
            yte, Xte, p1te = y[test_idx], Xg[test_idx], pb1[test_idx]
            Xpool, ypool, p1pool = Xg[pool_idx], y[pool_idx], pb1[pool_idx]
            for rep in range(n_rep):
                if bs == 0:
                    Xb, yb, p1b = Xs, ys, ps
                else:
                    sel = sample_bank(ypool, bs, f"p3|{outcome}|{target}|{bs}|{fi}|{rep}")
                    Xb, yb, p1b = np.vstack([Xs, Xpool[sel]]), np.concatenate([ys, ypool[sel]]), np.concatenate([ps, p1pool[sel]])
                pr = RETRIEVAL_FN({"X_tr_s": Xb, "y_tr": yb, "X_te_s": Xte, "y_te": yte, "mi": None, "mi_prob": None})
                icv = inner_cv_bank(Xb, yb, p1b, RETRIEVAL_FN, key=f"p3|{outcome}|{target}|{bs}|{fi}|{rep}")
                oof[rep, test_idx] = ((1 - icv["alpha"]) * p1te + icv["alpha"] * pr).astype(np.float32)
        for rep in range(n_rep):
            pm = oof[rep].astype(float)
            if np.isnan(pm).any():
                continue
            a_p, a_b, dd, v_std = delong_delta_var(y, pm, pb1)
            v_org = delong_delta_var(y, pm, pb1, original=True)[3]
            rows.append({"outcome": outcome, "target": target, "bank": str(bs), "rep": rep, "n_source": int(len(ys)),
                         "auroc_source": a_p, "auroc_frozen": a_b, "d_auroc": dd, "var_standard": v_std, "var_original": v_org,
                         "p_standard": delong_p(dd, v_std), "p_original": delong_p(dd, v_org)})
    return rows


if not (DATA_OK and RUN_RESCORE and RESCORE_COLDSTART and "F42" in FS_RUN):
    status("rescoring starting bank", "off", "RUN_RESCORE=False or RESCORE_COLDSTART=False or main inputs absent or no F42")
else:
    _Rc = Resumable(RESCORE_NAMES[1])
    _units_c = [("p3", oc, tg) for oc in OUTCOMES for tg in TARGETS]
    emit(f"### rescoring 2 , development-bank starting table units {len(_units_c)} (completed {sum(_Rc.unit_done(u) for u in _units_c)} ) | local bank {COLDSTART_BANKS} | draws {N_DRAWS_MAIN} (0 and full are 1)")
    _src_cache = {}
    _w_c = {u: cold_unit_cells(len(CTX[("F42", u[1], u[2])]["y"]), len(DEV[("F42", u[1])]["ys_src"])) for u in _units_c}
    _w_left_c = sum(w for u, w in _w_c.items() if not _Rc.unit_done(u))
    _t_all_c, _w_done_c = time.time(), 0.0
    for _u in _units_c:
        if _Rc.unit_done(_u):
            print("  [SKIP]", _u)
            continue
        _t0 = time.time()
        if _u[1] not in _src_cache:
            _src_cache[_u[1]] = coldstart_source(DEV[("F42", _u[1])])
        for _r in coldstart_unit(CTX[("F42", _u[1], _u[2])], _src_cache[_u[1]], _u[1], _u[2], COLDSTART_BANKS, N_DRAWS_MAIN):
            _Rc.add_row(_r)
        _Rc.mark_unit(_u)
        _w_done_c += _w_c[_u]
        _eta = (time.time() - _t_all_c) * (_w_left_c - _w_done_c) / max(_w_done_c, 1.0)
        print(f"  [DONE] starting bank {_u[1]} / {_u[2]} ({time.time() - _t0:.0f} s) remaining time about {_eta / 60.0:.0f} min (estimated by cost ratio)", flush=True)
    RS_COLD = _Rc.df()
    if len(RS_COLD):
        RS_S20 = RS_COLD.groupby(["outcome", "target", "bank"], sort=False).agg(
            n_draws=("rep", "nunique"), auroc_frozen=("auroc_frozen", "first"), auroc_source_median=("auroc_source", "median"),
            change_median=("d_auroc", "median"), p_standard_median=("p_standard", "median"), p_original_median=("p_original", "median"),
            share_sig_standard=("p_standard", lambda x: float((x < SIG_LEVEL).mean()))).reset_index()
        RS_S20["bank_order"] = [bank_sort_key(b) for b in RS_S20["bank"]]
        RS_S20 = RS_S20.sort_values(["outcome", "target", "bank_order"], kind="stable").drop(columns=["bank_order"]).reset_index(drop=True)
        _ref = pd.read_csv(io.StringIO(REF_S20_CSV), dtype={"local": str})
        _chk = RS_S20.copy()
        _chk["outcome_ms"] = _chk["outcome"].map(S20_NAMES["outcome"])
        _chk["target_ms"] = _chk["target"].map(S20_NAMES["target"])
        _chk["local_ms"] = [S20_NAMES["bank"].get(b, b) for b in _chk["bank"]]
        _chk = _chk.merge(_ref.rename(columns={"outcome": "outcome_ms", "target": "target_ms", "local": "local_ms", "frozen": "ms_frozen",
                                               "with_source": "ms_with_source", "change": "ms_change", "p": "ms_p"}),
                          on=["outcome_ms", "target_ms", "local_ms"], how="left")
        _chk["abs_diff_frozen"] = (_chk["auroc_frozen"] - _chk["ms_frozen"]).abs()
        _chk["abs_diff_with_source"] = (_chk["auroc_source_median"] - _chk["ms_with_source"]).abs()
        _chk["abs_diff_change"] = (_chk["change_median"] - _chk["ms_change"]).abs()
        _chk["abs_diff_p_original"] = (_chk["p_original_median"] - _chk["ms_p"]).abs()
        _chk["auroc_matches"] = (_chk[["abs_diff_frozen", "abs_diff_with_source", "abs_diff_change"]].max(axis=1) <= REPRO_TOL + 1e-9)
        _chk["p_original_matches"] = _chk["abs_diff_p_original"] <= REF_P_TOL
        RS_S20_CHECK = _chk.drop(columns=["outcome_ms", "target_ms", "local_ms"])
        _n_ref = int(_chk["ms_frozen"].notna().sum())
        emit_table("values to put in Supplementary Table S14 (AUROC with source bank and change are medians over draws, p_standard_median is the new p, p_original_median is the value using the original implementation on the same predictions)", RS_S20, digits=4, max_rows=60)
        emit(f"**comparison with Supplementary Table S14** , cells compared {_n_ref}/{len(_chk)} , cells where the three AUROC columns agree to the fourth decimal {int(_chk['auroc_matches'].sum())} , cells where the original-implementation p is within {REF_P_TOL} of the Table S20 p {int(_chk['p_original_matches'].sum())}")
        emit_table("cells that differ from Supplementary Table S14 (AUROC or p). If empty, all reproduced",
                   RS_S20_CHECK[~(RS_S20_CHECK["auroc_matches"] & RS_S20_CHECK["p_original_matches"])][
                       ["outcome", "target", "bank", "auroc_frozen", "ms_frozen", "auroc_source_median", "ms_with_source", "change_median", "ms_change", "p_original_median", "ms_p"]],
                   digits=4, max_rows=60)
        save_csv(RS_COLD, "RESCORE_coldstart_draws")
        save_csv(RS_S20, "RESCORE_coldstart_S20")
        save_csv(RS_S20_CHECK, "RESCORE_coldstart_check_vs_manuscript")
    _cold_ok = len(RS_COLD) > 0 and set(RS_COLD["bank"].unique()) == {str(b) for b in COLDSTART_BANKS} and len(RS_COLD.groupby(["outcome", "target"])) == len(_units_c)
    status("rescoring starting bank", "done" if _cold_ok else "not available", "")


## Closing: summary, map of result files, completion check, result zip

This cell does four things: (1) writes the key numbers and every table shown on the screen of this notebook to `PRAM-summary.md` and `PRAM-results.md`; (2) writes the map of which file goes to which table or figure position; (3) prints per section whether it ran or could not be produced, with the reason; (4) packs the result folder into a zip (packing stops if there are patient-level or identifier columns).

**The zip is created in the current working folder.**


In [ ]:
# ================================================================
# PRAM closing . summary | map of outputs | was it finished | zip
# ================================================================
MANUSCRIPT_MAP = pd.DataFrame([
    ["main results and limitations (temporal validation)", "A1_time_summary · A1_policy_contrast · A1d_hospital_time_summary · A1_period_summary", "A1a · A1b · A1c · A1d", "main table of the temporal validation"],
    ["tail risk (abstract, results and conclusion)", "A2_tail_summary · A2_safe_bank", "A2", "the limit is a post hoc definition"],
    ["main statistics section, results and first paragraph of the discussion (decision-curve thresholds)", "A3_disagreement_by_threshold · A3_range_summary · A3_area_winners · A3_dca_curves_matrix", "A3", "the outcome-specific ranges are set in code/pram_config.py"],
    ["methods and Supplementary S8 Panels F to H (update hierarchy)", "A4_update_hierarchy_table · A4_closed_test_step_shares · A4_ct_vs_pram_flag", "A4", "the closed-testing order follows Vergouwe et al. (2017); see notebook 02"],
    ["Supplementary S8 (retrieval + Platt)", "A5_combination_table · A5_combo_flags", "A5", ""],
    ["results, Fig. 3 and Supplementary S14 Panel D (tau and prediction interval)", "A6_meta_reml_hk · hospital_summary", "A6 | hospital summary", "the I squared of the main text is stay-weighted; the inverse-variance I squared is given alongside"],
    ["Supplementary S14 (event counts per hospital x test fold)", "hospital_events_summary · hospital_fold_counts_F42", "hospital summary", "the main text gives minimum, median and quartiles"],
    ["results and Supplementary S17 (unlabeled site and partner prevalence)", "A7_network_F42 · A7_network_F33 · A7_summary_F42 · A7_summary_F33 · A7_correlations", "A7", "gives the difference after mixing alongside the frozen model ECE, slope and net benefit"],
    ["features section, Supplementary S1 Panel A, S2, hospital level and unlabeled (33-column set)", "A8_compare_F42_F33 · A8_hospital_F42_F33 · A8_network_F42_F33 · main_table_F42_F33", "main analysis table | hospital summary | A7", "gain versus the size of the shift"],
    ["limitations and Supplementary S13 (modern model)", "A9_tabpfn_summary · A9_tabpfn_retrieval_gain · A9_base_strength_vs_gain · A9_tabpfn_model_info", "A9", "check that model_version is v2"],
    ["Supplementary S1 Panel A (feature count consistency)", "A10_input_features · A10_transformed_columns_F42 · A10_transformed_columns_F33", "A10", ""],
    ["first paragraph of the discussion (strategy selection)", "A11_selection_regret · A11_selection_frequency", "A11", "selected analysis"],
    ["main Table 3 and Supplementary S6 (main analysis update)", "main_table_F42_F33 · matrix_draws_F42 · matrix_draws_F33", "strategy grid", "read together with the check against the reference 42-column grid file"],
    ["main Table 2 q column and Supplementary S7 Panel A and B (42 columns, per-draw paired DeLong standard-implementation rescoring)", "RESCORE_table2_q · RESCORE_S7_panelA · RESCORE_S7_panelB · RESCORE_draws_F42 · RESCORE_check_vs_matrix · RESCORE_check_original_core", "rescoring 1", "10 draws, so Panel B has units of 1/10; the 33-column eICU is the value of A2"],
    ["Supplementary development-bank starting table (Table S14) p column", "RESCORE_coldstart_S20 · RESCORE_coldstart_draws · RESCORE_coldstart_check_vs_manuscript", "rescoring 2", "the AUROC columns are compared with Supplementary Table S14 (comparison csv)"],
], columns=["place in the paper", "csv (github_csv folder)", "notebook section", "note"])


def grab(name):
    """to be safe when a section was deleted or failed, takes the global table out by name (an empty table if absent)."""
    return globals().get(name, pd.DataFrame())


def headline_lines():
    out = []
    _a8, _a1, _a2, _a3, _a6, _a4, _a7 = (grab("A8TAB"), grab("A1CONTRAST"), grab("A2SAFE"), grab("A3RANGE"), grab("A6TAB"), grab("A4FLAG"), grab("A7COR"))
    if len(_a8):
        _e = _a8[(_a8["target"] == "eICU") & (_a8["method"] == "pram_b1") & (_a8["bank"] == "full")]
        for r in _e.itertuples():
            _s2 = "Supplementary Table S2 value +0.0266 versus +0.0199" if r.outcome == "AKI_168h" else "Supplementary Table S2 value +0.0357 versus +0.0120"
            out.append(f"- A8 eICU {r.outcome}, full-bank retrieval (on logistic) AUROC gain: 42 columns {r.gap_median_F42:+.4f} versus 33 columns {r.gap_median_F33:+.4f} ({_s2})")
    if len(_a1):
        _p = _a1[(_a1["method"] == "pram_b1") & (_a1["metric"] == "auroc") & (_a1["bank"] == "max")]
        for r in _p.itertuples():
            out.append(f"- A1 {r.analysis} {r.outcome}, maximum feasible bank retrieval gain: past bank {r.P:+.4f} | same period {r.S:+.4f} (P-S {r.P_minus_S:+.4f})")
    if len(_a2):
        for r in _a2[_a2["method"] == "pram_b1"].itertuples():
            out.append(f"- A2 {r.fs} {r.outcome} {r.target}: the smallest bank size at which retrieval (on logistic) becomes safe, AUROC criterion {r.safe_from_bank_AUROC} | net benefit criterion {getattr(r, 'safe_from_bank_NB_site', 'not applicable')} (the margin is a post hoc definition)")
    if len(_a3):
        for r in _a3[(_a3["fs"] == "F42") & (_a3["range"] == "single range 2 to 30 percent")].itertuples():
            out.append(f"- A3 {r.outcome}: hospitals where the AUROC winner and the net benefit winner differ at 20% {r.disagree_at_20:.0%} (AKI 78% | death 68%), range mean {r.disagree_mean_in_range:.0%}")
    if len(_a6):
        for r in _a6[(_a6["fs"] == "F42") & (_a6["method"] == "pram_b1")].itertuples():
            out.append(f"- A6 {r.outcome}: retrieval gain {r.mu_random:+.4f}, prediction interval [{r.pi_lo:+.4f}, {r.pi_hi:+.4f}], inverse-variance I squared {r.I2_invvar:.0f}%, τ {r.tau:.4f} "
                       f"(zero-variance hospitals {int(r.k_zero_var)} hospitals excluded , gain of the original function without exclusion {r.mu_random_orig:+.4f} τ {r.tau_orig:.4f})")
    if len(_a4):
        for fs, s in _a4.groupby("fs"):
            out.append(f"- A4 {fs}: cells where the closed-testing choice is equal or better than retrieval in calibration and net benefit and does not lose discrimination {int(s['ct_vs_pram_flag'].sum())}/{len(s)}")
    if len(_a7):
        for r in _a7[(_a7["fs"] == "F42") & (_a7["partner_prev_factor"] == "all")].itertuples():
            out.append(f"- A7 {r.outcome}: rank correlation between the partner prevalence difference and the calibration intercept {r.rho_prevdiff_intercept:+.2f} (n={r.n_points})")
    for _fs_, _n in globals().get("NET", {}).items():
        if len(_n) == 0:
            continue
        _s = _n[(_n["factor"] == 1.0) & np.isclose(_n["alpha"], NET_ALPHA_FIXED)]
        for _oc_, _t in _s.groupby("outcome"):
            out.append(f"- A7 {_fs_} {_oc_} unlabeled hospitals (mixing weight {NET_ALPHA_FIXED}, hospitals {_t['hospitalid'].nunique()} ): median AUROC gain {_t['d_auroc'].median():+.4f} (improved hospitals {100 * (_t['d_auroc'] > 0).mean():.0f}%), "
                       f"median ECE frozen {_t['base_ece'].median():.4f} versus after mixing {_t['ece'].median():.4f} (improved hospitals {100 * (_t['d_ece'] < 0).mean():.0f}%), "
                       f"median slope frozen {_t['base_slope'].median():.2f} versus after mixing {_t['slope'].median():.2f}, "
                       f"threshold {_t['rep_threshold'].iloc[0]} median net benefit frozen {_t['base_nb_rep'].median():+.4f} versus after mixing {_t['nb_rep'].median():+.4f} (improved hospitals {100 * (_t['d_nb_rep'] > 0).mean():.0f}%)")
    _t2, _s7a, _s20 = grab("RS_T2Q"), grab("RS_S7A"), grab("RS_S20")
    for r in _t2.itertuples():
        out.append(f"- rescoring Table 2 q (full bank, standard DeLong) {r.outcome} {r.target}: p {r.p_standard_median:.4f} q {r.q_standard:.4f} (original-implementation q on the same predictions {r.q_original:.4f}, q printed in Table 2 {r.q_original_implementation:.3f})")
    if len(_s7a):
        for oc, s_ in _s7a[_s7a["bank"].isin(["500", "5000", "full"])].groupby("outcome"):
            out.append(f"- rescoring S7 {oc}: fraction of significant draws (standard DeLong) " + " · ".join(f"{r.target} {r.bank} {r.share_sig_standard:.0%}" for r in s_.itertuples()))
    if len(_s20):
        for r in _s20[_s20["bank"].isin(["0", "full"])].itertuples():
            out.append(f"- rescoring starting bank {r.outcome} {r.target} local {'absent' if r.bank == '0' else 'all'}: AUROC change {r.change_median:+.4f}, p {r.p_standard_median:.4f} (original-implementation p {r.p_original_median:.4f})")
    return out


_HEAD = [f"# PRAM summary (profile {RUN_PROFILE}, run {time.strftime('%Y-%m-%d %H:%M')})", "",
         "**The numbers in this file are notebook run results.**", "",
         "## Key numbers", ""] + (headline_lines() or ["- (no section was computed)"]) + ["", "## Status by section", "",
         "| section | status | note |", "|---|---|---|"] + [f"| {k} | {v[0]} | {v[1]} |" for k, v in STATUS.items()] + ["", "## Map to manuscript table positions", "", md_table(MANUSCRIPT_MAP), ""]
if RUN_PROFILE != "full":
    _HEAD.insert(2, "**Result of the check profile. Not used in the paper.**")
save_csv(MANUSCRIPT_MAP, "manuscript_table_map")
(OUT_DIR / "PRAM-summary.md").write_text("\n".join(_HEAD), encoding="utf-8")
(OUT_DIR / "PRAM-results.md").write_text("\n".join(REPORT_LINES), encoding="utf-8")
emit_table("status by section (was it finished)", pd.DataFrame([{"section": k, "status": v[0], "note": v[1]} for k, v in STATUS.items()]), max_rows=60)
_not = [k for k, v in STATUS.items() if v[0] not in ("done", "off", "skipped")]
emit(f"- sections not completed {len(_not)} (sections turned off are excluded): {_not if _not else 'absent'}")

# ---- zip: stops if there are patient-level or identifier columns ----
ID_LIKE = {"stay_id", "patientunitstayid", "subject_id", "hadm_id", "icustay_id", "hospitalid", "uniquepid", "patienthealthsystemstayid", "pt_idx", "patient_id"}
MAX_CSV_ROWS = 250000
_bad = []
for _p in sorted(CSV_DIR.glob("*.csv")):
    _h = pd.read_csv(_p, nrows=2)
    if ID_LIKE & {c.lower() for c in _h.columns}:
        _bad.append((_p.name, "identifier column"))
    with open(_p, "rb") as _fh:
        _n = sum(1 for _ in _fh) - 1
    if _n > MAX_CSV_ROWS:
        _bad.append((_p.name, f"rows {_n} > {MAX_CSV_ROWS}"))
if _bad:
    emit(f"**The zip was not made.** There are files that look patient-level: {_bad}")
    status("zip", "not available", str(_bad))
else:
    with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as _z:
        for _p in sorted(OUT_DIR.rglob("*")):
            if _p.is_file() and RESUME_DIR not in _p.parents:
                _z.write(_p, arcname=_p.relative_to(OUT_DIR.parent).as_posix())
    with zipfile.ZipFile(ZIP_NAME) as _z:
        _names = _z.namelist()
    emit(f"- zip `{Path(ZIP_NAME).resolve()}` ({Path(ZIP_NAME).stat().st_size / 1e6:.1f} MB, files {len(_names)} files, resume folder excluded)")
    emit("**Keep this zip with the results.**")
    status("zip", "done", f"{len(_names)} files")
